# 自服务生图工作台（Colab GPU + ComfyUI）

**点「全部运行」就行。** 约 10 分钟出来画布，剩下的模型在后台继续下。

## 勾哪个模型

**默认已经勾好 Illustrious + NoobAI-XL**（约 14 GB / 10 分钟）—— 不折腾就这样。
想快、想省，把这两个取消、改勾 **Anima-Turbo**（3 GB、13 秒一张）。

下表**越靠前越值得先试**。

| 模型 | 主权重 | 速度（T4） | 什么时候选它 |
| --- | --- | --- | --- |
| Illustrious | 6.9 GB | ~46 s | **动漫首选。**最稳，LoRA 最多，还不用装 GGUF 节点 |
| NoobAI-XL | 7.1 GB | ~46 s | **值得一试。**标签打得更准、角色还原强。不能商用 |
| FLUX.2-klein | 4.3 GB | — | **想试新架构就选它。**小、许可宽松，还能多参考图改图 |
| Anima-Turbo | 3.0 GB | **~13 s** | **想快就选它。**3 GB、13 秒一张。不能用 IP-Adapter 垫图 |
| Animagine-XL | 6.9 GB | ~46 s | 需要**商用授权**的动漫档 |
| RealVisXL | 6.9 GB | — | 要**写实 / 真人**风格的时候 |
| FLUX.1-dev | 8.3 GB | 60–90 s | 生态最全但最慢，加 LoRA 还要更慢 |
| FLUX.1-schnell | 8.3 GB | 15–25 s | 要 FLUX 风格又要快的时候 |
| Z-Image-Turbo | 7.2 GB | — | 出大图会崩内存，得走 hires 放大 |
| Chroma1-HD | 9.7 GB | — | 要**负面提示词**控制的时候；也最大 |

> 「主权重」不含文本编码器与 VAE —— **FLUX / Z-Image / Chroma 那几档会自动多下约 3 GB**，
> 配置格会算进去，不用手工勾。
>
> 速度：Illustrious / Anima / FLUX 两档是**真测**的；NoobAI 与 Animagine 同配置，
> 按同值推算；`—` 是没测过。

## 几个开关

- `TUNNEL_MODE` —— 默认 `off`：**只用 3/3 的内嵌画布**，什么都不用配。
  想让手机、平板也能用，再改成开公网地址（三选一，见下表）。
- `ENABLE_MANAGER` —— 默认开。装 ComfyUI-Manager，可以在界面里装新节点、下新模型。
  代价是**拿到网址的人也能在你的 Colab 上装任意代码** —— 不想要就取消勾选。
- `PERSIST_TO_DRIVE` —— 默认关。打开后出图、设置、工作流和 LoRA 都存进同一个 Drive 根目录，
  会话回收也不丢。大模型仍留在本地盘，避免网络盘 I/O。每次新运行时可能要授权一次。

另外，**在 Colab 的 🔑（Secrets）里加一条 `HF_TOKEN`（HuggingFace 的 read token，免费）**
能让下载明显更快 —— 不加也能跑，只是会匿名下载、速率偏低。

## 从哪进界面

**默认用第 3 步的内嵌画布** —— 就在这一页里，什么都不用配。

想让手机等别的设备也能用，把 `TUNNEL_MODE` 改成下面某个值 —— 那时会**多**一个
公网地址（第 2 步打印），内嵌画布仍然在，两条路都能走。

| `TUNNEL_MODE` | 说明 |
| --- | --- |
| `off`（默认） | 不开公网地址，只用内嵌画布 |
| `quick` | **不用任何账号**，临时公网地址（`*.trycloudflare.com`）。**每次重启都会换**，地址在第 2 步打印、第 3 步也能查 |
| `named` | 你自己的 Cloudflare 隧道，域名固定。要先在 🔑 里存 `CF_TUNNEL_TOKEN`，并在 Cloudflare 侧把 ingress 指到 `http://127.0.0.1:8188` |

> ⚠️ ComfyUI 原生**没有鉴权**，公网地址一旦开出去就等于「谁拿到网址谁都能用」。


In [ ]:
#@title ⚙️ 配置（改这里，不用动代码）
# 一个勾选框 = 一个模型。**标签就是模型名**（Colab 表单拿变量名当标签，没有 label 语法）。
# 默认只勾 Illustrious + NoobAI-XL 两个核心模型（约 14 GB）。没勾的不下，也不会有对应应用。
Illustrious = True  #@param {type:"boolean"}
NoobAI_XL = True  #@param {type:"boolean"}
Animagine_XL = False  #@param {type:"boolean"}
RealVisXL = False  #@param {type:"boolean"}
FLUX_1_dev = False  #@param {type:"boolean"}
FLUX_1_schnell = False  #@param {type:"boolean"}
FLUX_2_klein = False  #@param {type:"boolean"}
Z_Image_Turbo = False  #@param {type:"boolean"}
Anima_Turbo = False  #@param {type:"boolean"}
Chroma1_HD = False  #@param {type:"boolean"}

TUNNEL_MODE = "off"  #@param ["off", "quick", "named"]
ENABLE_MANAGER = True  #@param {type:"boolean"}
PERSIST_TO_DRIVE = False  #@param {type:"boolean"}
EXTRA_NODES = ""  #@param ["", "controlnet"]

# 这一格同时持有所有大字面量（远端脚本、应用、下载清单），后面的格子直接用。
# 依赖顺序：**只有这一格 import 与定义路径** —— 挪走定义会让后面全 NameError。
import json, os, pathlib, subprocess, sys, time

# ---- HuggingFace token：**可选，有就自动用**
# 不设的话 huggingface_hub 会匿名下载，打印「unauthenticated requests」并给你更低的
# 速率和带宽。设了只是更快，没有别的影响 —— 所以不做开关，Secrets 里有没有它说了算。
try:
    from google.colab import userdata
    _hf = userdata.get('HF_TOKEN')
except Exception:
    _hf = None
if _hf:
    os.environ['HF_TOKEN'] = _hf.strip()
    print(f'HF token 已启用（{_hf[:4]}… 共 {len(_hf)} 字符）—— 下载走登录配额，更快。')
else:
    print('没设 HF_TOKEN —— 匿名下载，速度会受限。要更快就在左侧 🔑 加一条 '
          'HF_TOKEN（HuggingFace 的 read token，免费）。')

ENV = {'IMAGEGEN_CONTENT_DIR': '/content', 'IMAGEGEN_COMFY_DIR': '/content/ComfyUI', 'IMAGEGEN_REMOTE_DIR': '/content/imagegen', 'IMAGEGEN_TUNNEL_ENV': '/content/imagegen/cloudflared.env', 'IMAGEGEN_TUNNEL_LOG': '/content/cloudflared.log', 'COMFY_DIR': '/content/ComfyUI', 'COMFY_LOG': '/content/comfy.log', 'PYTHONUNBUFFERED': '1'}
REMOTE_SRC = {'comfy_bootstrap.py': '"""在新 VM 上装好 ComfyUI、节点与基础权重。幂等，可重复跑。\n\nComfyUI 本体和大模型留在 `/content/ComfyUI`；`IMAGEGEN_USER_DIR` 与\n`IMAGEGEN_LORA_DIR` 可由 Notebook 指向 Google Drive，让设置、工作流和 LoRA 跨会话保留。\n\n# 装配仍运行在 Colab 本地盘；持久化目录由调用方显式注入。\n\n阶段 B 的分工（重要）：\n\n* 本脚本只装**启动就要有的东西**：ComfyUI 本体 + 必装节点 + 基础权重。\n* warm 集（几十 GB）走网关的 `prefetch`，由 `vm_setup.py` 在网关起来后提交，\n  后台并行拉，不挡 `SETUP_DONE`；懒加载模型走网关的 `ensure_model`。\n  两条路都复用本模块的 `place()`，所以下载只有一个实现。\n\n`place()` / `coldstart()` 也由 `gateway.py` 与 `vm_setup.py` import —— 三个脚本\n在同一次冷启动里各跑一段，共用这两个基元。\n"""\nfrom __future__ import annotations\n\nimport json\nimport os\nimport re\nimport shutil\nimport subprocess\nimport sys\nimport tempfile\nimport time\n\nCONTENT = os.environ.get("IMAGEGEN_CONTENT_DIR", "/content")\nCOMFY = os.environ.get("IMAGEGEN_COMFY_DIR", f"{CONTENT}/ComfyUI")\nNODES = f"{COMFY}/custom_nodes"\nMODELS = f"{COMFY}/models"\nUSER_DIR = os.environ.get("IMAGEGEN_USER_DIR", f"{COMFY}/user")\nLORA_DIR = os.environ.get("IMAGEGEN_LORA_DIR", f"{MODELS}/loras")\n\nCLIP_VISION_NAME = "CLIP-ViT-H-14-laion2B-s32B-b79K.safetensors"\n\nBASE_CHECKPOINT = ("OnomaAIResearch/Illustrious-XL-v2.0", "Illustrious-XL-v2.0.safetensors")\n\n# h94/IP-Adapter 里的 sdxl_models/ 子目录\nIPADAPTER_FILES = (\n    "ip-adapter-plus_sdxl_vit-h.safetensors",\n    "ip-adapter-plus-face_sdxl_vit-h.safetensors",\n)\nIPADAPTER_REPO = "h94/IP-Adapter"\n\n# 注意：models/image_encoder（2.53GB）才是 ViT-H-14；\n# sdxl_models/image_encoder（3.69GB）是 ViT-bigG，只配 ip-adapter_sdxl。\nCLIP_VISION_REPO_FILE = ("h94/IP-Adapter", "models/image_encoder/model.safetensors")\n\nNODE_PACKS = (\n    "https://github.com/cubiq/ComfyUI_IPAdapter_plus.git",\n    # 模型池里的 8-bit 档是 GGUF Q8_0（T4 无 fp8/bf16 硬件路径），没有这个节点\n    # UnetLoaderGGUF / CLIPLoaderGGUF 一个都不存在。它的 requirements.txt 里有\n    # `gguf>=0.13.0` —— 这个包不在 ComfyUI 自己的 requirements.txt 里，必须装。\n    "https://github.com/city96/ComfyUI-GGUF.git",\n)\n\n# Manager（v4 线）和它的下载桥接插件都不在这里 —— 见 MANAGER_PIP 与 manager_on()。\n# 走 pip 线的原因：GUI 里「浏览/下载模型」需要一个桥接插件，而它硬性要求\n# `--enable-manager`（读它的 __init__.py 确认：`if not args.enable_manager: return`），\n# 那个开关对应的是 pip 形态的 Manager。\n\n# 模型下载桥接：把 Manager 的模型目录 + Civitai + HuggingFace + 任意直链带进**新版前端**。\n# 新版 UI 本身没有模型下载入口（已确证），官方 Manager 的 `Install Models` 只在已废弃的旧版 UI。\nBRIDGE_PACK = "https://github.com/Osterman-Designs/comfyui-manage-downloads-button.git"\n\n# Manager 走 pip。4.3 是当前最新（2026-10 核）。\nMANAGER_PIP = "comfyui-manager"\n\n# Manager 的配置只能靠写 `config.ini` 设置 —— 没有对应的启动参数、环境变量或 HTTP setter\n# （已对 comfy/cli_args.py 与两条产品线的全源码做过核查）。且**启动时读一次并缓存**，\n# 所以改完必须重启 ComfyUI 才生效。我们在起服务之前写，天然满足。\n#\n# 两个 `allow_*` 是**独立于 `security_level` 的开关，默认都是 False** —— 光把等级调到\n# `normal-` **不会**放开任意 git URL / pip 安装。用户要的是「能装新节点」，所以显式打开。\nMANAGER_CONFIG = """[default]\nsecurity_level = normal-\nallow_git_url_install = true\nallow_pip_install = true\n"""\n\n# 按需安装：`comfyui_controlnet_aux` 会现场 gcc 编译，实测 5~13 分钟，\n# 是整个装配最慢的一步，不用 ControlNet 就不该装它。\n# 用环境变量 COMFY_EXTRA_NODES=controlnet[,…] 按 key 打开，默认一个都不装。\nOPTIONAL_NODE_PACKS = {\n    "controlnet": "https://github.com/Fannovel16/comfyui_controlnet_aux.git",\n}\n\n# 按精确包名跳过，不能用前缀 —— "torchsde" 也会被 "torch" 前缀误伤\nSKIP_PACKAGES = {"torch", "torchvision", "torchaudio", "xformers"}\n\n\ndef package_name(line: str) -> str:\n    return re.split(r"[<>=!~\\[;\\s]", line.strip(), maxsplit=1)[0].lower()\n\n\ndef log(message: str) -> None:\n    print(f"[{time.strftime(\'%H:%M:%S\')}] {message}", flush=True)\n\n\ndef run(args: list[str], cwd: str | None = None) -> None:\n    subprocess.run(args, cwd=cwd, check=True)\n\n\ndef want_base_weights() -> bool:\n    """要不要下「基础权重」（Illustrious 检查点 + IP-Adapter + CLIP vision）。\n\n    这三样都是给 SDXL 系模型用的。笔记本按用户勾选的模型决定：**一个 SDXL 都没勾就不下**，\n    否则「没勾的不会下载」就是假话。\n    """\n    return os.environ.get("IMAGEGEN_BASE_WEIGHTS", "1") not in ("0", "false", "no")\n\n\ndef manager_on() -> bool:\n    """要不要装 Manager（+ 它的下载桥接）。\n\n    ⚠️ **Manager 按设计就是「能装任意代码」的工具。** 我们为了配隧道，ComfyUI 跑在\n    `--listen 127.0.0.1`，而它判断「是否本地」的依据**就是这个启动参数**\n    （`is_local_mode = is_loopback(args.listen)`，v3/v4 两线源码一致）**不是请求来源 IP**。\n    于是从公网进来的请求会被当成本地，拿到和本地用户同等的权限。默认打开，知情选择。\n    关掉：`IMAGEGEN_MANAGER=0`。\n    """\n    return os.environ.get("IMAGEGEN_MANAGER", "1") not in ("0", "false", "no")\n\n\ndef node_packs() -> tuple[str, ...]:\n    """要装的自定义节点。两个环境开关：\n\n    - `COMFY_EXTRA_NODES=controlnet` 打开可选项（见 `OPTIONAL_NODE_PACKS`）\n    - `IMAGEGEN_MANAGER=0` 去掉模型下载桥接（Manager 本体走 pip，见 `MANAGER_PIP`）\n    """\n    extra = [name.strip() for name in\n             os.environ.get("COMFY_EXTRA_NODES", "").split(",") if name.strip()]\n    unknown = [name for name in extra if name not in OPTIONAL_NODE_PACKS]\n    if unknown:\n        raise SystemExit(\n            f"未知的 COMFY_EXTRA_NODES 项：{unknown}，可选 {sorted(OPTIONAL_NODE_PACKS)}")\n    packs = (*NODE_PACKS, *(OPTIONAL_NODE_PACKS[name] for name in extra))\n    if manager_on():\n        packs += (BRIDGE_PACK,)\n    return packs\n\n\ndef write_manager_config() -> str | None:\n    """写 Manager 的 `config.ini`，返回路径；Manager 没装或已关闭就返回 None。\n\n    路径是 `<USER_DIRECTORY>/__manager/config.ini`（ComfyUI ≥0.3.76），Colab 上即\n    `/content/ComfyUI/user/__manager/config.ini`。**只有这一个途径能设 `security_level`**，\n    而且它在启动时读一次并缓存 —— 所以必须在 `comfy_start.py` 之前写完。\n\n    Manager 只会覆写它自己"拥有"的键，手写键与其它 section 会保留；但它**不会**替我们补\n    `allow_*` 的默认值，所以那两行必须由这里写死。\n    """\n    if not manager_on():\n        return None\n    os.makedirs(USER_DIR, exist_ok=True)\n    target = os.path.join(USER_DIR, "__manager")\n    os.makedirs(target, exist_ok=True)\n    path = os.path.join(target, "config.ini")\n    with open(path, "w", encoding="utf-8") as handle:\n        handle.write(MANAGER_CONFIG)\n    return path\n\n\ndef clone(url: str, dest: str) -> None:\n    """clone 一个仓库到 `dest`。\n\n    **不能用「目录存在」当「已 clone」的判据。** 别处只要 mkdir 过就会误判：先传 App Mode\n    工作流会创建 `ComfyUI/user/default/workflows/`，于是这里的 `isdir` 为真，\n    **ComfyUI 本体从来没被 clone**，`main.py` 不存在 → 起服务必然失败。\n\n    判据用 `.git`。目录在但不是仓库时 clone 到临时目录再**合并** —— 不能 `rm -rf`，里面\n    可能已经有用户的模型和自定义节点。\n    """\n    if os.path.isdir(os.path.join(dest, ".git")):\n        log(f"已存在，跳过 clone：{dest}")\n        return\n    if os.path.isdir(dest):\n        log(f"{dest} 存在但不是 git 仓库（别处 mkdir 过？），clone 到临时目录后合并")\n        with tempfile.TemporaryDirectory() as tmp:\n            run(["git", "clone", "--depth", "1", url, tmp])\n            run(["cp", "-a", f"{tmp}/.", dest])\n        return\n    log(f"clone {url}")\n    run(["git", "clone", "--depth", "1", url, dest])\n\n\ndef patch_userdata_routes() -> None:\n    """把 userdata 路由的 `{file}` 放宽成 `{file:.*}`。\n\n    aiohttp 的 `{file}` 只匹配**单段**路径。前端把文件名 `workflows/xxx.json` 编码成\n    `workflows%2Fxxx.json` 送出去，服务端 unquote 后拿到真斜杠 —— 正常能工作。\n\n    **但我们前面有反向代理**（Colab 的端口代理 / cloudflared），代理会把 `%2F` 解码成\n    真斜杠 ✗，于是服务端看到两段路径 → 路由匹配不上 → **404**。\n\n    在同一台机器上绕过代理直连 127.0.0.1:8188 实测：\n        /api/userdata/workflows%2F<名>.json  → 200\n        /api/userdata/workflows/<名>.json    → 404   ← 代理解码后才长这样\n\n    症状：侧栏里**能看到**工作流，点它/拖它都没反应，Console 报\n    `Failed to load file \'workflows/xxx.json\': 404`。\n\n    上游有 PR #12468 做同样的事，但**至今未合并**（master 仍是 `{file}`），所以自己补。\n    将来上游合了，这段可以删掉。\n    `get_request_user_filepath` 里的 commonpath 检查仍然拦着目录穿越，放宽匹配是安全的。\n    """\n    # **本模块全程用 os.path / 字符串路径** —— COMFY 是 str 不是 Path，\n    # 用 pathlib 的 `/` 会当场 TypeError（2026-10-12 就是这么挂的）。\n    path = os.path.join(COMFY, "app", "user_manager.py")\n    if not os.path.isfile(path):\n        log(f"没找到 {path}，跳过 userdata 路由补丁")\n        return\n    with open(path, encoding="utf-8") as handle:\n        source = handle.read()\n    patched = source\n    for old, new in (\n        (\'"/userdata/{file}"\', \'"/userdata/{file:.*}"\'),\n        (\'"/userdata/{file}/move/{dest}"\', \'"/userdata/{file:.*}/move/{dest:.*}"\'),\n    ):\n        patched = patched.replace(old, new)\n    if patched == source:\n        log("userdata 路由已是 {file:.*}，无需补丁")\n        return\n    with open(path, "w", encoding="utf-8") as handle:\n        handle.write(patched)\n    log("已放宽 userdata 路由（{file} → {file:.*}）—— 否则代理下工作流全部 404")\n\n\ndef pip_install(packages: tuple[str, ...] | list[str]) -> None:\n    """装 pip 包。走的是和节点依赖同一条路，所以要显式装、不经 requirements.txt。"""\n    log(f"pip install {\' \'.join(packages)}")\n    run([sys.executable, "-m", "pip", "install", "-q", *packages])\n\n\ndef install_requirements(path: str, cwd: str) -> None:\n    """装依赖但跳过 torch 系列 —— Colab 自带的 torch 就是对的，重装要拉 2GB+。"""\n    if not os.path.isfile(path):\n        return\n    with open(path, encoding="utf-8") as handle:\n        wanted = [\n            line.strip()\n            for line in handle\n            if line.strip()\n            and not line.strip().startswith("#")\n            and package_name(line) not in SKIP_PACKAGES\n        ]\n    if not wanted:\n        return\n    log(f"pip install {len(wanted)} 个依赖（{path}）")\n    run([sys.executable, "-m", "pip", "install", "-q", *wanted], cwd=cwd)\n\n\n# ---------------------------------------------------------------- 权重落地\n\n\ndef hf_download(repo: str, filename: str) -> str:\n    from huggingface_hub import hf_hub_download\n\n    return hf_hub_download(\n        repo_id=repo, filename=filename, token=os.environ.get("HF_TOKEN") or None\n    )\n\n\ndef link_or_copy(src: str, dest: str) -> None:\n    """优先软链（HF 缓存里的文件，几 GB 不重复拷）；不支持软链的文件系统才退回拷贝。"""\n    try:\n        os.symlink(src, dest)\n    except FileExistsError:\n        os.unlink(dest)\n        os.symlink(src, dest)\n    except OSError:\n        log(f"软链失败，退回复制 {dest}")\n        shutil.copy2(src, dest)\n\n\ndef place(repo: str, filename: str, dest_dir: str, dest_name: str | None = None) -> str:\n    """下载后放进 ComfyUI 的模型目录，返回目标路径。已存在就是空操作。\n\n    `dest_name` 默认只取 `filename` 的最后一段 —— 仓库内的子目录（如 IP-Adapter 的\n    `sdxl_models/`）不进 ComfyUI 的目录结构。要保留子目录结构就显式给 `dest_name`\n    （网关就是这么做的：客户端拿「相对模型根目录的路径」当模型 id）。\n    """\n    os.makedirs(dest_dir, exist_ok=True)\n    dest = os.path.join(dest_dir, dest_name or os.path.basename(filename))\n    if os.path.exists(dest):\n        log(f"已存在 {dest}")\n        return dest\n    if os.path.islink(dest):\n        # 断链（HF 缓存被清过）：留着会让 os.symlink 抛 FileExistsError，\n        # 再退回整份拷贝 —— 白等几个 GB 的磁盘。\n        os.unlink(dest)\n    src = hf_download(repo, filename)\n    link_or_copy(src, dest)\n    log(f"就位 {dest}")\n    return dest\n\n\n# ---------------------------------------------------------------- 冷启动计时\n\n\ndef coldstart_path() -> str:\n    return os.environ.get("IMAGEGEN_COLDSTART") or f"{CONTENT}/coldstart.jsonl"\n\n\ndef coldstart(phase: str, seconds: float, ok: bool = True, **detail) -> None:\n    """追加一行阶段耗时到 coldstart.jsonl（每行一个 JSON）。\n\n    一次冷启动横跨三个进程：本脚本装配、`vm_setup.py` 拉起、`gateway.py` 跑 warm\n    下载。每个进程只知道自己那段的耗时，所以各自追加。计时不能拖垮主流程，\n    写失败只打日志。\n\n    额外关键字参数原样进 `detail`，例如 `coldstart("comfy_core", 12.3, dir=COMFY)`。\n    """\n    record = {\n        "ts": round(time.time(), 3),\n        "iso": time.strftime("%Y-%m-%dT%H:%M:%S%z"),\n        "phase": phase,\n        "seconds": round(float(seconds), 2),\n        "ok": bool(ok),\n    }\n    if detail:\n        record["detail"] = detail\n    path = coldstart_path()\n    try:\n        os.makedirs(os.path.dirname(path) or ".", exist_ok=True)\n        with open(path, "a", encoding="utf-8") as handle:\n            handle.write(json.dumps(record, ensure_ascii=False) + "\\n")\n    except OSError as exc:\n        log(f"写 {path} 失败（忽略）：{exc}")\n\n\n# ---------------------------------------------------------------- 主流程\n\n\ndef main() -> int:\n    started = time.time()\n\n    log("=== 1/4 ComfyUI 本体 ===")\n    step = time.time()\n    clone("https://github.com/comfyanonymous/ComfyUI.git", COMFY)\n    install_requirements(f"{COMFY}/requirements.txt", COMFY)\n    patch_userdata_routes()\n    coldstart("comfy_core", time.time() - step, dir=COMFY)\n\n    log("=== 2/4 自定义节点 ===")\n    step = time.time()\n    packs = node_packs()\n    with_manager = manager_on()\n    if with_manager:\n        # **v4 线是 pip 包，不是 custom node。** 桥接插件硬性要求 `args.enable_manager`\n        # （它的 __init__.py 里 `if not args.enable_manager: return`），而那个开关对应\n        # 的就是 pip 形态的 Manager，所以两者必须一起。\n        #\n        # 老的 v3 clone 必须先清掉 —— 否则两个 Manager 并存，前端会加载两套 JS。\n        # 那是我们自己 clone 的产物（不是用户的东西），删掉是安全的。\n        legacy = os.path.join(NODES, "ComfyUI-Manager")\n        if os.path.isdir(legacy):\n            shutil.rmtree(legacy)\n            log(f"清掉旧的 v3 clone：{legacy}（Manager 已改走 pip）")\n        pip_install([MANAGER_PIP])\n    else:\n        log("IMAGEGEN_MANAGER=0 —— 不装 Manager，也不装模型下载桥接")\n    for url in packs:\n        dest = os.path.join(NODES, os.path.basename(url)[:-4])\n        clone(url, dest)\n        install_requirements(f"{dest}/requirements.txt", dest)\n        # **桥接插件的 install.py 不在这里跑。** 它是 Manager 的「运行时安装钩子」，\n        # 自己会检查 `import comfyui_manager`（那个包 import 时就拉 aiohttp，不是\n        # 装上就能用），插件 README 也明说要先 `launch once with --enable-manager`。\n        # 放到 comfy_start.py 里、ComfyUI 起来之后再跑 —— 见 install_bridge()。\n    if with_manager:\n        path = write_manager_config()\n        log(f"Manager 配置：{path}" if path\n            else "Manager 未装上，跳过配置（看上面的 clone 日志）")\n    coldstart("nodes", time.time() - step,\n              packs=[os.path.basename(url)[:-4] for url in packs])\n\n    log("=== 3/4 基础权重 ===")\n    step = time.time()\n    # **这一步只在勾了 SDXL 系模型时才做。** 它下的三样（Illustrious 检查点、IP-Adapter、\n    # CLIP vision）全是给 SDXL 用的；用户没勾任何 SDXL 模型时下它们就是白下 13 GB，\n    # 而且会让「没勾的不会下载」变成假话。\n    os.makedirs(USER_DIR, exist_ok=True)\n    os.makedirs(LORA_DIR, exist_ok=True)\n\n    if not want_base_weights():\n        log("没勾 SDXL 系模型 —— 跳过基础权重（Illustrious / IP-Adapter / CLIP vision）")\n        placed = []\n    else:\n        placed = [\n            place(*BASE_CHECKPOINT, f"{MODELS}/checkpoints"),\n            *(place(IPADAPTER_REPO, f"sdxl_models/{name}", f"{MODELS}/ipadapter")\n              for name in IPADAPTER_FILES),\n            place(*CLIP_VISION_REPO_FILE, f"{MODELS}/clip_vision", CLIP_VISION_NAME),\n        ]\n    coldstart("base_weights", time.time() - step,\n              files=[os.path.basename(path) for path in placed])\n\n    log("=== 4/4 汇总 ===")\n    for sub in ("checkpoints", "ipadapter", "clip_vision"):\n        target = f"{MODELS}/{sub}"\n        files = sorted(os.listdir(target)) if os.path.isdir(target) else []\n        log(f"{sub}: {files}")\n    coldstart("bootstrap_total", time.time() - started,\n              file=coldstart_path(), phases=["comfy_core", "nodes", "base_weights"])\n    log("SETUP_DONE")\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'comfy_client.py': '"""ComfyUI 无头客户端。只在 VM 内使用（stdlib，不依赖 requests 是否装了）。"""\nfrom __future__ import annotations\n\nimport json\nimport time\nimport urllib.error\nimport urllib.parse\nimport urllib.request\n\nHOST = "127.0.0.1"\nPORT = 8188\nBASE = f"http://{HOST}:{PORT}"\n\n\ndef _request(path: str, *, data: bytes | None = None, method: str | None = None,\n             headers: dict | None = None, timeout: float = 30):\n    req = urllib.request.Request(\n        f"{BASE}{path}", data=data, method=method, headers=headers or {}\n    )\n    return urllib.request.urlopen(req, timeout=timeout)\n\n\ndef get_json(path: str, timeout: float = 30):\n    with _request(path, timeout=timeout) as response:\n        return json.loads(response.read().decode("utf-8"))\n\n\ndef post_json(path: str, payload: dict, timeout: float = 60):\n    body = json.dumps(payload).encode("utf-8")\n    with _request(\n        path, data=body, method="POST",\n        headers={"Content-Type": "application/json"}, timeout=timeout,\n    ) as response:\n        return json.loads(response.read().decode("utf-8"))\n\n\ndef alive() -> bool:\n    try:\n        return get_json("/system_stats", timeout=3) is not None\n    except (urllib.error.URLError, OSError, TimeoutError):\n        return False\n\n\ndef wait_ready(seconds: float = 180) -> bool:\n    deadline = time.time() + seconds\n    while time.time() < deadline:\n        if alive():\n            return True\n        time.sleep(2)\n    return False\n\n\ndef upload_image(local_path: str, name: str | None = None) -> dict:\n    """把参考图塞进 ComfyUI 的 input 目录，返回 {name, subfolder, type}。"""\n    import mimetypes\n    import os\n    import uuid\n\n    name = name or os.path.basename(local_path)\n    boundary = uuid.uuid4().hex\n    content_type = mimetypes.guess_type(name)[0] or "application/octet-stream"\n    with open(local_path, "rb") as handle:\n        payload = handle.read()\n\n    parts = []\n    parts.append(\n        f"--{boundary}\\r\\nContent-Disposition: form-data; name=\\"image\\"; "\n        f"filename=\\"{name}\\"\\r\\nContent-Type: {content_type}\\r\\n\\r\\n".encode()\n    )\n    parts.append(payload)\n    parts.append(f"\\r\\n--{boundary}--\\r\\n".encode())\n    body = b"".join(parts)\n\n    with _request(\n        "/upload/image", data=body, method="POST",\n        headers={"Content-Type": f"multipart/form-data; boundary={boundary}"},\n        timeout=120,\n    ) as response:\n        return json.loads(response.read().decode("utf-8"))\n\n\ndef submit(workflow: dict, client_id: str = "imagegen") -> str:\n    result = post_json("/prompt", {"prompt": workflow, "client_id": client_id})\n    return result["prompt_id"]\n\n\ndef history(prompt_id: str) -> dict:\n    return get_json(f"/history/{prompt_id}", timeout=30)\n\n\ndef wait_result(prompt_id: str, seconds: float = 900) -> dict:\n    """轮询到作业出结果。失败时把 ComfyUI 记的执行错误一并抛出来。"""\n    deadline = time.time() + seconds\n    while time.time() < deadline:\n        entry = history(prompt_id).get(prompt_id)\n        if entry:\n            status = entry.get("status", {})\n            if status.get("status_str") == "error" or not status.get("completed", True):\n                raise RuntimeError(f"ComfyUI 执行失败：{json.dumps(status)[:1500]}")\n            if status.get("completed"):\n                return entry\n        time.sleep(2)\n    raise TimeoutError(f"等待 {prompt_id} 超时")\n\n\ndef fetch_image(entry: dict, node_id: str, index: int = 0) -> bytes:\n    info = entry["outputs"][node_id]["images"][index]\n    query = urllib.parse.urlencode(\n        {"filename": info["filename"], "subfolder": info.get("subfolder", ""),\n         "type": info.get("type", "output")}\n    )\n    with _request(f"/view?{query}", timeout=120) as response:\n        return response.read()\n', 'comfy_start.py': '"""确保 VM 里的 ComfyUI 在跑。没跑就以脱离进程方式拉起并等它就绪。"""\nfrom __future__ import annotations\n\nimport os\nimport subprocess\nimport sys\n\n# colab exec 是把文件当单元格发过来的，没有 __file__，靠环境变量定位\nsys.path.insert(0, os.environ.get("IMAGEGEN_REMOTE_DIR", "/content/imagegen"))\nimport comfy_client  # noqa: E402\n\nCOMFY = os.environ.get("COMFY_DIR", "/content/ComfyUI")\nLOG = os.environ.get("COMFY_LOG", "/content/comfy.log")\n\n\nBRIDGE = os.path.join(\n    os.environ.get("IMAGEGEN_NODES_DIR", "/content/ComfyUI/custom_nodes"),\n    "comfyui-manage-downloads-button",\n)\n\n\ndef install_bridge() -> None:\n    """把模型下载桥接的前端 JS 铺好 —— **必须在 ComfyUI 起来之后**。\n\n    它是 Manager 的运行时安装钩子，自己会 `import comfyui_manager`（那个包 import 时就\n    拉 aiohttp），插件 README 也明说要先 `launch once with --enable-manager`。\n    2026-10-12 我把它塞在 bootstrap 里跑，于是装配直接挂掉。\n\n    幂等判据用 `js/bridge-config.json` —— 那正是它自己写的产物，不用另外记状态。\n    """\n    setup = os.path.join(BRIDGE, "install.py")\n    marker = os.path.join(BRIDGE, "js", "bridge-config.json")\n    if not os.path.isfile(setup):\n        return\n    if os.path.isfile(marker):\n        print("模型下载桥接已就绪")\n        return\n    print("铺设模型下载桥接的前端资源…")\n    # **必须把 ComfyUI 根放进 PYTHONPATH。** `comfyui_manager` 自己 import\n    # `comfy.cli_args`，而这个包自身加载失败时报的是「pip package is not installed」\n    # —— 极具误导性。2026-10-12 本地对照实验确证：\n    #     cwd=插件目录            → 报上面那个假错\n    #     PYTHONPATH=<ComfyUI 根> → install.py complete，生成 js/bridge-config.json\n    # 注意 cwd 不够：Python 把**脚本所在目录**（不是 cwd）放在 sys.path[0]。\n    env = {**os.environ, "PYTHONPATH": os.pathsep.join(\n        [COMFY, os.environ.get("PYTHONPATH", "")]).rstrip(os.pathsep)}\n    done = subprocess.run([sys.executable, setup], cwd=BRIDGE, env=env)\n    if done.returncode != 0:\n        # **不把整个启动拖垮** —— ComfyUI 已经能出图了，桥接只是模型下载那一个入口。\n        print("⚠️ 桥接安装失败（上面有它的报错）。ComfyUI 本身可用，")\n        print("   只是界面里不会有模型下载入口。")\n        return\n    print("桥接就绪 —— **浏览器硬刷新（Ctrl+Shift+R）** 才会看到那个浮动按钮")\n\n\ndef desired_argv() -> list[str]:\n    """这次应该用哪些参数起 ComfyUI。"""\n    return [\n        sys.executable, "main.py",\n        "--listen", comfy_client.HOST,\n        "--port", str(comfy_client.PORT),\n    ] + os.environ.get("COMFY_ARGS", "").split()\n\n\ndef running_argv() -> list[str] | None:\n    """正在跑的 ComfyUI 的启动参数（读 `/proc/<pid>/cmdline`）。没在跑返回 None。"""\n    for entry in os.listdir("/proc"):\n        if not entry.isdigit():\n            continue\n        try:\n            with open(f"/proc/{entry}/cmdline", "rb") as handle:\n                raw = handle.read()\n        except OSError:\n            continue\n        argv = [part.decode(errors="replace") for part in raw.split(b"\\0") if part]\n        if len(argv) >= 2 and argv[1].endswith("main.py"):\n            return argv\n    return None\n\n\ndef stop() -> bool:\n    """让正在跑的 ComfyUI 退出；超时返回 False，禁止新进程抢占旧端口。"""\n    import signal\n    import time\n    for entry in os.listdir("/proc"):\n        if not entry.isdigit():\n            continue\n        try:\n            with open(f"/proc/{entry}/cmdline", "rb") as handle:\n                raw = handle.read()\n        except OSError:\n            continue\n        argv = [part.decode(errors="replace") for part in raw.split(b"\\0") if part]\n        if len(argv) >= 2 and argv[1].endswith("main.py"):\n            os.kill(int(entry), signal.SIGTERM)\n            break\n    for _ in range(30):\n        if not comfy_client.alive():\n            return True\n        time.sleep(0.5)\n    return False\n\n\ndef main() -> int:\n    args = desired_argv()\n    have = running_argv()\n\n    if have is not None:\n        # **参数一致才跳过。** 原来只看「在不在跑」就 return 0，于是改了\n        # `--enable-cors-header` / `--output-directory` / `--user-directory` 之后重跑本步\n        # 什么都不会发生 —— 那些都是启动参数，已运行的进程不会重新读它们。\n        if have[1:] == args[1:]:\n            print("ComfyUI 已在运行，参数一致")\n            install_bridge()\n            return 0\n        print("ComfyUI 在跑，但参数变了 —— 重启它")\n        print(f"    原来：{\' \'.join(have[1:])}")\n        print(f"    现在：{\' \'.join(args[1:])}")\n        if not stop():\n            print("旧 ComfyUI 未在 15 秒内退出，拒绝启动新进程")\n            return 1\n\n    if not os.path.isdir(COMFY):\n        print(f"找不到 {COMFY}，先跑 comfy_bootstrap.py")\n        return 1\n\n    with open(LOG, "wb") as handle:\n        proc = subprocess.Popen(\n            args, cwd=COMFY, stdout=handle, stderr=subprocess.STDOUT,\n            stdin=subprocess.DEVNULL, start_new_session=True,\n        )\n    print(f"启动 pid={proc.pid} {\' \'.join(args[2:])} -> {LOG}")\n\n    if not comfy_client.wait_ready(240):\n        print("等待就绪超时，日志尾部：")\n        with open(LOG, encoding="utf-8", errors="replace") as handle:\n            print("\\n".join(handle.read().splitlines()[-25:]))\n        return 1\n    print("READY")\n    # 桥接的 JS 要在 ComfyUI 活着的时候铺（它要 import Manager 的包）。\n    install_bridge()\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'cloudflared_up.py': '"""在 VM 上装并起 `cloudflared` 隧道（远端侧，由笔记本或 `scripts/open_tunnel.py` 调）。\n\n两种模式，由 `IMAGEGEN_TUNNEL_MODE` 选：\n\n- **`named`**（默认）：**remotely-managed** 命名隧道（`config_src=cloudflare`）—— ingress\n  规则存在 Cloudflare 那边，所以这台 VM 上**不需要任何配置文件**，只要一个 connector token。\n  域名固定（由 Cloudflare 侧的 ingress 决定），所以本脚本不打印域名。\n- **`quick`**：`cloudflared tunnel --url`，**不需要任何账号或 token**，但**每次重启都会换一个\n  随机域名**（`*.trycloudflare.com`）。所以这边必须把 URL 从日志里捞出来打印，否则用户拿不到入口。\n\n**token 只从文件读，不走命令行** —— 传 `--token` 会把它暴露在 `ps` 和启动日志里。\n`cloudflared tunnel run` 认 `TUNNEL_TOKEN` 环境变量，所以这里把它塞进子进程的 env。\n"""\nfrom __future__ import annotations\n\nimport os\nimport pathlib\nimport re\nimport subprocess\nimport sys\nimport time\n\nBIN = "/usr/local/bin/cloudflared"\nDOWNLOAD = ("https://github.com/cloudflare/cloudflared/releases/latest/download/"\n            "cloudflared-linux-amd64")\nENV_FILE = pathlib.Path(os.environ.get("IMAGEGEN_TUNNEL_ENV",\n                                       "/content/imagegen/cloudflared.env"))\nLOG = pathlib.Path(os.environ.get("IMAGEGEN_TUNNEL_LOG", "/content/cloudflared.log"))\nPORT = os.environ.get("IMAGEGEN_TUNNEL_PORT", "8188")\nMODE = os.environ.get("IMAGEGEN_TUNNEL_MODE", "named").strip().lower()\n# 匹配「在跑的 connector」。用方括号写法，免得 pgrep/pkill 匹到自己这条命令行。\nRUN_PATTERN = "cloudflared.*tunne[l]"\n# quick 隧道分配的地址长这样，从日志里捞。\nQUICK_URL = re.compile(r"https://[a-z0-9][a-z0-9-]*\\.trycloudflare\\.com")\n\n\ndef sh(cmd: str) -> str:\n    return subprocess.run(["bash", "-lc", cmd], capture_output=True, text=True).stdout\n\n\ndef running() -> bool:\n    return subprocess.run(["pgrep", "-f", RUN_PATTERN], capture_output=True).returncode == 0\n\n\ndef read_token() -> str:\n    if not ENV_FILE.is_file():\n        raise SystemExit(f"缺少 {ENV_FILE} —— 先由 scripts/open_tunnel.py 把 token 推上来")\n    for line in ENV_FILE.read_text(encoding="utf-8").splitlines():\n        if line.startswith("TUNNEL_TOKEN="):\n            token = line.split("=", 1)[1].strip()\n            if token:\n                return token\n    raise SystemExit(f"{ENV_FILE} 里没有 TUNNEL_TOKEN")\n\n\ndef install() -> None:\n    if os.path.exists(BIN):\n        return\n    print("下载 cloudflared …", flush=True)\n    proc = subprocess.run(["curl", "-fsSL", "-o", BIN, DOWNLOAD],\n                          capture_output=True, text=True)\n    if proc.returncode != 0:\n        raise SystemExit(f"下载失败（{proc.returncode}）：{(proc.stderr or \'\')[-300:]}")\n    os.chmod(BIN, 0o755)\n\n\ndef stop() -> None:\n    if subprocess.run(["pgrep", "-f", RUN_PATTERN], capture_output=True).returncode == 0:\n        print("已有 connector 在跑，先停掉", flush=True)\n        sh(f"pkill -f \'{RUN_PATTERN}\'; sleep 3")\n\n\ndef start(token: str | None) -> tuple[int, int]:\n    """起 connector，返回 `(pid, 日志偏移)`。\n\n    **偏移必须返回。** 日志是追加写的，而 quick 隧道**每次启动都分配一个新地址** ——\n    `wait_quick_url` 只能看这次启动之后写进去的那一段。\n\n    不记偏移就会捞到**上一次运行**留下的地址；那个隧道已被 `stop()` 杀掉，打开就是\n    Cloudflare 1033（隧道无活跃连接）。判据：两次运行打印出**同一个**地址 ——\n    而 quick 地址是不可能重复的。\n    """\n    env = dict(os.environ)\n    if token:\n        env["TUNNEL_TOKEN"] = token\n        argv = [BIN, "tunnel", "--no-autoupdate", "--loglevel", "info", "run"]\n    else:\n        argv = [BIN, "tunnel", "--no-autoupdate", "--loglevel", "info",\n                "--url", f"http://127.0.0.1:{PORT}"]\n    LOG.parent.mkdir(parents=True, exist_ok=True)\n    offset = LOG.stat().st_size if LOG.is_file() else 0\n    handle = open(LOG, "ab")\n    proc = subprocess.Popen(argv, stdout=handle, stderr=subprocess.STDOUT,\n                            stdin=subprocess.DEVNULL, start_new_session=True, env=env)\n    return proc.pid, offset\n\n\ndef wait_quick_url(offset: int = 0, seconds: int = 45) -> str:\n    """从日志里等 quick 隧道**这次**分配到的地址（只看 `offset` 之后的部分）。\n\n    必须等 —— 地址是启动后才由 Cloudflare 随机分配的，而且每次重启都换。拿不到就等于\n    用户没有入口，所以这属于「必须成功」而不是「尽力而为」。\n    """\n    deadline = time.time() + seconds\n    while time.time() < deadline:\n        if LOG.is_file():\n            fresh = LOG.read_text(errors="replace")[offset:]\n            found = QUICK_URL.search(fresh)\n            if found:\n                return found.group(0)\n        time.sleep(2)\n    raise SystemExit(f"{seconds}s 内没拿到 quick 隧道地址，看 {LOG} 的日志")\n\n\ndef disable() -> int:\n    """停止本 Notebook 管理的 connector；off 模式重复运行也必须真正关闭公网入口。"""\n    if running():\n        stop()\n        if running():\n            raise SystemExit("cloudflared connector 停止超时")\n        print("connector 已停止", flush=True)\n    else:\n        print("connector 未运行", flush=True)\n    return 0\n\n\ndef main() -> int:\n    if MODE not in ("off", "named", "quick"):\n        raise SystemExit(f"IMAGEGEN_TUNNEL_MODE 只能是 off / named / quick，收到 {MODE!r}")\n    if MODE == "off":\n        return disable()\n    # 自启路径会反复调本脚本（每次 `ensure_ready`），已在跑就别白白重启一遍 connector。\n    # 健康与否由主机侧的 `tunnel.verify()` 判：自检不过时它会带 `force` 再来。\n    if "--skip-if-running" in sys.argv and running():\n        print("connector 已在运行，跳过重启", flush=True)\n        return 0\n\n    if MODE not in ("named", "quick"):\n        raise SystemExit(f"IMAGEGEN_TUNNEL_MODE 只能是 named / quick，收到 {MODE!r}")\n\n    token = None if MODE == "quick" else read_token()\n    install()\n    print(sh(f"{BIN} --version").strip(), flush=True)\n    stop()\n    pid, offset = start(token)\n    print(f"connector pid={pid} -> {LOG}", flush=True)\n\n    if MODE == "quick":\n        url = wait_quick_url(offset)\n        print(f"\\n公网地址（quick 隧道，重启会换）：{url}", flush=True)\n        print(f"    本机端口 {PORT}；日志 {LOG}", flush=True)\n        return 0\n\n    time.sleep(20)\n    tail = sh(f"tail -c 2000 {LOG}")\n    print("--- cloudflared 日志尾部 ---", flush=True)\n    print(tail, flush=True)\n    alive = running()\n    registered = "Registered tunnel connection" in tail or "Connection registered" in tail\n    # **不打印域名** —— 命名隧道的域名由 Cloudflare 侧 ingress 决定，写进日志就等于\n    # 把个人域名落进可分享的文件/输出里。\n    print(f"\\n进程存活={alive} 已注册连接={registered}（域名见你自己 Cloudflare 的 ingress）",\n          flush=True)\n    return 0 if alive else 1\n\n\nif __name__ == "__main__":\n    sys.exit(main())\n', 'warm_models.py': '"""按 `warm.json` 把 warm 集装到 ComfyUI 的模型目录。\n\n**给自服务笔记本用的**，但也只有这一个用途：无头编排那边走网关的 `prefetch` 操作\n（并行、有任务表、可查进度），而笔记本里没有网关，需要一条单机的下载路径。\n\n直接复用 `comfy_bootstrap.place()` —— 就是线上装权重用的那个函数，带 HF 缓存软链与断链\n处理。目标路径**保留仓库内的子目录结构**，与网关的 `model_dest()` 一致：压成 basename 会\n让 ComfyUI 下拉里的路径对不上（`models.py` 的 `filename` 本身就是相对模型根目录的路径）。\n\n单个文件失败**不中断整批** —— 50 GB 里挂一条不该让其余四条跟着没。\n"""\nfrom __future__ import annotations\n\nimport json\nimport pathlib\nimport sys\nimport time\nimport fcntl\n\nsys.path.insert(0, "/content/imagegen")\n\nimport comfy_bootstrap as cb  # noqa: E402\n\nITEMS_FILE = pathlib.Path("/content/imagegen/warm.json")\n\n\nLOCK_FILE = pathlib.Path("/content/imagegen/warm.lock")\n\n\ndef main() -> int:\n    LOCK_FILE.parent.mkdir(parents=True, exist_ok=True)\n    lock_handle = LOCK_FILE.open("w", encoding="utf-8")\n    try:\n        fcntl.flock(lock_handle.fileno(), fcntl.LOCK_EX | fcntl.LOCK_NB)\n    except BlockingIOError:\n        print("已有 warm_models 下载在运行，本轮不重复启动", flush=True)\n        return 0\n    items = json.loads(ITEMS_FILE.read_text(encoding="utf-8"))\n    print(f"warm 集 {len(items)} 条", flush=True)\n    started = time.time()\n    failed: list[str] = []\n\n    for index, item in enumerate(items, 1):\n        root = pathlib.Path(cb.LORA_DIR) if item["subdir"] == "loras" else pathlib.Path(cb.MODELS)\n        dest = root / item["filename"] if item["subdir"] == "loras" else root / item["subdir"] / item["filename"]\n        label = f"{item[\'subdir\']}/{item[\'filename\']}"\n        print(f"[{index}/{len(items)}] {label}", flush=True)\n        step = time.time()\n        try:\n            cb.place(item["repo"], item["filename"], str(dest.parent), dest.name)\n        except Exception as exc:  # noqa: BLE001 单条失败不该拖垮整批\n            failed.append(label)\n            print(f"    FAILED {type(exc).__name__}: {exc}", flush=True)\n            continue\n        print(f"    ok {time.time() - step:.1f}s", flush=True)\n\n    print(f"--- 合计 {time.time() - started:.0f}s，失败 {len(failed)} 条 ---", flush=True)\n    for label in failed:\n        print(f"    ✗ {label}", flush=True)\n    print("WARM_DONE", flush=True)\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n'}
MODELS_SPEC = [{'id': 'illustrious-xl-v2.0', 'display': 'Illustrious', 'var': 'Illustrious', 'arch': 'sdxl', 'default': True, 'warm': [('checkpoints', 'Illustrious-XL-v2.0.safetensors')], 'apps': ['改图-Illustrious.json', '文生图-Illustrious.json']}, {'id': 'noobai-xl-1.1', 'display': 'NoobAI-XL', 'var': 'NoobAI_XL', 'arch': 'sdxl', 'default': True, 'warm': [('checkpoints', 'NoobAI-XL-v1.1.safetensors')], 'apps': ['改图-NoobAI-XL.json', '文生图-NoobAI-XL.json']}, {'id': 'animagine-xl-4.0', 'display': 'Animagine-XL', 'var': 'Animagine_XL', 'arch': 'sdxl', 'default': False, 'warm': [('checkpoints', 'animagine-xl-4.0.safetensors')], 'apps': ['改图-Animagine-XL.json', '文生图-Animagine-XL.json']}, {'id': 'realvisxl-v5.0', 'display': 'RealVisXL', 'var': 'RealVisXL', 'arch': 'sdxl', 'default': False, 'warm': [('checkpoints', 'RealVisXL_V5.0_fp16.safetensors')], 'apps': ['改图-RealVisXL.json', '文生图-RealVisXL.json']}, {'id': 'flux1-dev-q5ks', 'display': 'FLUX.1-dev', 'var': 'FLUX_1_dev', 'arch': 'flux', 'default': False, 'warm': [('text_encoders', 'clip_l.safetensors'), ('text_encoders', 't5-v1_1-xxl-encoder-Q5_K_M.gguf'), ('unet', 'flux1-dev-Q5_K_S.gguf'), ('vae', 'ae.safetensors')], 'apps': ['改图-FLUX.1-dev.json', '文生图-FLUX.1-dev.json']}, {'id': 'flux1-schnell-q5ks', 'display': 'FLUX.1-schnell', 'var': 'FLUX_1_schnell', 'arch': 'flux', 'default': False, 'warm': [('text_encoders', 'clip_l.safetensors'), ('text_encoders', 't5-v1_1-xxl-encoder-Q5_K_M.gguf'), ('unet', 'flux1-schnell-Q5_K_S.gguf'), ('vae', 'ae.safetensors')], 'apps': ['改图-FLUX.1-schnell.json', '文生图-FLUX.1-schnell.json']}, {'id': 'flux2-klein-4b-q8', 'display': 'FLUX.2-klein', 'var': 'FLUX_2_klein', 'arch': 'flux2', 'default': False, 'warm': [('text_encoders', 'Qwen3-4B-Q8_0.gguf'), ('unet', 'flux-2-klein-4b-Q8_0.gguf'), ('vae', 'split_files/vae/flux2-vae.safetensors')], 'apps': ['改图-FLUX.2-klein.json', '文生图-FLUX.2-klein.json']}, {'id': 'z-image-turbo-q8', 'display': 'Z-Image-Turbo', 'var': 'Z_Image_Turbo', 'arch': 'zimage', 'default': False, 'warm': [('text_encoders', 'Qwen3-4B-Q8_0.gguf'), ('unet', 'z_image_turbo-Q8_0.gguf'), ('vae', 'split_files/vae/ae.safetensors')], 'apps': ['改图-Z-Image-Turbo.json', '文生图-Z-Image-Turbo.json']}, {'id': 'anima-turbo-q5km', 'display': 'Anima-Turbo', 'var': 'Anima_Turbo', 'arch': 'anima', 'default': False, 'warm': [('text_encoders', 'split_files/text_encoders/qwen_3_06b_base.safetensors'), ('unet', 'anima-turbo-v1.1-Q5_K_M.gguf'), ('vae', 'split_files/vae/qwen_image_vae.safetensors')], 'apps': ['改图-Anima-Turbo.json', '文生图-Anima-Turbo.json']}, {'id': 'chroma1-hd-q8', 'display': 'Chroma1-HD', 'var': 'Chroma1_HD', 'arch': 'chroma', 'default': False, 'warm': [('text_encoders', 'clip_l.safetensors'), ('text_encoders', 't5-v1_1-xxl-encoder-Q5_K_M.gguf'), ('unet', 'Chroma1-HD-Q8_0.gguf'), ('vae', 'ae.safetensors')], 'apps': ['改图-Chroma1-HD.json', '文生图-Chroma1-HD.json']}]
LORA_SPEC = [{'key': ('loras', 'EbiblueIllustriousXL_byKonan.safetensors'), 'arch': 'sdxl'}]
ALL_WARM_ITEMS = [{'id': 'Illustrious-XL-v2.0.safetensors', 'warm': True, 'repo': 'OnomaAIResearch/Illustrious-XL-v2.0', 'filename': 'Illustrious-XL-v2.0.safetensors', 'subdir': 'checkpoints'}, {'id': 'NoobAI-XL-v1.1.safetensors', 'warm': True, 'repo': 'Laxhar/noobai-XL-1.1', 'filename': 'NoobAI-XL-v1.1.safetensors', 'subdir': 'checkpoints'}, {'id': 'animagine-xl-4.0.safetensors', 'warm': True, 'repo': 'cagliostrolab/animagine-xl-4.0', 'filename': 'animagine-xl-4.0.safetensors', 'subdir': 'checkpoints'}, {'id': 'RealVisXL_V5.0_fp16.safetensors', 'warm': True, 'repo': 'SG161222/RealVisXL_V5.0', 'filename': 'RealVisXL_V5.0_fp16.safetensors', 'subdir': 'checkpoints'}, {'id': 'flux1-dev-Q5_K_S.gguf', 'warm': True, 'repo': 'city96/FLUX.1-dev-gguf', 'filename': 'flux1-dev-Q5_K_S.gguf', 'subdir': 'unet'}, {'id': 'clip_l.safetensors', 'warm': True, 'repo': 'comfyanonymous/flux_text_encoders', 'filename': 'clip_l.safetensors', 'subdir': 'text_encoders'}, {'id': 't5-v1_1-xxl-encoder-Q5_K_M.gguf', 'warm': True, 'repo': 'city96/t5-v1_1-xxl-encoder-gguf', 'filename': 't5-v1_1-xxl-encoder-Q5_K_M.gguf', 'subdir': 'text_encoders'}, {'id': 'ae.safetensors', 'warm': True, 'repo': 'lodestones/Chroma', 'filename': 'ae.safetensors', 'subdir': 'vae'}, {'id': 'flux1-schnell-Q5_K_S.gguf', 'warm': True, 'repo': 'city96/FLUX.1-schnell-gguf', 'filename': 'flux1-schnell-Q5_K_S.gguf', 'subdir': 'unet'}, {'id': 'flux-2-klein-4b-Q8_0.gguf', 'warm': True, 'repo': 'unsloth/FLUX.2-klein-4B-GGUF', 'filename': 'flux-2-klein-4b-Q8_0.gguf', 'subdir': 'unet'}, {'id': 'Qwen3-4B-Q8_0.gguf', 'warm': True, 'repo': 'unsloth/Qwen3-4B-GGUF', 'filename': 'Qwen3-4B-Q8_0.gguf', 'subdir': 'text_encoders'}, {'id': 'split_files/vae/flux2-vae.safetensors', 'warm': True, 'repo': 'Comfy-Org/vae-text-encorder-for-flux-klein-4b', 'filename': 'split_files/vae/flux2-vae.safetensors', 'subdir': 'vae'}, {'id': 'z_image_turbo-Q8_0.gguf', 'warm': True, 'repo': 'jayn7/Z-Image-Turbo-GGUF', 'filename': 'z_image_turbo-Q8_0.gguf', 'subdir': 'unet'}, {'id': 'split_files/vae/ae.safetensors', 'warm': True, 'repo': 'Comfy-Org/z_image_turbo', 'filename': 'split_files/vae/ae.safetensors', 'subdir': 'vae'}, {'id': 'anima-turbo-v1.1-Q5_K_M.gguf', 'warm': True, 'repo': 'vanes430/Anima-Turbo-V1.1-GGUF', 'filename': 'anima-turbo-v1.1-Q5_K_M.gguf', 'subdir': 'unet'}, {'id': 'split_files/text_encoders/qwen_3_06b_base.safetensors', 'warm': True, 'repo': 'circlestone-labs/Anima', 'filename': 'split_files/text_encoders/qwen_3_06b_base.safetensors', 'subdir': 'text_encoders'}, {'id': 'split_files/vae/qwen_image_vae.safetensors', 'warm': True, 'repo': 'circlestone-labs/Anima', 'filename': 'split_files/vae/qwen_image_vae.safetensors', 'subdir': 'vae'}, {'id': 'Chroma1-HD-Q8_0.gguf', 'warm': True, 'repo': 'silveroxides/Chroma1-HD-GGUF', 'filename': 'Chroma1-HD-Q8_0.gguf', 'subdir': 'unet'}, {'id': 'EbiblueIllustriousXL_byKonan.safetensors', 'warm': True, 'repo': 'Yevrey921/Ebiblue_Gesugao_Style_IllustriousXL_LORA', 'filename': 'EbiblueIllustriousXL_byKonan.safetensors', 'subdir': 'loras'}]
ALL_APP_JSON = {'文生图-Illustrious.json': '{"id": null, "revision": 0, "last_node_id": 9, "last_link_id": 12, "nodes": [{"id": 1, "type": "CheckpointLoaderSimple", "pos": [80, 120], "size": [300, 100], "flags": {}, "order": 0, "mode": 0, "inputs": [], "outputs": [{"name": "MODEL", "type": "MODEL", "links": [1, 2]}, {"name": "CLIP", "type": "CLIP", "links": [4, 5]}, {"name": "VAE", "type": "VAE", "links": [11]}], "properties": {"Node name for S&R": "CheckpointLoaderSimple"}, "widgets_values": ["Illustrious-XL-v2.0.safetensors"], "widgets_values_named": {"ckpt_name": "Illustrious-XL-v2.0.safetensors"}}, {"id": 2, "type": "LoraLoaderModelOnly", "pos": [410, 120], "size": [300, 100], "flags": {}, "order": 1, "mode": 0, "inputs": [{"name": "model", "type": "MODEL", "link": 1}], "outputs": [{"name": "MODEL", "type": "MODEL", "links": [3]}], "properties": {"Node name for S&R": "LoraLoaderModelOnly"}, "widgets_values": ["EbiblueIllustriousXL_byKonan.safetensors", 0.8], "widgets_values_named": {"lora_name": "EbiblueIllustriousXL_byKonan.safetensors", "strength_model": 0.8}}, {"id": 3, "type": "ComfySwitchNode", "pos": [740, 120], "size": [300, 100], "flags": {}, "order": 2, "mode": 0, "inputs": [{"name": "on_false", "type": "COMFY_MATCHTYPE_V3", "link": 2}, {"name": "on_true", "type": "COMFY_MATCHTYPE_V3", "link": 3}], "outputs": [{"name": "output", "type": "COMFY_MATCHTYPE_V3", "links": [6]}], "properties": {"Node name for S&R": "ComfySwitchNode"}, "widgets_values": [false], "widgets_values_named": {"switch": false}}, {"id": 4, "type": "CLIPTextEncode", "pos": [410, 340], "size": [300, 100], "flags": {}, "order": 3, "mode": 0, "inputs": [{"name": "clip", "type": "CLIP", "link": 4}], "outputs": [{"name": "CONDITIONING", "type": "CONDITIONING", "links": [7]}], "properties": {"Node name for S&R": "CLIPTextEncode"}, "widgets_values": ["1girl, solo, looking at viewer, upper body, detailed face, soft lighting"], "widgets_values_named": {"text": "1girl, solo, looking at viewer, upper body, detailed face, soft lighting"}}, {"id": 5, "type": "CLIPTextEncode", "pos": [410, 560], "size": [300, 100], "flags": {}, "order": 4, "mode": 0, "inputs": [{"name": "clip", "type": "CLIP", "link": 5}], "outputs": [{"name": "CONDITIONING", "type": "CONDITIONING", "links": [8]}], "properties": {"Node name for S&R": "CLIPTextEncode"}, "widgets_values": ["lowres, bad anatomy, bad hands, extra digits, fewer digits, cropped, worst quality, low quality, jpeg artifacts, signature, watermark, username, blurry"], "widgets_values_named": {"text": "lowres, bad anatomy, bad hands, extra digits, fewer digits, cropped, worst quality, low quality, jpeg artifacts, signature, watermark, username, blurry"}}, {"id": 6, "type": "EmptyLatentImage", "pos": [80, 340], "size": [300, 100], "flags": {}, "order": 5, "mode": 0, "inputs": [], "outputs": [{"name": "LATENT", "type": "LATENT", "links": [9]}], "properties": {"Node name for S&R": "EmptyLatentImage"}, "widgets_values": [832, 1216, 1], "widgets_values_named": {"width": 832, "height": 1216, "batch_size": 1}}, {"id": 7, "type": "KSampler", "pos": [1070, 120], "size": [300, 100], "flags": {}, "order": 6, "mode": 0, "inputs": [{"name": "model", "type": "MODEL", "link": 6}, {"name": "positive", "type": "CONDITIONING", "link": 7}, {"name": "negative", "type": "CONDITIONING", "link": 8}, {"name": "latent_image", "type": "LATENT", "link": 9}], "outputs": [{"name": "LATENT", "type": "LATENT", "links": [10]}], "properties": {"Node name for S&R": "KSampler"}, "widgets_values": [0, "randomize", 28, 5.5, "euler_ancestral", "normal", 1.0], "widgets_values_named": {"seed": 0, "control_after_generate": "randomize", "steps": 28, "cfg": 5.5, "sampler_name": "euler_ancestral", "scheduler": "normal", "denoise": 1.0}}, {"id": 8, "type": "VAEDecode", "pos": [1400, 120], "size": [300, 100], "flags": {}, "order": 7, "mode": 0, "inputs": [{"name": "samples", "type": "LATENT", "link": 10}, {"name": "vae", "type": "VAE", "link": 11}], "outputs": [{"name": "IMAGE", "type": "IMAGE", "links": [12]}], "properties": {"Node name for S&R": "VAEDecode"}, "widgets_values": [], "widgets_values_named": {}}, {"id": 9, "type": "SaveImage", "pos": [1730, 120], "size": [300, 100], "flags": {}, "order": 8, "mode": 0, "inputs": [{"name": "images", "type": "IMAGE", "link": 12}], "outputs": [{"name": "images", "type": "IMAGE", "links": []}], "properties": {"Node name for S&R": "SaveImage"}, "widgets_values": ["app/txt2img-illustrious-xl-v2.0"], "widgets_values_named": {"filename_prefix": "app/txt2img-illustrious-xl-v2.0"}}], "links": [[1, 1, 0, 2, 0, "MODEL"], [2, 1, 0, 3, 0, "MODEL"], [3, 2, 0, 3, 1, "MODEL"], [4, 1, 1, 4, 0, "CLIP"], [5, 1, 1, 5, 0, "CLIP"], [6, 3, 0, 7, 0, "COMFY_MATCHTYPE_V3"], [7, 4, 0, 7, 1, "CONDITIONING"], [8, 5, 0, 7, 2, "CONDITIONING"], [9, 6, 0, 7, 3, "LATENT"], [10, 7, 0, 8, 0, "LATENT"], [11, 1, 2, 8, 1, "VAE"], [12, 8, 0, 9, 0, "IMAGE"]], "groups": [], "config": {}, "extra": {"linearMode": true, "linearData": {"inputs": [[4, "text"], [5, "text"], [6, "width"], [6, "height"], [7, "seed"], [7, "steps"], [7, "cfg"], [3, "switch", {"description": "打开才启用下面的 LoRA。默认关闭时不会加载 LoRA 文件，所以 LoRA 没下载好也能正常出图。"}], [2, "lora_name"], [2, "strength_model"]], "outputs": [9]}, "ds": {"scale": 0.75, "offset": [0, 0]}}, "version": 0.4}', '改图-Illustrious.json': '{"id": null, "revision": 0, "last_node_id": 11, "last_link_id": 15, "nodes": [{"id": 1, "type": "CheckpointLoaderSimple", "pos": [80, 120], "size": [300, 100], "flags": {}, "order": 0, "mode": 0, "inputs": [], "outputs": [{"name": "MODEL", "type": "MODEL", "links": [1, 2]}, {"name": "CLIP", "type": "CLIP", "links": [4, 5]}, {"name": "VAE", "type": "VAE", "links": [8, 14]}], "properties": {"Node name for S&R": "CheckpointLoaderSimple"}, "widgets_values": ["Illustrious-XL-v2.0.safetensors"], "widgets_values_named": {"ckpt_name": "Illustrious-XL-v2.0.safetensors"}}, {"id": 2, "type": "LoraLoaderModelOnly", "pos": [410, 120], "size": [300, 100], "flags": {}, "order": 1, "mode": 0, "inputs": [{"name": "model", "type": "MODEL", "link": 1}], "outputs": [{"name": "MODEL", "type": "MODEL", "links": [3]}], "properties": {"Node name for S&R": "LoraLoaderModelOnly"}, "widgets_values": ["EbiblueIllustriousXL_byKonan.safetensors", 0.8], "widgets_values_named": {"lora_name": "EbiblueIllustriousXL_byKonan.safetensors", "strength_model": 0.8}}, {"id": 3, "type": "ComfySwitchNode", "pos": [740, 120], "size": [300, 100], "flags": {}, "order": 2, "mode": 0, "inputs": [{"name": "on_false", "type": "COMFY_MATCHTYPE_V3", "link": 2}, {"name": "on_true", "type": "COMFY_MATCHTYPE_V3", "link": 3}], "outputs": [{"name": "output", "type": "COMFY_MATCHTYPE_V3", "links": [9]}], "properties": {"Node name for S&R": "ComfySwitchNode"}, "widgets_values": [false], "widgets_values_named": {"switch": false}}, {"id": 4, "type": "CLIPTextEncode", "pos": [410, 340], "size": [300, 100], "flags": {}, "order": 3, "mode": 0, "inputs": [{"name": "clip", "type": "CLIP", "link": 4}], "outputs": [{"name": "CONDITIONING", "type": "CONDITIONING", "links": [10]}], "properties": {"Node name for S&R": "CLIPTextEncode"}, "widgets_values": ["1girl, solo, looking at viewer, upper body, detailed face, soft lighting"], "widgets_values_named": {"text": "1girl, solo, looking at viewer, upper body, detailed face, soft lighting"}}, {"id": 5, "type": "CLIPTextEncode", "pos": [410, 560], "size": [300, 100], "flags": {}, "order": 4, "mode": 0, "inputs": [{"name": "clip", "type": "CLIP", "link": 5}], "outputs": [{"name": "CONDITIONING", "type": "CONDITIONING", "links": [11]}], "properties": {"Node name for S&R": "CLIPTextEncode"}, "widgets_values": ["lowres, bad anatomy, bad hands, extra digits, fewer digits, cropped, worst quality, low quality, jpeg artifacts, signature, watermark, username, blurry"], "widgets_values_named": {"text": "lowres, bad anatomy, bad hands, extra digits, fewer digits, cropped, worst quality, low quality, jpeg artifacts, signature, watermark, username, blurry"}}, {"id": 6, "type": "LoadImage", "pos": [80, 340], "size": [300, 100], "flags": {}, "order": 5, "mode": 0, "inputs": [], "outputs": [{"name": "IMAGE", "type": "IMAGE", "links": [6]}, {"name": "MASK", "type": "MASK", "links": []}], "properties": {"Node name for S&R": "LoadImage"}, "widgets_values": ["__REFERENCE__"], "widgets_values_named": {"image": "__REFERENCE__"}}, {"id": 7, "type": "ImageScale", "pos": [410, 780], "size": [300, 100], "flags": {}, "order": 6, "mode": 0, "inputs": [{"name": "image", "type": "IMAGE", "link": 6}], "outputs": [{"name": "IMAGE", "type": "IMAGE", "links": [7]}], "properties": {"Node name for S&R": "ImageScale"}, "widgets_values": ["lanczos", 832, 1216, "disabled"], "widgets_values_named": {"upscale_method": "lanczos", "width": 832, "height": 1216, "crop": "disabled"}}, {"id": 8, "type": "VAEEncode", "pos": [740, 340], "size": [300, 100], "flags": {}, "order": 7, "mode": 0, "inputs": [{"name": "pixels", "type": "IMAGE", "link": 7}, {"name": "vae", "type": "VAE", "link": 8}], "outputs": [{"name": "LATENT", "type": "LATENT", "links": [12]}], "properties": {"Node name for S&R": "VAEEncode"}, "widgets_values": [], "widgets_values_named": {}}, {"id": 9, "type": "KSampler", "pos": [1070, 120], "size": [300, 100], "flags": {}, "order": 8, "mode": 0, "inputs": [{"name": "model", "type": "MODEL", "link": 9}, {"name": "positive", "type": "CONDITIONING", "link": 10}, {"name": "negative", "type": "CONDITIONING", "link": 11}, {"name": "latent_image", "type": "LATENT", "link": 12}], "outputs": [{"name": "LATENT", "type": "LATENT", "links": [13]}], "properties": {"Node name for S&R": "KSampler"}, "widgets_values": [0, "randomize", 28, 5.5, "euler_ancestral", "normal", 0.6], "widgets_values_named": {"seed": 0, "control_after_generate": "randomize", "steps": 28, "cfg": 5.5, "sampler_name": "euler_ancestral", "scheduler": "normal", "denoise": 0.6}}, {"id": 10, "type": "VAEDecode", "pos": [1400, 120], "size": [300, 100], "flags": {}, "order": 9, "mode": 0, "inputs": [{"name": "samples", "type": "LATENT", "link": 13}, {"name": "vae", "type": "VAE", "link": 14}], "outputs": [{"name": "IMAGE", "type": "IMAGE", "links": [15]}], "properties": {"Node name for S&R": "VAEDecode"}, "widgets_values": [], "widgets_values_named": {}}, {"id": 11, "type": "SaveImage", "pos": [1730, 120], "size": [300, 100], "flags": {}, "order": 10, "mode": 0, "inputs": [{"name": "images", "type": "IMAGE", "link": 15}], "outputs": [{"name": "images", "type": "IMAGE", "links": []}], "properties": {"Node name for S&R": "SaveImage"}, "widgets_values": ["app/img2img-illustrious-xl-v2.0"], "widgets_values_named": {"filename_prefix": "app/img2img-illustrious-xl-v2.0"}}], "links": [[1, 1, 0, 2, 0, "MODEL"], [2, 1, 0, 3, 0, "MODEL"], [3, 2, 0, 3, 1, "MODEL"], [4, 1, 1, 4, 0, "CLIP"], [5, 1, 1, 5, 0, "CLIP"], [6, 6, 0, 7, 0, "IMAGE"], [7, 7, 0, 8, 0, "IMAGE"], [8, 1, 2, 8, 1, "VAE"], [9, 3, 0, 9, 0, "COMFY_MATCHTYPE_V3"], [10, 4, 0, 9, 1, "CONDITIONING"], [11, 5, 0, 9, 2, "CONDITIONING"], [12, 8, 0, 9, 3, "LATENT"], [13, 9, 0, 10, 0, "LATENT"], [14, 1, 2, 10, 1, "VAE"], [15, 10, 0, 11, 0, "IMAGE"]], "groups": [], "config": {}, "extra": {"linearMode": true, "linearData": {"inputs": [[4, "text"], [5, "text"], [6, "image"], [9, "denoise"], [7, "width"], [7, "height"], [9, "seed"], [9, "steps"], [9, "cfg"], [3, "switch", {"description": "打开才启用下面的 LoRA。默认关闭时不会加载 LoRA 文件，所以 LoRA 没下载好也能正常出图。"}], [2, "lora_name"], [2, "strength_model"]], "outputs": [11]}, "ds": {"scale": 0.75, "offset": [0, 0]}}, "version": 0.4}', '文生图-NoobAI-XL.json': '{"id": null, "revision": 0, "last_node_id": 9, "last_link_id": 12, "nodes": [{"id": 1, "type": "CheckpointLoaderSimple", "pos": [80, 120], "size": [300, 100], "flags": {}, "order": 0, "mode": 0, "inputs": [], "outputs": [{"name": "MODEL", "type": "MODEL", "links": [1, 2]}, {"name": "CLIP", "type": "CLIP", "links": [4, 5]}, {"name": "VAE", "type": "VAE", "links": [11]}], "properties": {"Node name for S&R": "CheckpointLoaderSimple"}, "widgets_values": ["NoobAI-XL-v1.1.safetensors"], "widgets_values_named": {"ckpt_name": "NoobAI-XL-v1.1.safetensors"}}, {"id": 2, "type": "LoraLoaderModelOnly", "pos": [410, 120], "size": [300, 100], "flags": {}, "order": 1, "mode": 0, "inputs": [{"name": "model", "type": "MODEL", "link": 1}], "outputs": [{"name": "MODEL", "type": "MODEL", "links": [3]}], "properties": {"Node name for S&R": "LoraLoaderModelOnly"}, "widgets_values": ["EbiblueIllustriousXL_byKonan.safetensors", 0.8], "widgets_values_named": {"lora_name": "EbiblueIllustriousXL_byKonan.safetensors", "strength_model": 0.8}}, {"id": 3, "type": "ComfySwitchNode", "pos": [740, 120], "size": [300, 100], "flags": {}, "order": 2, "mode": 0, "inputs": [{"name": "on_false", "type": "COMFY_MATCHTYPE_V3", "link": 2}, {"name": "on_true", "type": "COMFY_MATCHTYPE_V3", "link": 3}], "outputs": [{"name": "output", "type": "COMFY_MATCHTYPE_V3", "links": [6]}], "properties": {"Node name for S&R": "ComfySwitchNode"}, "widgets_values": [false], "widgets_values_named": {"switch": false}}, {"id": 4, "type": "CLIPTextEncode", "pos": [410, 340], "size": [300, 100], "flags": {}, "order": 3, "mode": 0, "inputs": [{"name": "clip", "type": "CLIP", "link": 4}], "outputs": [{"name": "CONDITIONING", "type": "CONDITIONING", "links": [7]}], "properties": {"Node name for S&R": "CLIPTextEncode"}, "widgets_values": ["1girl, solo, looking at viewer, upper body, detailed face, soft lighting"], "widgets_values_named": {"text": "1girl, solo, looking at viewer, upper body, detailed face, soft lighting"}}, {"id": 5, "type": "CLIPTextEncode", "pos": [410, 560], "size": [300, 100], "flags": {}, "order": 4, "mode": 0, "inputs": [{"name": "clip", "type": "CLIP", "link": 5}], "outputs": [{"name": "CONDITIONING", "type": "CONDITIONING", "links": [8]}], "properties": {"Node name for S&R": "CLIPTextEncode"}, "widgets_values": ["lowres, bad anatomy, bad hands, extra digits, fewer digits, cropped, worst quality, low quality, jpeg artifacts, signature, watermark, username, blurry"], "widgets_values_named": {"text": "lowres, bad anatomy, bad hands, extra digits, fewer digits, cropped, worst quality, low quality, jpeg artifacts, signature, watermark, username, blurry"}}, {"id": 6, "type": "EmptyLatentImage", "pos": [80, 340], "size": [300, 100], "flags": {}, "order": 5, "mode": 0, "inputs": [], "outputs": [{"name": "LATENT", "type": "LATENT", "links": [9]}], "properties": {"Node name for S&R": "EmptyLatentImage"}, "widgets_values": [832, 1216, 1], "widgets_values_named": {"width": 832, "height": 1216, "batch_size": 1}}, {"id": 7, "type": "KSampler", "pos": [1070, 120], "size": [300, 100], "flags": {}, "order": 6, "mode": 0, "inputs": [{"name": "model", "type": "MODEL", "link": 6}, {"name": "positive", "type": "CONDITIONING", "link": 7}, {"name": "negative", "type": "CONDITIONING", "link": 8}, {"name": "latent_image", "type": "LATENT", "link": 9}], "outputs": [{"name": "LATENT", "type": "LATENT", "links": [10]}], "properties": {"Node name for S&R": "KSampler"}, "widgets_values": [0, "randomize", 28, 5.5, "euler_ancestral", "normal", 1.0], "widgets_values_named": {"seed": 0, "control_after_generate": "randomize", "steps": 28, "cfg": 5.5, "sampler_name": "euler_ancestral", "scheduler": "normal", "denoise": 1.0}}, {"id": 8, "type": "VAEDecode", "pos": [1400, 120], "size": [300, 100], "flags": {}, "order": 7, "mode": 0, "inputs": [{"name": "samples", "type": "LATENT", "link": 10}, {"name": "vae", "type": "VAE", "link": 11}], "outputs": [{"name": "IMAGE", "type": "IMAGE", "links": [12]}], "properties": {"Node name for S&R": "VAEDecode"}, "widgets_values": [], "widgets_values_named": {}}, {"id": 9, "type": "SaveImage", "pos": [1730, 120], "size": [300, 100], "flags": {}, "order": 8, "mode": 0, "inputs": [{"name": "images", "type": "IMAGE", "link": 12}], "outputs": [{"name": "images", "type": "IMAGE", "links": []}], "properties": {"Node name for S&R": "SaveImage"}, "widgets_values": ["app/txt2img-noobai-xl-1.1"], "widgets_values_named": {"filename_prefix": "app/txt2img-noobai-xl-1.1"}}], "links": [[1, 1, 0, 2, 0, "MODEL"], [2, 1, 0, 3, 0, "MODEL"], [3, 2, 0, 3, 1, "MODEL"], [4, 1, 1, 4, 0, "CLIP"], [5, 1, 1, 5, 0, "CLIP"], [6, 3, 0, 7, 0, "COMFY_MATCHTYPE_V3"], [7, 4, 0, 7, 1, "CONDITIONING"], [8, 5, 0, 7, 2, "CONDITIONING"], [9, 6, 0, 7, 3, "LATENT"], [10, 7, 0, 8, 0, "LATENT"], [11, 1, 2, 8, 1, "VAE"], [12, 8, 0, 9, 0, "IMAGE"]], "groups": [], "config": {}, "extra": {"linearMode": true, "linearData": {"inputs": [[4, "text"], [5, "text"], [6, "width"], [6, "height"], [7, "seed"], [7, "steps"], [7, "cfg"], [3, "switch", {"description": "打开才启用下面的 LoRA。默认关闭时不会加载 LoRA 文件，所以 LoRA 没下载好也能正常出图。"}], [2, "lora_name"], [2, "strength_model"]], "outputs": [9]}, "ds": {"scale": 0.75, "offset": [0, 0]}}, "version": 0.4}', '改图-NoobAI-XL.json': '{"id": null, "revision": 0, "last_node_id": 11, "last_link_id": 15, "nodes": [{"id": 1, "type": "CheckpointLoaderSimple", "pos": [80, 120], "size": [300, 100], "flags": {}, "order": 0, "mode": 0, "inputs": [], "outputs": [{"name": "MODEL", "type": "MODEL", "links": [1, 2]}, {"name": "CLIP", "type": "CLIP", "links": [4, 5]}, {"name": "VAE", "type": "VAE", "links": [8, 14]}], "properties": {"Node name for S&R": "CheckpointLoaderSimple"}, "widgets_values": ["NoobAI-XL-v1.1.safetensors"], "widgets_values_named": {"ckpt_name": "NoobAI-XL-v1.1.safetensors"}}, {"id": 2, "type": "LoraLoaderModelOnly", "pos": [410, 120], "size": [300, 100], "flags": {}, "order": 1, "mode": 0, "inputs": [{"name": "model", "type": "MODEL", "link": 1}], "outputs": [{"name": "MODEL", "type": "MODEL", "links": [3]}], "properties": {"Node name for S&R": "LoraLoaderModelOnly"}, "widgets_values": ["EbiblueIllustriousXL_byKonan.safetensors", 0.8], "widgets_values_named": {"lora_name": "EbiblueIllustriousXL_byKonan.safetensors", "strength_model": 0.8}}, {"id": 3, "type": "ComfySwitchNode", "pos": [740, 120], "size": [300, 100], "flags": {}, "order": 2, "mode": 0, "inputs": [{"name": "on_false", "type": "COMFY_MATCHTYPE_V3", "link": 2}, {"name": "on_true", "type": "COMFY_MATCHTYPE_V3", "link": 3}], "outputs": [{"name": "output", "type": "COMFY_MATCHTYPE_V3", "links": [9]}], "properties": {"Node name for S&R": "ComfySwitchNode"}, "widgets_values": [false], "widgets_values_named": {"switch": false}}, {"id": 4, "type": "CLIPTextEncode", "pos": [410, 340], "size": [300, 100], "flags": {}, "order": 3, "mode": 0, "inputs": [{"name": "clip", "type": "CLIP", "link": 4}], "outputs": [{"name": "CONDITIONING", "type": "CONDITIONING", "links": [10]}], "properties": {"Node name for S&R": "CLIPTextEncode"}, "widgets_values": ["1girl, solo, looking at viewer, upper body, detailed face, soft lighting"], "widgets_values_named": {"text": "1girl, solo, looking at viewer, upper body, detailed face, soft lighting"}}, {"id": 5, "type": "CLIPTextEncode", "pos": [410, 560], "size": [300, 100], "flags": {}, "order": 4, "mode": 0, "inputs": [{"name": "clip", "type": "CLIP", "link": 5}], "outputs": [{"name": "CONDITIONING", "type": "CONDITIONING", "links": [11]}], "properties": {"Node name for S&R": "CLIPTextEncode"}, "widgets_values": ["lowres, bad anatomy, bad hands, extra digits, fewer digits, cropped, worst quality, low quality, jpeg artifacts, signature, watermark, username, blurry"], "widgets_values_named": {"text": "lowres, bad anatomy, bad hands, extra digits, fewer digits, cropped, worst quality, low quality, jpeg artifacts, signature, watermark, username, blurry"}}, {"id": 6, "type": "LoadImage", "pos": [80, 340], "size": [300, 100], "flags": {}, "order": 5, "mode": 0, "inputs": [], "outputs": [{"name": "IMAGE", "type": "IMAGE", "links": [6]}, {"name": "MASK", "type": "MASK", "links": []}], "properties": {"Node name for S&R": "LoadImage"}, "widgets_values": ["__REFERENCE__"], "widgets_values_named": {"image": "__REFERENCE__"}}, {"id": 7, "type": "ImageScale", "pos": [410, 780], "size": [300, 100], "flags": {}, "order": 6, "mode": 0, "inputs": [{"name": "image", "type": "IMAGE", "link": 6}], "outputs": [{"name": "IMAGE", "type": "IMAGE", "links": [7]}], "properties": {"Node name for S&R": "ImageScale"}, "widgets_values": ["lanczos", 832, 1216, "disabled"], "widgets_values_named": {"upscale_method": "lanczos", "width": 832, "height": 1216, "crop": "disabled"}}, {"id": 8, "type": "VAEEncode", "pos": [740, 340], "size": [300, 100], "flags": {}, "order": 7, "mode": 0, "inputs": [{"name": "pixels", "type": "IMAGE", "link": 7}, {"name": "vae", "type": "VAE", "link": 8}], "outputs": [{"name": "LATENT", "type": "LATENT", "links": [12]}], "properties": {"Node name for S&R": "VAEEncode"}, "widgets_values": [], "widgets_values_named": {}}, {"id": 9, "type": "KSampler", "pos": [1070, 120], "size": [300, 100], "flags": {}, "order": 8, "mode": 0, "inputs": [{"name": "model", "type": "MODEL", "link": 9}, {"name": "positive", "type": "CONDITIONING", "link": 10}, {"name": "negative", "type": "CONDITIONING", "link": 11}, {"name": "latent_image", "type": "LATENT", "link": 12}], "outputs": [{"name": "LATENT", "type": "LATENT", "links": [13]}], "properties": {"Node name for S&R": "KSampler"}, "widgets_values": [0, "randomize", 28, 5.5, "euler_ancestral", "normal", 0.6], "widgets_values_named": {"seed": 0, "control_after_generate": "randomize", "steps": 28, "cfg": 5.5, "sampler_name": "euler_ancestral", "scheduler": "normal", "denoise": 0.6}}, {"id": 10, "type": "VAEDecode", "pos": [1400, 120], "size": [300, 100], "flags": {}, "order": 9, "mode": 0, "inputs": [{"name": "samples", "type": "LATENT", "link": 13}, {"name": "vae", "type": "VAE", "link": 14}], "outputs": [{"name": "IMAGE", "type": "IMAGE", "links": [15]}], "properties": {"Node name for S&R": "VAEDecode"}, "widgets_values": [], "widgets_values_named": {}}, {"id": 11, "type": "SaveImage", "pos": [1730, 120], "size": [300, 100], "flags": {}, "order": 10, "mode": 0, "inputs": [{"name": "images", "type": "IMAGE", "link": 15}], "outputs": [{"name": "images", "type": "IMAGE", "links": []}], "properties": {"Node name for S&R": "SaveImage"}, "widgets_values": ["app/img2img-noobai-xl-1.1"], "widgets_values_named": {"filename_prefix": "app/img2img-noobai-xl-1.1"}}], "links": [[1, 1, 0, 2, 0, "MODEL"], [2, 1, 0, 3, 0, "MODEL"], [3, 2, 0, 3, 1, "MODEL"], [4, 1, 1, 4, 0, "CLIP"], [5, 1, 1, 5, 0, "CLIP"], [6, 6, 0, 7, 0, "IMAGE"], [7, 7, 0, 8, 0, "IMAGE"], [8, 1, 2, 8, 1, "VAE"], [9, 3, 0, 9, 0, "COMFY_MATCHTYPE_V3"], [10, 4, 0, 9, 1, "CONDITIONING"], [11, 5, 0, 9, 2, "CONDITIONING"], [12, 8, 0, 9, 3, "LATENT"], [13, 9, 0, 10, 0, "LATENT"], [14, 1, 2, 10, 1, "VAE"], [15, 10, 0, 11, 0, "IMAGE"]], "groups": [], "config": {}, "extra": {"linearMode": true, "linearData": {"inputs": [[4, "text"], [5, "text"], [6, "image"], [9, "denoise"], [7, "width"], [7, "height"], [9, "seed"], [9, "steps"], [9, "cfg"], [3, "switch", {"description": "打开才启用下面的 LoRA。默认关闭时不会加载 LoRA 文件，所以 LoRA 没下载好也能正常出图。"}], [2, "lora_name"], [2, "strength_model"]], "outputs": [11]}, "ds": {"scale": 0.75, "offset": [0, 0]}}, "version": 0.4}', '文生图-Animagine-XL.json': '{"id": null, "revision": 0, "last_node_id": 9, "last_link_id": 12, "nodes": [{"id": 1, "type": "CheckpointLoaderSimple", "pos": [80, 120], "size": [300, 100], "flags": {}, "order": 0, "mode": 0, "inputs": [], "outputs": [{"name": "MODEL", "type": "MODEL", "links": [1, 2]}, {"name": "CLIP", "type": "CLIP", "links": [4, 5]}, {"name": "VAE", "type": "VAE", "links": [11]}], "properties": {"Node name for S&R": "CheckpointLoaderSimple"}, "widgets_values": ["animagine-xl-4.0.safetensors"], "widgets_values_named": {"ckpt_name": "animagine-xl-4.0.safetensors"}}, {"id": 2, "type": "LoraLoaderModelOnly", "pos": [410, 120], "size": [300, 100], "flags": {}, "order": 1, "mode": 0, "inputs": [{"name": "model", "type": "MODEL", "link": 1}], "outputs": [{"name": "MODEL", "type": "MODEL", "links": [3]}], "properties": {"Node name for S&R": "LoraLoaderModelOnly"}, "widgets_values": ["EbiblueIllustriousXL_byKonan.safetensors", 0.8], "widgets_values_named": {"lora_name": "EbiblueIllustriousXL_byKonan.safetensors", "strength_model": 0.8}}, {"id": 3, "type": "ComfySwitchNode", "pos": [740, 120], "size": [300, 100], "flags": {}, "order": 2, "mode": 0, "inputs": [{"name": "on_false", "type": "COMFY_MATCHTYPE_V3", "link": 2}, {"name": "on_true", "type": "COMFY_MATCHTYPE_V3", "link": 3}], "outputs": [{"name": "output", "type": "COMFY_MATCHTYPE_V3", "links": [6]}], "properties": {"Node name for S&R": "ComfySwitchNode"}, "widgets_values": [false], "widgets_values_named": {"switch": false}}, {"id": 4, "type": "CLIPTextEncode", "pos": [410, 340], "size": [300, 100], "flags": {}, "order": 3, "mode": 0, "inputs": [{"name": "clip", "type": "CLIP", "link": 4}], "outputs": [{"name": "CONDITIONING", "type": "CONDITIONING", "links": [7]}], "properties": {"Node name for S&R": "CLIPTextEncode"}, "widgets_values": ["1girl, solo, looking at viewer, upper body, detailed face, soft lighting"], "widgets_values_named": {"text": "1girl, solo, looking at viewer, upper body, detailed face, soft lighting"}}, {"id": 5, "type": "CLIPTextEncode", "pos": [410, 560], "size": [300, 100], "flags": {}, "order": 4, "mode": 0, "inputs": [{"name": "clip", "type": "CLIP", "link": 5}], "outputs": [{"name": "CONDITIONING", "type": "CONDITIONING", "links": [8]}], "properties": {"Node name for S&R": "CLIPTextEncode"}, "widgets_values": ["lowres, bad anatomy, bad hands, extra digits, fewer digits, cropped, worst quality, low quality, jpeg artifacts, signature, watermark, username, blurry"], "widgets_values_named": {"text": "lowres, bad anatomy, bad hands, extra digits, fewer digits, cropped, worst quality, low quality, jpeg artifacts, signature, watermark, username, blurry"}}, {"id": 6, "type": "EmptyLatentImage", "pos": [80, 340], "size": [300, 100], "flags": {}, "order": 5, "mode": 0, "inputs": [], "outputs": [{"name": "LATENT", "type": "LATENT", "links": [9]}], "properties": {"Node name for S&R": "EmptyLatentImage"}, "widgets_values": [832, 1216, 1], "widgets_values_named": {"width": 832, "height": 1216, "batch_size": 1}}, {"id": 7, "type": "KSampler", "pos": [1070, 120], "size": [300, 100], "flags": {}, "order": 6, "mode": 0, "inputs": [{"name": "model", "type": "MODEL", "link": 6}, {"name": "positive", "type": "CONDITIONING", "link": 7}, {"name": "negative", "type": "CONDITIONING", "link": 8}, {"name": "latent_image", "type": "LATENT", "link": 9}], "outputs": [{"name": "LATENT", "type": "LATENT", "links": [10]}], "properties": {"Node name for S&R": "KSampler"}, "widgets_values": [0, "randomize", 28, 5.5, "euler_ancestral", "normal", 1.0], "widgets_values_named": {"seed": 0, "control_after_generate": "randomize", "steps": 28, "cfg": 5.5, "sampler_name": "euler_ancestral", "scheduler": "normal", "denoise": 1.0}}, {"id": 8, "type": "VAEDecode", "pos": [1400, 120], "size": [300, 100], "flags": {}, "order": 7, "mode": 0, "inputs": [{"name": "samples", "type": "LATENT", "link": 10}, {"name": "vae", "type": "VAE", "link": 11}], "outputs": [{"name": "IMAGE", "type": "IMAGE", "links": [12]}], "properties": {"Node name for S&R": "VAEDecode"}, "widgets_values": [], "widgets_values_named": {}}, {"id": 9, "type": "SaveImage", "pos": [1730, 120], "size": [300, 100], "flags": {}, "order": 8, "mode": 0, "inputs": [{"name": "images", "type": "IMAGE", "link": 12}], "outputs": [{"name": "images", "type": "IMAGE", "links": []}], "properties": {"Node name for S&R": "SaveImage"}, "widgets_values": ["app/txt2img-animagine-xl-4.0"], "widgets_values_named": {"filename_prefix": "app/txt2img-animagine-xl-4.0"}}], "links": [[1, 1, 0, 2, 0, "MODEL"], [2, 1, 0, 3, 0, "MODEL"], [3, 2, 0, 3, 1, "MODEL"], [4, 1, 1, 4, 0, "CLIP"], [5, 1, 1, 5, 0, "CLIP"], [6, 3, 0, 7, 0, "COMFY_MATCHTYPE_V3"], [7, 4, 0, 7, 1, "CONDITIONING"], [8, 5, 0, 7, 2, "CONDITIONING"], [9, 6, 0, 7, 3, "LATENT"], [10, 7, 0, 8, 0, "LATENT"], [11, 1, 2, 8, 1, "VAE"], [12, 8, 0, 9, 0, "IMAGE"]], "groups": [], "config": {}, "extra": {"linearMode": true, "linearData": {"inputs": [[4, "text"], [5, "text"], [6, "width"], [6, "height"], [7, "seed"], [7, "steps"], [7, "cfg"], [3, "switch", {"description": "打开才启用下面的 LoRA。默认关闭时不会加载 LoRA 文件，所以 LoRA 没下载好也能正常出图。"}], [2, "lora_name"], [2, "strength_model"]], "outputs": [9]}, "ds": {"scale": 0.75, "offset": [0, 0]}}, "version": 0.4}', '改图-Animagine-XL.json': '{"id": null, "revision": 0, "last_node_id": 11, "last_link_id": 15, "nodes": [{"id": 1, "type": "CheckpointLoaderSimple", "pos": [80, 120], "size": [300, 100], "flags": {}, "order": 0, "mode": 0, "inputs": [], "outputs": [{"name": "MODEL", "type": "MODEL", "links": [1, 2]}, {"name": "CLIP", "type": "CLIP", "links": [4, 5]}, {"name": "VAE", "type": "VAE", "links": [8, 14]}], "properties": {"Node name for S&R": "CheckpointLoaderSimple"}, "widgets_values": ["animagine-xl-4.0.safetensors"], "widgets_values_named": {"ckpt_name": "animagine-xl-4.0.safetensors"}}, {"id": 2, "type": "LoraLoaderModelOnly", "pos": [410, 120], "size": [300, 100], "flags": {}, "order": 1, "mode": 0, "inputs": [{"name": "model", "type": "MODEL", "link": 1}], "outputs": [{"name": "MODEL", "type": "MODEL", "links": [3]}], "properties": {"Node name for S&R": "LoraLoaderModelOnly"}, "widgets_values": ["EbiblueIllustriousXL_byKonan.safetensors", 0.8], "widgets_values_named": {"lora_name": "EbiblueIllustriousXL_byKonan.safetensors", "strength_model": 0.8}}, {"id": 3, "type": "ComfySwitchNode", "pos": [740, 120], "size": [300, 100], "flags": {}, "order": 2, "mode": 0, "inputs": [{"name": "on_false", "type": "COMFY_MATCHTYPE_V3", "link": 2}, {"name": "on_true", "type": "COMFY_MATCHTYPE_V3", "link": 3}], "outputs": [{"name": "output", "type": "COMFY_MATCHTYPE_V3", "links": [9]}], "properties": {"Node name for S&R": "ComfySwitchNode"}, "widgets_values": [false], "widgets_values_named": {"switch": false}}, {"id": 4, "type": "CLIPTextEncode", "pos": [410, 340], "size": [300, 100], "flags": {}, "order": 3, "mode": 0, "inputs": [{"name": "clip", "type": "CLIP", "link": 4}], "outputs": [{"name": "CONDITIONING", "type": "CONDITIONING", "links": [10]}], "properties": {"Node name for S&R": "CLIPTextEncode"}, "widgets_values": ["1girl, solo, looking at viewer, upper body, detailed face, soft lighting"], "widgets_values_named": {"text": "1girl, solo, looking at viewer, upper body, detailed face, soft lighting"}}, {"id": 5, "type": "CLIPTextEncode", "pos": [410, 560], "size": [300, 100], "flags": {}, "order": 4, "mode": 0, "inputs": [{"name": "clip", "type": "CLIP", "link": 5}], "outputs": [{"name": "CONDITIONING", "type": "CONDITIONING", "links": [11]}], "properties": {"Node name for S&R": "CLIPTextEncode"}, "widgets_values": ["lowres, bad anatomy, bad hands, extra digits, fewer digits, cropped, worst quality, low quality, jpeg artifacts, signature, watermark, username, blurry"], "widgets_values_named": {"text": "lowres, bad anatomy, bad hands, extra digits, fewer digits, cropped, worst quality, low quality, jpeg artifacts, signature, watermark, username, blurry"}}, {"id": 6, "type": "LoadImage", "pos": [80, 340], "size": [300, 100], "flags": {}, "order": 5, "mode": 0, "inputs": [], "outputs": [{"name": "IMAGE", "type": "IMAGE", "links": [6]}, {"name": "MASK", "type": "MASK", "links": []}], "properties": {"Node name for S&R": "LoadImage"}, "widgets_values": ["__REFERENCE__"], "widgets_values_named": {"image": "__REFERENCE__"}}, {"id": 7, "type": "ImageScale", "pos": [410, 780], "size": [300, 100], "flags": {}, "order": 6, "mode": 0, "inputs": [{"name": "image", "type": "IMAGE", "link": 6}], "outputs": [{"name": "IMAGE", "type": "IMAGE", "links": [7]}], "properties": {"Node name for S&R": "ImageScale"}, "widgets_values": ["lanczos", 832, 1216, "disabled"], "widgets_values_named": {"upscale_method": "lanczos", "width": 832, "height": 1216, "crop": "disabled"}}, {"id": 8, "type": "VAEEncode", "pos": [740, 340], "size": [300, 100], "flags": {}, "order": 7, "mode": 0, "inputs": [{"name": "pixels", "type": "IMAGE", "link": 7}, {"name": "vae", "type": "VAE", "link": 8}], "outputs": [{"name": "LATENT", "type": "LATENT", "links": [12]}], "properties": {"Node name for S&R": "VAEEncode"}, "widgets_values": [], "widgets_values_named": {}}, {"id": 9, "type": "KSampler", "pos": [1070, 120], "size": [300, 100], "flags": {}, "order": 8, "mode": 0, "inputs": [{"name": "model", "type": "MODEL", "link": 9}, {"name": "positive", "type": "CONDITIONING", "link": 10}, {"name": "negative", "type": "CONDITIONING", "link": 11}, {"name": "latent_image", "type": "LATENT", "link": 12}], "outputs": [{"name": "LATENT", "type": "LATENT", "links": [13]}], "properties": {"Node name for S&R": "KSampler"}, "widgets_values": [0, "randomize", 28, 5.5, "euler_ancestral", "normal", 0.6], "widgets_values_named": {"seed": 0, "control_after_generate": "randomize", "steps": 28, "cfg": 5.5, "sampler_name": "euler_ancestral", "scheduler": "normal", "denoise": 0.6}}, {"id": 10, "type": "VAEDecode", "pos": [1400, 120], "size": [300, 100], "flags": {}, "order": 9, "mode": 0, "inputs": [{"name": "samples", "type": "LATENT", "link": 13}, {"name": "vae", "type": "VAE", "link": 14}], "outputs": [{"name": "IMAGE", "type": "IMAGE", "links": [15]}], "properties": {"Node name for S&R": "VAEDecode"}, "widgets_values": [], "widgets_values_named": {}}, {"id": 11, "type": "SaveImage", "pos": [1730, 120], "size": [300, 100], "flags": {}, "order": 10, "mode": 0, "inputs": [{"name": "images", "type": "IMAGE", "link": 15}], "outputs": [{"name": "images", "type": "IMAGE", "links": []}], "properties": {"Node name for S&R": "SaveImage"}, "widgets_values": ["app/img2img-animagine-xl-4.0"], "widgets_values_named": {"filename_prefix": "app/img2img-animagine-xl-4.0"}}], "links": [[1, 1, 0, 2, 0, "MODEL"], [2, 1, 0, 3, 0, "MODEL"], [3, 2, 0, 3, 1, "MODEL"], [4, 1, 1, 4, 0, "CLIP"], [5, 1, 1, 5, 0, "CLIP"], [6, 6, 0, 7, 0, "IMAGE"], [7, 7, 0, 8, 0, "IMAGE"], [8, 1, 2, 8, 1, "VAE"], [9, 3, 0, 9, 0, "COMFY_MATCHTYPE_V3"], [10, 4, 0, 9, 1, "CONDITIONING"], [11, 5, 0, 9, 2, "CONDITIONING"], [12, 8, 0, 9, 3, "LATENT"], [13, 9, 0, 10, 0, "LATENT"], [14, 1, 2, 10, 1, "VAE"], [15, 10, 0, 11, 0, "IMAGE"]], "groups": [], "config": {}, "extra": {"linearMode": true, "linearData": {"inputs": [[4, "text"], [5, "text"], [6, "image"], [9, "denoise"], [7, "width"], [7, "height"], [9, "seed"], [9, "steps"], [9, "cfg"], [3, "switch", {"description": "打开才启用下面的 LoRA。默认关闭时不会加载 LoRA 文件，所以 LoRA 没下载好也能正常出图。"}], [2, "lora_name"], [2, "strength_model"]], "outputs": [11]}, "ds": {"scale": 0.75, "offset": [0, 0]}}, "version": 0.4}', '文生图-RealVisXL.json': '{"id": null, "revision": 0, "last_node_id": 9, "last_link_id": 12, "nodes": [{"id": 1, "type": "CheckpointLoaderSimple", "pos": [80, 120], "size": [300, 100], "flags": {}, "order": 0, "mode": 0, "inputs": [], "outputs": [{"name": "MODEL", "type": "MODEL", "links": [1, 2]}, {"name": "CLIP", "type": "CLIP", "links": [4, 5]}, {"name": "VAE", "type": "VAE", "links": [11]}], "properties": {"Node name for S&R": "CheckpointLoaderSimple"}, "widgets_values": ["RealVisXL_V5.0_fp16.safetensors"], "widgets_values_named": {"ckpt_name": "RealVisXL_V5.0_fp16.safetensors"}}, {"id": 2, "type": "LoraLoaderModelOnly", "pos": [410, 120], "size": [300, 100], "flags": {}, "order": 1, "mode": 0, "inputs": [{"name": "model", "type": "MODEL", "link": 1}], "outputs": [{"name": "MODEL", "type": "MODEL", "links": [3]}], "properties": {"Node name for S&R": "LoraLoaderModelOnly"}, "widgets_values": ["EbiblueIllustriousXL_byKonan.safetensors", 0.8], "widgets_values_named": {"lora_name": "EbiblueIllustriousXL_byKonan.safetensors", "strength_model": 0.8}}, {"id": 3, "type": "ComfySwitchNode", "pos": [740, 120], "size": [300, 100], "flags": {}, "order": 2, "mode": 0, "inputs": [{"name": "on_false", "type": "COMFY_MATCHTYPE_V3", "link": 2}, {"name": "on_true", "type": "COMFY_MATCHTYPE_V3", "link": 3}], "outputs": [{"name": "output", "type": "COMFY_MATCHTYPE_V3", "links": [6]}], "properties": {"Node name for S&R": "ComfySwitchNode"}, "widgets_values": [false], "widgets_values_named": {"switch": false}}, {"id": 4, "type": "CLIPTextEncode", "pos": [410, 340], "size": [300, 100], "flags": {}, "order": 3, "mode": 0, "inputs": [{"name": "clip", "type": "CLIP", "link": 4}], "outputs": [{"name": "CONDITIONING", "type": "CONDITIONING", "links": [7]}], "properties": {"Node name for S&R": "CLIPTextEncode"}, "widgets_values": ["1girl, solo, looking at viewer, upper body, detailed face, soft lighting"], "widgets_values_named": {"text": "1girl, solo, looking at viewer, upper body, detailed face, soft lighting"}}, {"id": 5, "type": "CLIPTextEncode", "pos": [410, 560], "size": [300, 100], "flags": {}, "order": 4, "mode": 0, "inputs": [{"name": "clip", "type": "CLIP", "link": 5}], "outputs": [{"name": "CONDITIONING", "type": "CONDITIONING", "links": [8]}], "properties": {"Node name for S&R": "CLIPTextEncode"}, "widgets_values": ["lowres, bad anatomy, bad hands, extra digits, fewer digits, cropped, worst quality, low quality, jpeg artifacts, signature, watermark, username, blurry"], "widgets_values_named": {"text": "lowres, bad anatomy, bad hands, extra digits, fewer digits, cropped, worst quality, low quality, jpeg artifacts, signature, watermark, username, blurry"}}, {"id": 6, "type": "EmptyLatentImage", "pos": [80, 340], "size": [300, 100], "flags": {}, "order": 5, "mode": 0, "inputs": [], "outputs": [{"name": "LATENT", "type": "LATENT", "links": [9]}], "properties": {"Node name for S&R": "EmptyLatentImage"}, "widgets_values": [832, 1216, 1], "widgets_values_named": {"width": 832, "height": 1216, "batch_size": 1}}, {"id": 7, "type": "KSampler", "pos": [1070, 120], "size": [300, 100], "flags": {}, "order": 6, "mode": 0, "inputs": [{"name": "model", "type": "MODEL", "link": 6}, {"name": "positive", "type": "CONDITIONING", "link": 7}, {"name": "negative", "type": "CONDITIONING", "link": 8}, {"name": "latent_image", "type": "LATENT", "link": 9}], "outputs": [{"name": "LATENT", "type": "LATENT", "links": [10]}], "properties": {"Node name for S&R": "KSampler"}, "widgets_values": [0, "randomize", 30, 5.0, "euler_ancestral", "normal", 1.0], "widgets_values_named": {"seed": 0, "control_after_generate": "randomize", "steps": 30, "cfg": 5.0, "sampler_name": "euler_ancestral", "scheduler": "normal", "denoise": 1.0}}, {"id": 8, "type": "VAEDecode", "pos": [1400, 120], "size": [300, 100], "flags": {}, "order": 7, "mode": 0, "inputs": [{"name": "samples", "type": "LATENT", "link": 10}, {"name": "vae", "type": "VAE", "link": 11}], "outputs": [{"name": "IMAGE", "type": "IMAGE", "links": [12]}], "properties": {"Node name for S&R": "VAEDecode"}, "widgets_values": [], "widgets_values_named": {}}, {"id": 9, "type": "SaveImage", "pos": [1730, 120], "size": [300, 100], "flags": {}, "order": 8, "mode": 0, "inputs": [{"name": "images", "type": "IMAGE", "link": 12}], "outputs": [{"name": "images", "type": "IMAGE", "links": []}], "properties": {"Node name for S&R": "SaveImage"}, "widgets_values": ["app/txt2img-realvisxl-v5.0"], "widgets_values_named": {"filename_prefix": "app/txt2img-realvisxl-v5.0"}}], "links": [[1, 1, 0, 2, 0, "MODEL"], [2, 1, 0, 3, 0, "MODEL"], [3, 2, 0, 3, 1, "MODEL"], [4, 1, 1, 4, 0, "CLIP"], [5, 1, 1, 5, 0, "CLIP"], [6, 3, 0, 7, 0, "COMFY_MATCHTYPE_V3"], [7, 4, 0, 7, 1, "CONDITIONING"], [8, 5, 0, 7, 2, "CONDITIONING"], [9, 6, 0, 7, 3, "LATENT"], [10, 7, 0, 8, 0, "LATENT"], [11, 1, 2, 8, 1, "VAE"], [12, 8, 0, 9, 0, "IMAGE"]], "groups": [], "config": {}, "extra": {"linearMode": true, "linearData": {"inputs": [[4, "text"], [5, "text"], [6, "width"], [6, "height"], [7, "seed"], [7, "steps"], [7, "cfg"], [3, "switch", {"description": "打开才启用下面的 LoRA。默认关闭时不会加载 LoRA 文件，所以 LoRA 没下载好也能正常出图。"}], [2, "lora_name"], [2, "strength_model"]], "outputs": [9]}, "ds": {"scale": 0.75, "offset": [0, 0]}}, "version": 0.4}', '改图-RealVisXL.json': '{"id": null, "revision": 0, "last_node_id": 11, "last_link_id": 15, "nodes": [{"id": 1, "type": "CheckpointLoaderSimple", "pos": [80, 120], "size": [300, 100], "flags": {}, "order": 0, "mode": 0, "inputs": [], "outputs": [{"name": "MODEL", "type": "MODEL", "links": [1, 2]}, {"name": "CLIP", "type": "CLIP", "links": [4, 5]}, {"name": "VAE", "type": "VAE", "links": [8, 14]}], "properties": {"Node name for S&R": "CheckpointLoaderSimple"}, "widgets_values": ["RealVisXL_V5.0_fp16.safetensors"], "widgets_values_named": {"ckpt_name": "RealVisXL_V5.0_fp16.safetensors"}}, {"id": 2, "type": "LoraLoaderModelOnly", "pos": [410, 120], "size": [300, 100], "flags": {}, "order": 1, "mode": 0, "inputs": [{"name": "model", "type": "MODEL", "link": 1}], "outputs": [{"name": "MODEL", "type": "MODEL", "links": [3]}], "properties": {"Node name for S&R": "LoraLoaderModelOnly"}, "widgets_values": ["EbiblueIllustriousXL_byKonan.safetensors", 0.8], "widgets_values_named": {"lora_name": "EbiblueIllustriousXL_byKonan.safetensors", "strength_model": 0.8}}, {"id": 3, "type": "ComfySwitchNode", "pos": [740, 120], "size": [300, 100], "flags": {}, "order": 2, "mode": 0, "inputs": [{"name": "on_false", "type": "COMFY_MATCHTYPE_V3", "link": 2}, {"name": "on_true", "type": "COMFY_MATCHTYPE_V3", "link": 3}], "outputs": [{"name": "output", "type": "COMFY_MATCHTYPE_V3", "links": [9]}], "properties": {"Node name for S&R": "ComfySwitchNode"}, "widgets_values": [false], "widgets_values_named": {"switch": false}}, {"id": 4, "type": "CLIPTextEncode", "pos": [410, 340], "size": [300, 100], "flags": {}, "order": 3, "mode": 0, "inputs": [{"name": "clip", "type": "CLIP", "link": 4}], "outputs": [{"name": "CONDITIONING", "type": "CONDITIONING", "links": [10]}], "properties": {"Node name for S&R": "CLIPTextEncode"}, "widgets_values": ["1girl, solo, looking at viewer, upper body, detailed face, soft lighting"], "widgets_values_named": {"text": "1girl, solo, looking at viewer, upper body, detailed face, soft lighting"}}, {"id": 5, "type": "CLIPTextEncode", "pos": [410, 560], "size": [300, 100], "flags": {}, "order": 4, "mode": 0, "inputs": [{"name": "clip", "type": "CLIP", "link": 5}], "outputs": [{"name": "CONDITIONING", "type": "CONDITIONING", "links": [11]}], "properties": {"Node name for S&R": "CLIPTextEncode"}, "widgets_values": ["lowres, bad anatomy, bad hands, extra digits, fewer digits, cropped, worst quality, low quality, jpeg artifacts, signature, watermark, username, blurry"], "widgets_values_named": {"text": "lowres, bad anatomy, bad hands, extra digits, fewer digits, cropped, worst quality, low quality, jpeg artifacts, signature, watermark, username, blurry"}}, {"id": 6, "type": "LoadImage", "pos": [80, 340], "size": [300, 100], "flags": {}, "order": 5, "mode": 0, "inputs": [], "outputs": [{"name": "IMAGE", "type": "IMAGE", "links": [6]}, {"name": "MASK", "type": "MASK", "links": []}], "properties": {"Node name for S&R": "LoadImage"}, "widgets_values": ["__REFERENCE__"], "widgets_values_named": {"image": "__REFERENCE__"}}, {"id": 7, "type": "ImageScale", "pos": [410, 780], "size": [300, 100], "flags": {}, "order": 6, "mode": 0, "inputs": [{"name": "image", "type": "IMAGE", "link": 6}], "outputs": [{"name": "IMAGE", "type": "IMAGE", "links": [7]}], "properties": {"Node name for S&R": "ImageScale"}, "widgets_values": ["lanczos", 832, 1216, "disabled"], "widgets_values_named": {"upscale_method": "lanczos", "width": 832, "height": 1216, "crop": "disabled"}}, {"id": 8, "type": "VAEEncode", "pos": [740, 340], "size": [300, 100], "flags": {}, "order": 7, "mode": 0, "inputs": [{"name": "pixels", "type": "IMAGE", "link": 7}, {"name": "vae", "type": "VAE", "link": 8}], "outputs": [{"name": "LATENT", "type": "LATENT", "links": [12]}], "properties": {"Node name for S&R": "VAEEncode"}, "widgets_values": [], "widgets_values_named": {}}, {"id": 9, "type": "KSampler", "pos": [1070, 120], "size": [300, 100], "flags": {}, "order": 8, "mode": 0, "inputs": [{"name": "model", "type": "MODEL", "link": 9}, {"name": "positive", "type": "CONDITIONING", "link": 10}, {"name": "negative", "type": "CONDITIONING", "link": 11}, {"name": "latent_image", "type": "LATENT", "link": 12}], "outputs": [{"name": "LATENT", "type": "LATENT", "links": [13]}], "properties": {"Node name for S&R": "KSampler"}, "widgets_values": [0, "randomize", 30, 5.0, "euler_ancestral", "normal", 0.6], "widgets_values_named": {"seed": 0, "control_after_generate": "randomize", "steps": 30, "cfg": 5.0, "sampler_name": "euler_ancestral", "scheduler": "normal", "denoise": 0.6}}, {"id": 10, "type": "VAEDecode", "pos": [1400, 120], "size": [300, 100], "flags": {}, "order": 9, "mode": 0, "inputs": [{"name": "samples", "type": "LATENT", "link": 13}, {"name": "vae", "type": "VAE", "link": 14}], "outputs": [{"name": "IMAGE", "type": "IMAGE", "links": [15]}], "properties": {"Node name for S&R": "VAEDecode"}, "widgets_values": [], "widgets_values_named": {}}, {"id": 11, "type": "SaveImage", "pos": [1730, 120], "size": [300, 100], "flags": {}, "order": 10, "mode": 0, "inputs": [{"name": "images", "type": "IMAGE", "link": 15}], "outputs": [{"name": "images", "type": "IMAGE", "links": []}], "properties": {"Node name for S&R": "SaveImage"}, "widgets_values": ["app/img2img-realvisxl-v5.0"], "widgets_values_named": {"filename_prefix": "app/img2img-realvisxl-v5.0"}}], "links": [[1, 1, 0, 2, 0, "MODEL"], [2, 1, 0, 3, 0, "MODEL"], [3, 2, 0, 3, 1, "MODEL"], [4, 1, 1, 4, 0, "CLIP"], [5, 1, 1, 5, 0, "CLIP"], [6, 6, 0, 7, 0, "IMAGE"], [7, 7, 0, 8, 0, "IMAGE"], [8, 1, 2, 8, 1, "VAE"], [9, 3, 0, 9, 0, "COMFY_MATCHTYPE_V3"], [10, 4, 0, 9, 1, "CONDITIONING"], [11, 5, 0, 9, 2, "CONDITIONING"], [12, 8, 0, 9, 3, "LATENT"], [13, 9, 0, 10, 0, "LATENT"], [14, 1, 2, 10, 1, "VAE"], [15, 10, 0, 11, 0, "IMAGE"]], "groups": [], "config": {}, "extra": {"linearMode": true, "linearData": {"inputs": [[4, "text"], [5, "text"], [6, "image"], [9, "denoise"], [7, "width"], [7, "height"], [9, "seed"], [9, "steps"], [9, "cfg"], [3, "switch", {"description": "打开才启用下面的 LoRA。默认关闭时不会加载 LoRA 文件，所以 LoRA 没下载好也能正常出图。"}], [2, "lora_name"], [2, "strength_model"]], "outputs": [11]}, "ds": {"scale": 0.75, "offset": [0, 0]}}, "version": 0.4}', '文生图-FLUX.1-dev.json': '{"id": null, "revision": 0, "last_node_id": 10, "last_link_id": 10, "nodes": [{"id": 1, "type": "UnetLoaderGGUF", "pos": [80, 120], "size": [300, 100], "flags": {}, "order": 0, "mode": 0, "inputs": [], "outputs": [{"name": "MODEL", "type": "MODEL", "links": [4]}], "properties": {"Node name for S&R": "UnetLoaderGGUF"}, "widgets_values": ["flux1-dev-Q5_K_S.gguf"], "widgets_values_named": {"unet_name": "flux1-dev-Q5_K_S.gguf"}}, {"id": 2, "type": "DualCLIPLoaderGGUF", "pos": [80, 340], "size": [300, 100], "flags": {}, "order": 1, "mode": 0, "inputs": [], "outputs": [{"name": "CLIP", "type": "CLIP", "links": [1, 2]}], "properties": {"Node name for S&R": "DualCLIPLoaderGGUF"}, "widgets_values": ["clip_l.safetensors", "t5-v1_1-xxl-encoder-Q5_K_M.gguf", "flux"], "widgets_values_named": {"clip_name1": "clip_l.safetensors", "clip_name2": "t5-v1_1-xxl-encoder-Q5_K_M.gguf", "type": "flux"}}, {"id": 3, "type": "VAELoader", "pos": [80, 560], "size": [300, 100], "flags": {}, "order": 2, "mode": 0, "inputs": [], "outputs": [{"name": "VAE", "type": "VAE", "links": [9]}], "properties": {"Node name for S&R": "VAELoader"}, "widgets_values": ["ae.safetensors"], "widgets_values_named": {"vae_name": "ae.safetensors"}}, {"id": 4, "type": "CLIPTextEncode", "pos": [410, 120], "size": [300, 100], "flags": {}, "order": 3, "mode": 0, "inputs": [{"name": "clip", "type": "CLIP", "link": 1}], "outputs": [{"name": "CONDITIONING", "type": "CONDITIONING", "links": [3]}], "properties": {"Node name for S&R": "CLIPTextEncode"}, "widgets_values": ["1girl, solo, looking at viewer, upper body, detailed face, soft lighting"], "widgets_values_named": {"text": "1girl, solo, looking at viewer, upper body, detailed face, soft lighting"}}, {"id": 5, "type": "CLIPTextEncode", "pos": [410, 340], "size": [300, 100], "flags": {}, "order": 4, "mode": 0, "inputs": [{"name": "clip", "type": "CLIP", "link": 2}], "outputs": [{"name": "CONDITIONING", "type": "CONDITIONING", "links": [6]}], "properties": {"Node name for S&R": "CLIPTextEncode"}, "widgets_values": ["lowres, bad anatomy, bad hands, extra digits, fewer digits, cropped, worst quality, low quality, jpeg artifacts, signature, watermark, username, blurry"], "widgets_values_named": {"text": "lowres, bad anatomy, bad hands, extra digits, fewer digits, cropped, worst quality, low quality, jpeg artifacts, signature, watermark, username, blurry"}}, {"id": 6, "type": "FluxGuidance", "pos": [740, 120], "size": [300, 100], "flags": {}, "order": 5, "mode": 0, "inputs": [{"name": "conditioning", "type": "CONDITIONING", "link": 3}], "outputs": [{"name": "CONDITIONING", "type": "CONDITIONING", "links": [5]}], "properties": {"Node name for S&R": "FluxGuidance"}, "widgets_values": [3.5], "widgets_values_named": {"guidance": 3.5}}, {"id": 7, "type": "EmptySD3LatentImage", "pos": [80, 780], "size": [300, 100], "flags": {}, "order": 6, "mode": 0, "inputs": [], "outputs": [{"name": "LATENT", "type": "LATENT", "links": [7]}], "properties": {"Node name for S&R": "EmptySD3LatentImage"}, "widgets_values": [1024, 1024, 1], "widgets_values_named": {"width": 1024, "height": 1024, "batch_size": 1}}, {"id": 8, "type": "KSampler", "pos": [1070, 120], "size": [300, 100], "flags": {}, "order": 7, "mode": 0, "inputs": [{"name": "model", "type": "MODEL", "link": 4}, {"name": "positive", "type": "CONDITIONING", "link": 5}, {"name": "negative", "type": "CONDITIONING", "link": 6}, {"name": "latent_image", "type": "LATENT", "link": 7}], "outputs": [{"name": "LATENT", "type": "LATENT", "links": [8]}], "properties": {"Node name for S&R": "KSampler"}, "widgets_values": [0, "randomize", 20, 1.0, "euler_ancestral", "normal", 1.0], "widgets_values_named": {"seed": 0, "control_after_generate": "randomize", "steps": 20, "cfg": 1.0, "sampler_name": "euler_ancestral", "scheduler": "normal", "denoise": 1.0}}, {"id": 9, "type": "VAEDecode", "pos": [1400, 120], "size": [300, 100], "flags": {}, "order": 8, "mode": 0, "inputs": [{"name": "samples", "type": "LATENT", "link": 8}, {"name": "vae", "type": "VAE", "link": 9}], "outputs": [{"name": "IMAGE", "type": "IMAGE", "links": [10]}], "properties": {"Node name for S&R": "VAEDecode"}, "widgets_values": [], "widgets_values_named": {}}, {"id": 10, "type": "SaveImage", "pos": [1730, 120], "size": [300, 100], "flags": {}, "order": 9, "mode": 0, "inputs": [{"name": "images", "type": "IMAGE", "link": 10}], "outputs": [{"name": "images", "type": "IMAGE", "links": []}], "properties": {"Node name for S&R": "SaveImage"}, "widgets_values": ["app/txt2img-flux1-dev-q5ks"], "widgets_values_named": {"filename_prefix": "app/txt2img-flux1-dev-q5ks"}}], "links": [[1, 2, 0, 4, 0, "CLIP"], [2, 2, 0, 5, 0, "CLIP"], [3, 4, 0, 6, 0, "CONDITIONING"], [4, 1, 0, 8, 0, "MODEL"], [5, 6, 0, 8, 1, "CONDITIONING"], [6, 5, 0, 8, 2, "CONDITIONING"], [7, 7, 0, 8, 3, "LATENT"], [8, 8, 0, 9, 0, "LATENT"], [9, 3, 0, 9, 1, "VAE"], [10, 9, 0, 10, 0, "IMAGE"]], "groups": [], "config": {}, "extra": {"linearMode": true, "linearData": {"inputs": [[4, "text"], [7, "width"], [7, "height"], [8, "seed"], [8, "steps"]], "outputs": [10]}, "ds": {"scale": 0.75, "offset": [0, 0]}}, "version": 0.4}', '改图-FLUX.1-dev.json': '{"id": null, "revision": 0, "last_node_id": 12, "last_link_id": 13, "nodes": [{"id": 1, "type": "UnetLoaderGGUF", "pos": [80, 120], "size": [300, 100], "flags": {}, "order": 0, "mode": 0, "inputs": [], "outputs": [{"name": "MODEL", "type": "MODEL", "links": [7]}], "properties": {"Node name for S&R": "UnetLoaderGGUF"}, "widgets_values": ["flux1-dev-Q5_K_S.gguf"], "widgets_values_named": {"unet_name": "flux1-dev-Q5_K_S.gguf"}}, {"id": 2, "type": "DualCLIPLoaderGGUF", "pos": [80, 340], "size": [300, 100], "flags": {}, "order": 1, "mode": 0, "inputs": [], "outputs": [{"name": "CLIP", "type": "CLIP", "links": [1, 2]}], "properties": {"Node name for S&R": "DualCLIPLoaderGGUF"}, "widgets_values": ["clip_l.safetensors", "t5-v1_1-xxl-encoder-Q5_K_M.gguf", "flux"], "widgets_values_named": {"clip_name1": "clip_l.safetensors", "clip_name2": "t5-v1_1-xxl-encoder-Q5_K_M.gguf", "type": "flux"}}, {"id": 3, "type": "VAELoader", "pos": [80, 560], "size": [300, 100], "flags": {}, "order": 2, "mode": 0, "inputs": [], "outputs": [{"name": "VAE", "type": "VAE", "links": [6, 12]}], "properties": {"Node name for S&R": "VAELoader"}, "widgets_values": ["ae.safetensors"], "widgets_values_named": {"vae_name": "ae.safetensors"}}, {"id": 4, "type": "CLIPTextEncode", "pos": [410, 120], "size": [300, 100], "flags": {}, "order": 3, "mode": 0, "inputs": [{"name": "clip", "type": "CLIP", "link": 1}], "outputs": [{"name": "CONDITIONING", "type": "CONDITIONING", "links": [3]}], "properties": {"Node name for S&R": "CLIPTextEncode"}, "widgets_values": ["1girl, solo, looking at viewer, upper body, detailed face, soft lighting"], "widgets_values_named": {"text": "1girl, solo, looking at viewer, upper body, detailed face, soft lighting"}}, {"id": 5, "type": "CLIPTextEncode", "pos": [410, 340], "size": [300, 100], "flags": {}, "order": 4, "mode": 0, "inputs": [{"name": "clip", "type": "CLIP", "link": 2}], "outputs": [{"name": "CONDITIONING", "type": "CONDITIONING", "links": [9]}], "properties": {"Node name for S&R": "CLIPTextEncode"}, "widgets_values": ["lowres, bad anatomy, bad hands, extra digits, fewer digits, cropped, worst quality, low quality, jpeg artifacts, signature, watermark, username, blurry"], "widgets_values_named": {"text": "lowres, bad anatomy, bad hands, extra digits, fewer digits, cropped, worst quality, low quality, jpeg artifacts, signature, watermark, username, blurry"}}, {"id": 6, "type": "FluxGuidance", "pos": [740, 120], "size": [300, 100], "flags": {}, "order": 5, "mode": 0, "inputs": [{"name": "conditioning", "type": "CONDITIONING", "link": 3}], "outputs": [{"name": "CONDITIONING", "type": "CONDITIONING", "links": [8]}], "properties": {"Node name for S&R": "FluxGuidance"}, "widgets_values": [3.5], "widgets_values_named": {"guidance": 3.5}}, {"id": 7, "type": "LoadImage", "pos": [80, 780], "size": [300, 100], "flags": {}, "order": 6, "mode": 0, "inputs": [], "outputs": [{"name": "IMAGE", "type": "IMAGE", "links": [4]}, {"name": "MASK", "type": "MASK", "links": []}], "properties": {"Node name for S&R": "LoadImage"}, "widgets_values": ["__REFERENCE__"], "widgets_values_named": {"image": "__REFERENCE__"}}, {"id": 8, "type": "ImageScale", "pos": [410, 560], "size": [300, 100], "flags": {}, "order": 7, "mode": 0, "inputs": [{"name": "image", "type": "IMAGE", "link": 4}], "outputs": [{"name": "IMAGE", "type": "IMAGE", "links": [5]}], "properties": {"Node name for S&R": "ImageScale"}, "widgets_values": ["lanczos", 1024, 1024, "disabled"], "widgets_values_named": {"upscale_method": "lanczos", "width": 1024, "height": 1024, "crop": "disabled"}}, {"id": 9, "type": "VAEEncode", "pos": [740, 340], "size": [300, 100], "flags": {}, "order": 8, "mode": 0, "inputs": [{"name": "pixels", "type": "IMAGE", "link": 5}, {"name": "vae", "type": "VAE", "link": 6}], "outputs": [{"name": "LATENT", "type": "LATENT", "links": [10]}], "properties": {"Node name for S&R": "VAEEncode"}, "widgets_values": [], "widgets_values_named": {}}, {"id": 10, "type": "KSampler", "pos": [1070, 120], "size": [300, 100], "flags": {}, "order": 9, "mode": 0, "inputs": [{"name": "model", "type": "MODEL", "link": 7}, {"name": "positive", "type": "CONDITIONING", "link": 8}, {"name": "negative", "type": "CONDITIONING", "link": 9}, {"name": "latent_image", "type": "LATENT", "link": 10}], "outputs": [{"name": "LATENT", "type": "LATENT", "links": [11]}], "properties": {"Node name for S&R": "KSampler"}, "widgets_values": [0, "randomize", 20, 1.0, "euler_ancestral", "normal", 0.6], "widgets_values_named": {"seed": 0, "control_after_generate": "randomize", "steps": 20, "cfg": 1.0, "sampler_name": "euler_ancestral", "scheduler": "normal", "denoise": 0.6}}, {"id": 11, "type": "VAEDecode", "pos": [1400, 120], "size": [300, 100], "flags": {}, "order": 10, "mode": 0, "inputs": [{"name": "samples", "type": "LATENT", "link": 11}, {"name": "vae", "type": "VAE", "link": 12}], "outputs": [{"name": "IMAGE", "type": "IMAGE", "links": [13]}], "properties": {"Node name for S&R": "VAEDecode"}, "widgets_values": [], "widgets_values_named": {}}, {"id": 12, "type": "SaveImage", "pos": [1730, 120], "size": [300, 100], "flags": {}, "order": 11, "mode": 0, "inputs": [{"name": "images", "type": "IMAGE", "link": 13}], "outputs": [{"name": "images", "type": "IMAGE", "links": []}], "properties": {"Node name for S&R": "SaveImage"}, "widgets_values": ["app/img2img-flux1-dev-q5ks"], "widgets_values_named": {"filename_prefix": "app/img2img-flux1-dev-q5ks"}}], "links": [[1, 2, 0, 4, 0, "CLIP"], [2, 2, 0, 5, 0, "CLIP"], [3, 4, 0, 6, 0, "CONDITIONING"], [4, 7, 0, 8, 0, "IMAGE"], [5, 8, 0, 9, 0, "IMAGE"], [6, 3, 0, 9, 1, "VAE"], [7, 1, 0, 10, 0, "MODEL"], [8, 6, 0, 10, 1, "CONDITIONING"], [9, 5, 0, 10, 2, "CONDITIONING"], [10, 9, 0, 10, 3, "LATENT"], [11, 10, 0, 11, 0, "LATENT"], [12, 3, 0, 11, 1, "VAE"], [13, 11, 0, 12, 0, "IMAGE"]], "groups": [], "config": {}, "extra": {"linearMode": true, "linearData": {"inputs": [[4, "text"], [7, "image"], [10, "denoise"], [8, "width"], [8, "height"], [10, "seed"], [10, "steps"]], "outputs": [12]}, "ds": {"scale": 0.75, "offset": [0, 0]}}, "version": 0.4}', '文生图-FLUX.1-schnell.json': '{"id": null, "revision": 0, "last_node_id": 9, "last_link_id": 9, "nodes": [{"id": 1, "type": "UnetLoaderGGUF", "pos": [80, 120], "size": [300, 100], "flags": {}, "order": 0, "mode": 0, "inputs": [], "outputs": [{"name": "MODEL", "type": "MODEL", "links": [3]}], "properties": {"Node name for S&R": "UnetLoaderGGUF"}, "widgets_values": ["flux1-schnell-Q5_K_S.gguf"], "widgets_values_named": {"unet_name": "flux1-schnell-Q5_K_S.gguf"}}, {"id": 2, "type": "DualCLIPLoaderGGUF", "pos": [80, 340], "size": [300, 100], "flags": {}, "order": 1, "mode": 0, "inputs": [], "outputs": [{"name": "CLIP", "type": "CLIP", "links": [1, 2]}], "properties": {"Node name for S&R": "DualCLIPLoaderGGUF"}, "widgets_values": ["clip_l.safetensors", "t5-v1_1-xxl-encoder-Q5_K_M.gguf", "flux"], "widgets_values_named": {"clip_name1": "clip_l.safetensors", "clip_name2": "t5-v1_1-xxl-encoder-Q5_K_M.gguf", "type": "flux"}}, {"id": 3, "type": "VAELoader", "pos": [80, 560], "size": [300, 100], "flags": {}, "order": 2, "mode": 0, "inputs": [], "outputs": [{"name": "VAE", "type": "VAE", "links": [8]}], "properties": {"Node name for S&R": "VAELoader"}, "widgets_values": ["ae.safetensors"], "widgets_values_named": {"vae_name": "ae.safetensors"}}, {"id": 4, "type": "CLIPTextEncode", "pos": [410, 120], "size": [300, 100], "flags": {}, "order": 3, "mode": 0, "inputs": [{"name": "clip", "type": "CLIP", "link": 1}], "outputs": [{"name": "CONDITIONING", "type": "CONDITIONING", "links": [4]}], "properties": {"Node name for S&R": "CLIPTextEncode"}, "widgets_values": ["1girl, solo, looking at viewer, upper body, detailed face, soft lighting"], "widgets_values_named": {"text": "1girl, solo, looking at viewer, upper body, detailed face, soft lighting"}}, {"id": 5, "type": "CLIPTextEncode", "pos": [410, 340], "size": [300, 100], "flags": {}, "order": 4, "mode": 0, "inputs": [{"name": "clip", "type": "CLIP", "link": 2}], "outputs": [{"name": "CONDITIONING", "type": "CONDITIONING", "links": [5]}], "properties": {"Node name for S&R": "CLIPTextEncode"}, "widgets_values": ["lowres, bad anatomy, bad hands, extra digits, fewer digits, cropped, worst quality, low quality, jpeg artifacts, signature, watermark, username, blurry"], "widgets_values_named": {"text": "lowres, bad anatomy, bad hands, extra digits, fewer digits, cropped, worst quality, low quality, jpeg artifacts, signature, watermark, username, blurry"}}, {"id": 6, "type": "EmptySD3LatentImage", "pos": [80, 780], "size": [300, 100], "flags": {}, "order": 5, "mode": 0, "inputs": [], "outputs": [{"name": "LATENT", "type": "LATENT", "links": [6]}], "properties": {"Node name for S&R": "EmptySD3LatentImage"}, "widgets_values": [1024, 1024, 1], "widgets_values_named": {"width": 1024, "height": 1024, "batch_size": 1}}, {"id": 7, "type": "KSampler", "pos": [740, 120], "size": [300, 100], "flags": {}, "order": 6, "mode": 0, "inputs": [{"name": "model", "type": "MODEL", "link": 3}, {"name": "positive", "type": "CONDITIONING", "link": 4}, {"name": "negative", "type": "CONDITIONING", "link": 5}, {"name": "latent_image", "type": "LATENT", "link": 6}], "outputs": [{"name": "LATENT", "type": "LATENT", "links": [7]}], "properties": {"Node name for S&R": "KSampler"}, "widgets_values": [0, "randomize", 4, 1.0, "euler_ancestral", "normal", 1.0], "widgets_values_named": {"seed": 0, "control_after_generate": "randomize", "steps": 4, "cfg": 1.0, "sampler_name": "euler_ancestral", "scheduler": "normal", "denoise": 1.0}}, {"id": 8, "type": "VAEDecode", "pos": [1070, 120], "size": [300, 100], "flags": {}, "order": 7, "mode": 0, "inputs": [{"name": "samples", "type": "LATENT", "link": 7}, {"name": "vae", "type": "VAE", "link": 8}], "outputs": [{"name": "IMAGE", "type": "IMAGE", "links": [9]}], "properties": {"Node name for S&R": "VAEDecode"}, "widgets_values": [], "widgets_values_named": {}}, {"id": 9, "type": "SaveImage", "pos": [1400, 120], "size": [300, 100], "flags": {}, "order": 8, "mode": 0, "inputs": [{"name": "images", "type": "IMAGE", "link": 9}], "outputs": [{"name": "images", "type": "IMAGE", "links": []}], "properties": {"Node name for S&R": "SaveImage"}, "widgets_values": ["app/txt2img-flux1-schnell-q5ks"], "widgets_values_named": {"filename_prefix": "app/txt2img-flux1-schnell-q5ks"}}], "links": [[1, 2, 0, 4, 0, "CLIP"], [2, 2, 0, 5, 0, "CLIP"], [3, 1, 0, 7, 0, "MODEL"], [4, 4, 0, 7, 1, "CONDITIONING"], [5, 5, 0, 7, 2, "CONDITIONING"], [6, 6, 0, 7, 3, "LATENT"], [7, 7, 0, 8, 0, "LATENT"], [8, 3, 0, 8, 1, "VAE"], [9, 8, 0, 9, 0, "IMAGE"]], "groups": [], "config": {}, "extra": {"linearMode": true, "linearData": {"inputs": [[4, "text"], [6, "width"], [6, "height"], [7, "seed"], [7, "steps"]], "outputs": [9]}, "ds": {"scale": 0.75, "offset": [0, 0]}}, "version": 0.4}', '改图-FLUX.1-schnell.json': '{"id": null, "revision": 0, "last_node_id": 11, "last_link_id": 12, "nodes": [{"id": 1, "type": "UnetLoaderGGUF", "pos": [80, 120], "size": [300, 100], "flags": {}, "order": 0, "mode": 0, "inputs": [], "outputs": [{"name": "MODEL", "type": "MODEL", "links": [6]}], "properties": {"Node name for S&R": "UnetLoaderGGUF"}, "widgets_values": ["flux1-schnell-Q5_K_S.gguf"], "widgets_values_named": {"unet_name": "flux1-schnell-Q5_K_S.gguf"}}, {"id": 2, "type": "DualCLIPLoaderGGUF", "pos": [80, 340], "size": [300, 100], "flags": {}, "order": 1, "mode": 0, "inputs": [], "outputs": [{"name": "CLIP", "type": "CLIP", "links": [1, 2]}], "properties": {"Node name for S&R": "DualCLIPLoaderGGUF"}, "widgets_values": ["clip_l.safetensors", "t5-v1_1-xxl-encoder-Q5_K_M.gguf", "flux"], "widgets_values_named": {"clip_name1": "clip_l.safetensors", "clip_name2": "t5-v1_1-xxl-encoder-Q5_K_M.gguf", "type": "flux"}}, {"id": 3, "type": "VAELoader", "pos": [80, 560], "size": [300, 100], "flags": {}, "order": 2, "mode": 0, "inputs": [], "outputs": [{"name": "VAE", "type": "VAE", "links": [5, 11]}], "properties": {"Node name for S&R": "VAELoader"}, "widgets_values": ["ae.safetensors"], "widgets_values_named": {"vae_name": "ae.safetensors"}}, {"id": 4, "type": "CLIPTextEncode", "pos": [410, 120], "size": [300, 100], "flags": {}, "order": 3, "mode": 0, "inputs": [{"name": "clip", "type": "CLIP", "link": 1}], "outputs": [{"name": "CONDITIONING", "type": "CONDITIONING", "links": [7]}], "properties": {"Node name for S&R": "CLIPTextEncode"}, "widgets_values": ["1girl, solo, looking at viewer, upper body, detailed face, soft lighting"], "widgets_values_named": {"text": "1girl, solo, looking at viewer, upper body, detailed face, soft lighting"}}, {"id": 5, "type": "CLIPTextEncode", "pos": [410, 340], "size": [300, 100], "flags": {}, "order": 4, "mode": 0, "inputs": [{"name": "clip", "type": "CLIP", "link": 2}], "outputs": [{"name": "CONDITIONING", "type": "CONDITIONING", "links": [8]}], "properties": {"Node name for S&R": "CLIPTextEncode"}, "widgets_values": ["lowres, bad anatomy, bad hands, extra digits, fewer digits, cropped, worst quality, low quality, jpeg artifacts, signature, watermark, username, blurry"], "widgets_values_named": {"text": "lowres, bad anatomy, bad hands, extra digits, fewer digits, cropped, worst quality, low quality, jpeg artifacts, signature, watermark, username, blurry"}}, {"id": 6, "type": "LoadImage", "pos": [80, 780], "size": [300, 100], "flags": {}, "order": 5, "mode": 0, "inputs": [], "outputs": [{"name": "IMAGE", "type": "IMAGE", "links": [3]}, {"name": "MASK", "type": "MASK", "links": []}], "properties": {"Node name for S&R": "LoadImage"}, "widgets_values": ["__REFERENCE__"], "widgets_values_named": {"image": "__REFERENCE__"}}, {"id": 7, "type": "ImageScale", "pos": [410, 560], "size": [300, 100], "flags": {}, "order": 6, "mode": 0, "inputs": [{"name": "image", "type": "IMAGE", "link": 3}], "outputs": [{"name": "IMAGE", "type": "IMAGE", "links": [4]}], "properties": {"Node name for S&R": "ImageScale"}, "widgets_values": ["lanczos", 1024, 1024, "disabled"], "widgets_values_named": {"upscale_method": "lanczos", "width": 1024, "height": 1024, "crop": "disabled"}}, {"id": 8, "type": "VAEEncode", "pos": [740, 120], "size": [300, 100], "flags": {}, "order": 7, "mode": 0, "inputs": [{"name": "pixels", "type": "IMAGE", "link": 4}, {"name": "vae", "type": "VAE", "link": 5}], "outputs": [{"name": "LATENT", "type": "LATENT", "links": [9]}], "properties": {"Node name for S&R": "VAEEncode"}, "widgets_values": [], "widgets_values_named": {}}, {"id": 9, "type": "KSampler", "pos": [1070, 120], "size": [300, 100], "flags": {}, "order": 8, "mode": 0, "inputs": [{"name": "model", "type": "MODEL", "link": 6}, {"name": "positive", "type": "CONDITIONING", "link": 7}, {"name": "negative", "type": "CONDITIONING", "link": 8}, {"name": "latent_image", "type": "LATENT", "link": 9}], "outputs": [{"name": "LATENT", "type": "LATENT", "links": [10]}], "properties": {"Node name for S&R": "KSampler"}, "widgets_values": [0, "randomize", 4, 1.0, "euler_ancestral", "normal", 0.6], "widgets_values_named": {"seed": 0, "control_after_generate": "randomize", "steps": 4, "cfg": 1.0, "sampler_name": "euler_ancestral", "scheduler": "normal", "denoise": 0.6}}, {"id": 10, "type": "VAEDecode", "pos": [1400, 120], "size": [300, 100], "flags": {}, "order": 9, "mode": 0, "inputs": [{"name": "samples", "type": "LATENT", "link": 10}, {"name": "vae", "type": "VAE", "link": 11}], "outputs": [{"name": "IMAGE", "type": "IMAGE", "links": [12]}], "properties": {"Node name for S&R": "VAEDecode"}, "widgets_values": [], "widgets_values_named": {}}, {"id": 11, "type": "SaveImage", "pos": [1730, 120], "size": [300, 100], "flags": {}, "order": 10, "mode": 0, "inputs": [{"name": "images", "type": "IMAGE", "link": 12}], "outputs": [{"name": "images", "type": "IMAGE", "links": []}], "properties": {"Node name for S&R": "SaveImage"}, "widgets_values": ["app/img2img-flux1-schnell-q5ks"], "widgets_values_named": {"filename_prefix": "app/img2img-flux1-schnell-q5ks"}}], "links": [[1, 2, 0, 4, 0, "CLIP"], [2, 2, 0, 5, 0, "CLIP"], [3, 6, 0, 7, 0, "IMAGE"], [4, 7, 0, 8, 0, "IMAGE"], [5, 3, 0, 8, 1, "VAE"], [6, 1, 0, 9, 0, "MODEL"], [7, 4, 0, 9, 1, "CONDITIONING"], [8, 5, 0, 9, 2, "CONDITIONING"], [9, 8, 0, 9, 3, "LATENT"], [10, 9, 0, 10, 0, "LATENT"], [11, 3, 0, 10, 1, "VAE"], [12, 10, 0, 11, 0, "IMAGE"]], "groups": [], "config": {}, "extra": {"linearMode": true, "linearData": {"inputs": [[4, "text"], [6, "image"], [9, "denoise"], [7, "width"], [7, "height"], [9, "seed"], [9, "steps"]], "outputs": [11]}, "ds": {"scale": 0.75, "offset": [0, 0]}}, "version": 0.4}', '文生图-FLUX.2-klein.json': '{"id": null, "revision": 0, "last_node_id": 10, "last_link_id": 10, "nodes": [{"id": 1, "type": "UnetLoaderGGUF", "pos": [80, 120], "size": [300, 100], "flags": {}, "order": 0, "mode": 0, "inputs": [], "outputs": [{"name": "MODEL", "type": "MODEL", "links": [4]}], "properties": {"Node name for S&R": "UnetLoaderGGUF"}, "widgets_values": ["flux-2-klein-4b-Q8_0.gguf"], "widgets_values_named": {"unet_name": "flux-2-klein-4b-Q8_0.gguf"}}, {"id": 2, "type": "CLIPLoaderGGUF", "pos": [80, 340], "size": [300, 100], "flags": {}, "order": 1, "mode": 0, "inputs": [], "outputs": [{"name": "CLIP", "type": "CLIP", "links": [1, 2]}], "properties": {"Node name for S&R": "CLIPLoaderGGUF"}, "widgets_values": ["Qwen3-4B-Q8_0.gguf", "flux2"], "widgets_values_named": {"clip_name": "Qwen3-4B-Q8_0.gguf", "type": "flux2"}}, {"id": 3, "type": "VAELoader", "pos": [80, 560], "size": [300, 100], "flags": {}, "order": 2, "mode": 0, "inputs": [], "outputs": [{"name": "VAE", "type": "VAE", "links": [9]}], "properties": {"Node name for S&R": "VAELoader"}, "widgets_values": ["split_files/vae/flux2-vae.safetensors"], "widgets_values_named": {"vae_name": "split_files/vae/flux2-vae.safetensors"}}, {"id": 4, "type": "CLIPTextEncode", "pos": [410, 120], "size": [300, 100], "flags": {}, "order": 3, "mode": 0, "inputs": [{"name": "clip", "type": "CLIP", "link": 1}], "outputs": [{"name": "CONDITIONING", "type": "CONDITIONING", "links": [3]}], "properties": {"Node name for S&R": "CLIPTextEncode"}, "widgets_values": ["1girl, solo, looking at viewer, upper body, detailed face, soft lighting"], "widgets_values_named": {"text": "1girl, solo, looking at viewer, upper body, detailed face, soft lighting"}}, {"id": 5, "type": "CLIPTextEncode", "pos": [410, 340], "size": [300, 100], "flags": {}, "order": 4, "mode": 0, "inputs": [{"name": "clip", "type": "CLIP", "link": 2}], "outputs": [{"name": "CONDITIONING", "type": "CONDITIONING", "links": [6]}], "properties": {"Node name for S&R": "CLIPTextEncode"}, "widgets_values": ["lowres, bad anatomy, bad hands, extra digits, fewer digits, cropped, worst quality, low quality, jpeg artifacts, signature, watermark, username, blurry"], "widgets_values_named": {"text": "lowres, bad anatomy, bad hands, extra digits, fewer digits, cropped, worst quality, low quality, jpeg artifacts, signature, watermark, username, blurry"}}, {"id": 6, "type": "FluxGuidance", "pos": [740, 120], "size": [300, 100], "flags": {}, "order": 5, "mode": 0, "inputs": [{"name": "conditioning", "type": "CONDITIONING", "link": 3}], "outputs": [{"name": "CONDITIONING", "type": "CONDITIONING", "links": [5]}], "properties": {"Node name for S&R": "FluxGuidance"}, "widgets_values": [1.0], "widgets_values_named": {"guidance": 1.0}}, {"id": 7, "type": "EmptyFlux2LatentImage", "pos": [80, 780], "size": [300, 100], "flags": {}, "order": 6, "mode": 0, "inputs": [], "outputs": [{"name": "LATENT", "type": "LATENT", "links": [7]}], "properties": {"Node name for S&R": "EmptyFlux2LatentImage"}, "widgets_values": [1024, 1024, 1], "widgets_values_named": {"width": 1024, "height": 1024, "batch_size": 1}}, {"id": 8, "type": "KSampler", "pos": [1070, 120], "size": [300, 100], "flags": {}, "order": 7, "mode": 0, "inputs": [{"name": "model", "type": "MODEL", "link": 4}, {"name": "positive", "type": "CONDITIONING", "link": 5}, {"name": "negative", "type": "CONDITIONING", "link": 6}, {"name": "latent_image", "type": "LATENT", "link": 7}], "outputs": [{"name": "LATENT", "type": "LATENT", "links": [8]}], "properties": {"Node name for S&R": "KSampler"}, "widgets_values": [0, "randomize", 4, 1.0, "euler_ancestral", "normal", 1.0], "widgets_values_named": {"seed": 0, "control_after_generate": "randomize", "steps": 4, "cfg": 1.0, "sampler_name": "euler_ancestral", "scheduler": "normal", "denoise": 1.0}}, {"id": 9, "type": "VAEDecode", "pos": [1400, 120], "size": [300, 100], "flags": {}, "order": 8, "mode": 0, "inputs": [{"name": "samples", "type": "LATENT", "link": 8}, {"name": "vae", "type": "VAE", "link": 9}], "outputs": [{"name": "IMAGE", "type": "IMAGE", "links": [10]}], "properties": {"Node name for S&R": "VAEDecode"}, "widgets_values": [], "widgets_values_named": {}}, {"id": 10, "type": "SaveImage", "pos": [1730, 120], "size": [300, 100], "flags": {}, "order": 9, "mode": 0, "inputs": [{"name": "images", "type": "IMAGE", "link": 10}], "outputs": [{"name": "images", "type": "IMAGE", "links": []}], "properties": {"Node name for S&R": "SaveImage"}, "widgets_values": ["app/txt2img-flux2-klein-4b-q8"], "widgets_values_named": {"filename_prefix": "app/txt2img-flux2-klein-4b-q8"}}], "links": [[1, 2, 0, 4, 0, "CLIP"], [2, 2, 0, 5, 0, "CLIP"], [3, 4, 0, 6, 0, "CONDITIONING"], [4, 1, 0, 8, 0, "MODEL"], [5, 6, 0, 8, 1, "CONDITIONING"], [6, 5, 0, 8, 2, "CONDITIONING"], [7, 7, 0, 8, 3, "LATENT"], [8, 8, 0, 9, 0, "LATENT"], [9, 3, 0, 9, 1, "VAE"], [10, 9, 0, 10, 0, "IMAGE"]], "groups": [], "config": {}, "extra": {"linearMode": true, "linearData": {"inputs": [[4, "text"], [7, "width"], [7, "height"], [8, "seed"], [8, "steps"]], "outputs": [10]}, "ds": {"scale": 0.75, "offset": [0, 0]}}, "version": 0.4}', '改图-FLUX.2-klein.json': '{"id": null, "revision": 0, "last_node_id": 12, "last_link_id": 13, "nodes": [{"id": 1, "type": "UnetLoaderGGUF", "pos": [80, 120], "size": [300, 100], "flags": {}, "order": 0, "mode": 0, "inputs": [], "outputs": [{"name": "MODEL", "type": "MODEL", "links": [7]}], "properties": {"Node name for S&R": "UnetLoaderGGUF"}, "widgets_values": ["flux-2-klein-4b-Q8_0.gguf"], "widgets_values_named": {"unet_name": "flux-2-klein-4b-Q8_0.gguf"}}, {"id": 2, "type": "CLIPLoaderGGUF", "pos": [80, 340], "size": [300, 100], "flags": {}, "order": 1, "mode": 0, "inputs": [], "outputs": [{"name": "CLIP", "type": "CLIP", "links": [1, 2]}], "properties": {"Node name for S&R": "CLIPLoaderGGUF"}, "widgets_values": ["Qwen3-4B-Q8_0.gguf", "flux2"], "widgets_values_named": {"clip_name": "Qwen3-4B-Q8_0.gguf", "type": "flux2"}}, {"id": 3, "type": "VAELoader", "pos": [80, 560], "size": [300, 100], "flags": {}, "order": 2, "mode": 0, "inputs": [], "outputs": [{"name": "VAE", "type": "VAE", "links": [6, 12]}], "properties": {"Node name for S&R": "VAELoader"}, "widgets_values": ["split_files/vae/flux2-vae.safetensors"], "widgets_values_named": {"vae_name": "split_files/vae/flux2-vae.safetensors"}}, {"id": 4, "type": "CLIPTextEncode", "pos": [410, 120], "size": [300, 100], "flags": {}, "order": 3, "mode": 0, "inputs": [{"name": "clip", "type": "CLIP", "link": 1}], "outputs": [{"name": "CONDITIONING", "type": "CONDITIONING", "links": [3]}], "properties": {"Node name for S&R": "CLIPTextEncode"}, "widgets_values": ["1girl, solo, looking at viewer, upper body, detailed face, soft lighting"], "widgets_values_named": {"text": "1girl, solo, looking at viewer, upper body, detailed face, soft lighting"}}, {"id": 5, "type": "CLIPTextEncode", "pos": [410, 340], "size": [300, 100], "flags": {}, "order": 4, "mode": 0, "inputs": [{"name": "clip", "type": "CLIP", "link": 2}], "outputs": [{"name": "CONDITIONING", "type": "CONDITIONING", "links": [9]}], "properties": {"Node name for S&R": "CLIPTextEncode"}, "widgets_values": ["lowres, bad anatomy, bad hands, extra digits, fewer digits, cropped, worst quality, low quality, jpeg artifacts, signature, watermark, username, blurry"], "widgets_values_named": {"text": "lowres, bad anatomy, bad hands, extra digits, fewer digits, cropped, worst quality, low quality, jpeg artifacts, signature, watermark, username, blurry"}}, {"id": 6, "type": "FluxGuidance", "pos": [740, 120], "size": [300, 100], "flags": {}, "order": 5, "mode": 0, "inputs": [{"name": "conditioning", "type": "CONDITIONING", "link": 3}], "outputs": [{"name": "CONDITIONING", "type": "CONDITIONING", "links": [8]}], "properties": {"Node name for S&R": "FluxGuidance"}, "widgets_values": [1.0], "widgets_values_named": {"guidance": 1.0}}, {"id": 7, "type": "LoadImage", "pos": [80, 780], "size": [300, 100], "flags": {}, "order": 6, "mode": 0, "inputs": [], "outputs": [{"name": "IMAGE", "type": "IMAGE", "links": [4]}, {"name": "MASK", "type": "MASK", "links": []}], "properties": {"Node name for S&R": "LoadImage"}, "widgets_values": ["__REFERENCE__"], "widgets_values_named": {"image": "__REFERENCE__"}}, {"id": 8, "type": "ImageScale", "pos": [410, 560], "size": [300, 100], "flags": {}, "order": 7, "mode": 0, "inputs": [{"name": "image", "type": "IMAGE", "link": 4}], "outputs": [{"name": "IMAGE", "type": "IMAGE", "links": [5]}], "properties": {"Node name for S&R": "ImageScale"}, "widgets_values": ["lanczos", 1024, 1024, "disabled"], "widgets_values_named": {"upscale_method": "lanczos", "width": 1024, "height": 1024, "crop": "disabled"}}, {"id": 9, "type": "VAEEncode", "pos": [740, 340], "size": [300, 100], "flags": {}, "order": 8, "mode": 0, "inputs": [{"name": "pixels", "type": "IMAGE", "link": 5}, {"name": "vae", "type": "VAE", "link": 6}], "outputs": [{"name": "LATENT", "type": "LATENT", "links": [10]}], "properties": {"Node name for S&R": "VAEEncode"}, "widgets_values": [], "widgets_values_named": {}}, {"id": 10, "type": "KSampler", "pos": [1070, 120], "size": [300, 100], "flags": {}, "order": 9, "mode": 0, "inputs": [{"name": "model", "type": "MODEL", "link": 7}, {"name": "positive", "type": "CONDITIONING", "link": 8}, {"name": "negative", "type": "CONDITIONING", "link": 9}, {"name": "latent_image", "type": "LATENT", "link": 10}], "outputs": [{"name": "LATENT", "type": "LATENT", "links": [11]}], "properties": {"Node name for S&R": "KSampler"}, "widgets_values": [0, "randomize", 4, 1.0, "euler_ancestral", "normal", 0.6], "widgets_values_named": {"seed": 0, "control_after_generate": "randomize", "steps": 4, "cfg": 1.0, "sampler_name": "euler_ancestral", "scheduler": "normal", "denoise": 0.6}}, {"id": 11, "type": "VAEDecode", "pos": [1400, 120], "size": [300, 100], "flags": {}, "order": 10, "mode": 0, "inputs": [{"name": "samples", "type": "LATENT", "link": 11}, {"name": "vae", "type": "VAE", "link": 12}], "outputs": [{"name": "IMAGE", "type": "IMAGE", "links": [13]}], "properties": {"Node name for S&R": "VAEDecode"}, "widgets_values": [], "widgets_values_named": {}}, {"id": 12, "type": "SaveImage", "pos": [1730, 120], "size": [300, 100], "flags": {}, "order": 11, "mode": 0, "inputs": [{"name": "images", "type": "IMAGE", "link": 13}], "outputs": [{"name": "images", "type": "IMAGE", "links": []}], "properties": {"Node name for S&R": "SaveImage"}, "widgets_values": ["app/img2img-flux2-klein-4b-q8"], "widgets_values_named": {"filename_prefix": "app/img2img-flux2-klein-4b-q8"}}], "links": [[1, 2, 0, 4, 0, "CLIP"], [2, 2, 0, 5, 0, "CLIP"], [3, 4, 0, 6, 0, "CONDITIONING"], [4, 7, 0, 8, 0, "IMAGE"], [5, 8, 0, 9, 0, "IMAGE"], [6, 3, 0, 9, 1, "VAE"], [7, 1, 0, 10, 0, "MODEL"], [8, 6, 0, 10, 1, "CONDITIONING"], [9, 5, 0, 10, 2, "CONDITIONING"], [10, 9, 0, 10, 3, "LATENT"], [11, 10, 0, 11, 0, "LATENT"], [12, 3, 0, 11, 1, "VAE"], [13, 11, 0, 12, 0, "IMAGE"]], "groups": [], "config": {}, "extra": {"linearMode": true, "linearData": {"inputs": [[4, "text"], [7, "image"], [10, "denoise"], [8, "width"], [8, "height"], [10, "seed"], [10, "steps"]], "outputs": [12]}, "ds": {"scale": 0.75, "offset": [0, 0]}}, "version": 0.4}', '文生图-Z-Image-Turbo.json': '{"id": null, "revision": 0, "last_node_id": 10, "last_link_id": 10, "nodes": [{"id": 1, "type": "UnetLoaderGGUF", "pos": [80, 120], "size": [300, 100], "flags": {}, "order": 0, "mode": 0, "inputs": [], "outputs": [{"name": "MODEL", "type": "MODEL", "links": [1]}], "properties": {"Node name for S&R": "UnetLoaderGGUF"}, "widgets_values": ["z_image_turbo-Q8_0.gguf"], "widgets_values_named": {"unet_name": "z_image_turbo-Q8_0.gguf"}}, {"id": 2, "type": "CLIPLoaderGGUF", "pos": [80, 340], "size": [300, 100], "flags": {}, "order": 1, "mode": 0, "inputs": [], "outputs": [{"name": "CLIP", "type": "CLIP", "links": [2, 3]}], "properties": {"Node name for S&R": "CLIPLoaderGGUF"}, "widgets_values": ["Qwen3-4B-Q8_0.gguf", "lumina2"], "widgets_values_named": {"clip_name": "Qwen3-4B-Q8_0.gguf", "type": "lumina2"}}, {"id": 3, "type": "VAELoader", "pos": [80, 560], "size": [300, 100], "flags": {}, "order": 2, "mode": 0, "inputs": [], "outputs": [{"name": "VAE", "type": "VAE", "links": [9]}], "properties": {"Node name for S&R": "VAELoader"}, "widgets_values": ["split_files/vae/ae.safetensors"], "widgets_values_named": {"vae_name": "split_files/vae/ae.safetensors"}}, {"id": 4, "type": "ModelSamplingAuraFlow", "pos": [410, 120], "size": [300, 100], "flags": {}, "order": 3, "mode": 0, "inputs": [{"name": "model", "type": "MODEL", "link": 1}], "outputs": [{"name": "MODEL", "type": "MODEL", "links": [4]}], "properties": {"Node name for S&R": "ModelSamplingAuraFlow"}, "widgets_values": [3.0, "flow"], "widgets_values_named": {"shift": 3.0, "sampling": "flow"}}, {"id": 5, "type": "CLIPTextEncode", "pos": [410, 340], "size": [300, 100], "flags": {}, "order": 4, "mode": 0, "inputs": [{"name": "clip", "type": "CLIP", "link": 2}], "outputs": [{"name": "CONDITIONING", "type": "CONDITIONING", "links": [5]}], "properties": {"Node name for S&R": "CLIPTextEncode"}, "widgets_values": ["1girl, solo, looking at viewer, upper body, detailed face, soft lighting"], "widgets_values_named": {"text": "1girl, solo, looking at viewer, upper body, detailed face, soft lighting"}}, {"id": 6, "type": "CLIPTextEncode", "pos": [410, 560], "size": [300, 100], "flags": {}, "order": 5, "mode": 0, "inputs": [{"name": "clip", "type": "CLIP", "link": 3}], "outputs": [{"name": "CONDITIONING", "type": "CONDITIONING", "links": [6]}], "properties": {"Node name for S&R": "CLIPTextEncode"}, "widgets_values": ["lowres, bad anatomy, bad hands, extra digits, fewer digits, cropped, worst quality, low quality, jpeg artifacts, signature, watermark, username, blurry"], "widgets_values_named": {"text": "lowres, bad anatomy, bad hands, extra digits, fewer digits, cropped, worst quality, low quality, jpeg artifacts, signature, watermark, username, blurry"}}, {"id": 7, "type": "EmptySD3LatentImage", "pos": [80, 780], "size": [300, 100], "flags": {}, "order": 6, "mode": 0, "inputs": [], "outputs": [{"name": "LATENT", "type": "LATENT", "links": [7]}], "properties": {"Node name for S&R": "EmptySD3LatentImage"}, "widgets_values": [1024, 1024, 1], "widgets_values_named": {"width": 1024, "height": 1024, "batch_size": 1}}, {"id": 8, "type": "KSampler", "pos": [740, 120], "size": [300, 100], "flags": {}, "order": 7, "mode": 0, "inputs": [{"name": "model", "type": "MODEL", "link": 4}, {"name": "positive", "type": "CONDITIONING", "link": 5}, {"name": "negative", "type": "CONDITIONING", "link": 6}, {"name": "latent_image", "type": "LATENT", "link": 7}], "outputs": [{"name": "LATENT", "type": "LATENT", "links": [8]}], "properties": {"Node name for S&R": "KSampler"}, "widgets_values": [0, "randomize", 9, 1.0, "euler_ancestral", "normal", 1.0], "widgets_values_named": {"seed": 0, "control_after_generate": "randomize", "steps": 9, "cfg": 1.0, "sampler_name": "euler_ancestral", "scheduler": "normal", "denoise": 1.0}}, {"id": 9, "type": "VAEDecode", "pos": [1070, 120], "size": [300, 100], "flags": {}, "order": 8, "mode": 0, "inputs": [{"name": "samples", "type": "LATENT", "link": 8}, {"name": "vae", "type": "VAE", "link": 9}], "outputs": [{"name": "IMAGE", "type": "IMAGE", "links": [10]}], "properties": {"Node name for S&R": "VAEDecode"}, "widgets_values": [], "widgets_values_named": {}}, {"id": 10, "type": "SaveImage", "pos": [1400, 120], "size": [300, 100], "flags": {}, "order": 9, "mode": 0, "inputs": [{"name": "images", "type": "IMAGE", "link": 10}], "outputs": [{"name": "images", "type": "IMAGE", "links": []}], "properties": {"Node name for S&R": "SaveImage"}, "widgets_values": ["app/txt2img-z-image-turbo-q8"], "widgets_values_named": {"filename_prefix": "app/txt2img-z-image-turbo-q8"}}], "links": [[1, 1, 0, 4, 0, "MODEL"], [2, 2, 0, 5, 0, "CLIP"], [3, 2, 0, 6, 0, "CLIP"], [4, 4, 0, 8, 0, "MODEL"], [5, 5, 0, 8, 1, "CONDITIONING"], [6, 6, 0, 8, 2, "CONDITIONING"], [7, 7, 0, 8, 3, "LATENT"], [8, 8, 0, 9, 0, "LATENT"], [9, 3, 0, 9, 1, "VAE"], [10, 9, 0, 10, 0, "IMAGE"]], "groups": [], "config": {}, "extra": {"linearMode": true, "linearData": {"inputs": [[5, "text"], [7, "width"], [7, "height"], [8, "seed"], [8, "steps"]], "outputs": [10]}, "ds": {"scale": 0.75, "offset": [0, 0]}}, "version": 0.4}', '改图-Z-Image-Turbo.json': '{"id": null, "revision": 0, "last_node_id": 12, "last_link_id": 13, "nodes": [{"id": 1, "type": "UnetLoaderGGUF", "pos": [80, 120], "size": [300, 100], "flags": {}, "order": 0, "mode": 0, "inputs": [], "outputs": [{"name": "MODEL", "type": "MODEL", "links": [1]}], "properties": {"Node name for S&R": "UnetLoaderGGUF"}, "widgets_values": ["z_image_turbo-Q8_0.gguf"], "widgets_values_named": {"unet_name": "z_image_turbo-Q8_0.gguf"}}, {"id": 2, "type": "CLIPLoaderGGUF", "pos": [80, 340], "size": [300, 100], "flags": {}, "order": 1, "mode": 0, "inputs": [], "outputs": [{"name": "CLIP", "type": "CLIP", "links": [2, 3]}], "properties": {"Node name for S&R": "CLIPLoaderGGUF"}, "widgets_values": ["Qwen3-4B-Q8_0.gguf", "lumina2"], "widgets_values_named": {"clip_name": "Qwen3-4B-Q8_0.gguf", "type": "lumina2"}}, {"id": 3, "type": "VAELoader", "pos": [80, 560], "size": [300, 100], "flags": {}, "order": 2, "mode": 0, "inputs": [], "outputs": [{"name": "VAE", "type": "VAE", "links": [6, 12]}], "properties": {"Node name for S&R": "VAELoader"}, "widgets_values": ["split_files/vae/ae.safetensors"], "widgets_values_named": {"vae_name": "split_files/vae/ae.safetensors"}}, {"id": 4, "type": "ModelSamplingAuraFlow", "pos": [410, 120], "size": [300, 100], "flags": {}, "order": 3, "mode": 0, "inputs": [{"name": "model", "type": "MODEL", "link": 1}], "outputs": [{"name": "MODEL", "type": "MODEL", "links": [7]}], "properties": {"Node name for S&R": "ModelSamplingAuraFlow"}, "widgets_values": [3.0, "flow"], "widgets_values_named": {"shift": 3.0, "sampling": "flow"}}, {"id": 5, "type": "CLIPTextEncode", "pos": [410, 340], "size": [300, 100], "flags": {}, "order": 4, "mode": 0, "inputs": [{"name": "clip", "type": "CLIP", "link": 2}], "outputs": [{"name": "CONDITIONING", "type": "CONDITIONING", "links": [8]}], "properties": {"Node name for S&R": "CLIPTextEncode"}, "widgets_values": ["1girl, solo, looking at viewer, upper body, detailed face, soft lighting"], "widgets_values_named": {"text": "1girl, solo, looking at viewer, upper body, detailed face, soft lighting"}}, {"id": 6, "type": "CLIPTextEncode", "pos": [410, 560], "size": [300, 100], "flags": {}, "order": 5, "mode": 0, "inputs": [{"name": "clip", "type": "CLIP", "link": 3}], "outputs": [{"name": "CONDITIONING", "type": "CONDITIONING", "links": [9]}], "properties": {"Node name for S&R": "CLIPTextEncode"}, "widgets_values": ["lowres, bad anatomy, bad hands, extra digits, fewer digits, cropped, worst quality, low quality, jpeg artifacts, signature, watermark, username, blurry"], "widgets_values_named": {"text": "lowres, bad anatomy, bad hands, extra digits, fewer digits, cropped, worst quality, low quality, jpeg artifacts, signature, watermark, username, blurry"}}, {"id": 7, "type": "LoadImage", "pos": [80, 780], "size": [300, 100], "flags": {}, "order": 6, "mode": 0, "inputs": [], "outputs": [{"name": "IMAGE", "type": "IMAGE", "links": [4]}, {"name": "MASK", "type": "MASK", "links": []}], "properties": {"Node name for S&R": "LoadImage"}, "widgets_values": ["__REFERENCE__"], "widgets_values_named": {"image": "__REFERENCE__"}}, {"id": 8, "type": "ImageScale", "pos": [410, 780], "size": [300, 100], "flags": {}, "order": 7, "mode": 0, "inputs": [{"name": "image", "type": "IMAGE", "link": 4}], "outputs": [{"name": "IMAGE", "type": "IMAGE", "links": [5]}], "properties": {"Node name for S&R": "ImageScale"}, "widgets_values": ["lanczos", 1024, 1024, "disabled"], "widgets_values_named": {"upscale_method": "lanczos", "width": 1024, "height": 1024, "crop": "disabled"}}, {"id": 9, "type": "VAEEncode", "pos": [740, 120], "size": [300, 100], "flags": {}, "order": 8, "mode": 0, "inputs": [{"name": "pixels", "type": "IMAGE", "link": 5}, {"name": "vae", "type": "VAE", "link": 6}], "outputs": [{"name": "LATENT", "type": "LATENT", "links": [10]}], "properties": {"Node name for S&R": "VAEEncode"}, "widgets_values": [], "widgets_values_named": {}}, {"id": 10, "type": "KSampler", "pos": [1070, 120], "size": [300, 100], "flags": {}, "order": 9, "mode": 0, "inputs": [{"name": "model", "type": "MODEL", "link": 7}, {"name": "positive", "type": "CONDITIONING", "link": 8}, {"name": "negative", "type": "CONDITIONING", "link": 9}, {"name": "latent_image", "type": "LATENT", "link": 10}], "outputs": [{"name": "LATENT", "type": "LATENT", "links": [11]}], "properties": {"Node name for S&R": "KSampler"}, "widgets_values": [0, "randomize", 9, 1.0, "euler_ancestral", "normal", 0.6], "widgets_values_named": {"seed": 0, "control_after_generate": "randomize", "steps": 9, "cfg": 1.0, "sampler_name": "euler_ancestral", "scheduler": "normal", "denoise": 0.6}}, {"id": 11, "type": "VAEDecode", "pos": [1400, 120], "size": [300, 100], "flags": {}, "order": 10, "mode": 0, "inputs": [{"name": "samples", "type": "LATENT", "link": 11}, {"name": "vae", "type": "VAE", "link": 12}], "outputs": [{"name": "IMAGE", "type": "IMAGE", "links": [13]}], "properties": {"Node name for S&R": "VAEDecode"}, "widgets_values": [], "widgets_values_named": {}}, {"id": 12, "type": "SaveImage", "pos": [1730, 120], "size": [300, 100], "flags": {}, "order": 11, "mode": 0, "inputs": [{"name": "images", "type": "IMAGE", "link": 13}], "outputs": [{"name": "images", "type": "IMAGE", "links": []}], "properties": {"Node name for S&R": "SaveImage"}, "widgets_values": ["app/img2img-z-image-turbo-q8"], "widgets_values_named": {"filename_prefix": "app/img2img-z-image-turbo-q8"}}], "links": [[1, 1, 0, 4, 0, "MODEL"], [2, 2, 0, 5, 0, "CLIP"], [3, 2, 0, 6, 0, "CLIP"], [4, 7, 0, 8, 0, "IMAGE"], [5, 8, 0, 9, 0, "IMAGE"], [6, 3, 0, 9, 1, "VAE"], [7, 4, 0, 10, 0, "MODEL"], [8, 5, 0, 10, 1, "CONDITIONING"], [9, 6, 0, 10, 2, "CONDITIONING"], [10, 9, 0, 10, 3, "LATENT"], [11, 10, 0, 11, 0, "LATENT"], [12, 3, 0, 11, 1, "VAE"], [13, 11, 0, 12, 0, "IMAGE"]], "groups": [], "config": {}, "extra": {"linearMode": true, "linearData": {"inputs": [[5, "text"], [7, "image"], [10, "denoise"], [8, "width"], [8, "height"], [10, "seed"], [10, "steps"]], "outputs": [12]}, "ds": {"scale": 0.75, "offset": [0, 0]}}, "version": 0.4}', '文生图-Anima-Turbo.json': '{"id": null, "revision": 0, "last_node_id": 10, "last_link_id": 10, "nodes": [{"id": 1, "type": "UnetLoaderGGUF", "pos": [80, 120], "size": [300, 100], "flags": {}, "order": 0, "mode": 0, "inputs": [], "outputs": [{"name": "MODEL", "type": "MODEL", "links": [1]}], "properties": {"Node name for S&R": "UnetLoaderGGUF"}, "widgets_values": ["anima-turbo-v1.1-Q5_K_M.gguf"], "widgets_values_named": {"unet_name": "anima-turbo-v1.1-Q5_K_M.gguf"}}, {"id": 2, "type": "CLIPLoader", "pos": [80, 340], "size": [300, 100], "flags": {}, "order": 1, "mode": 0, "inputs": [], "outputs": [{"name": "CLIP", "type": "CLIP", "links": [2, 3]}], "properties": {"Node name for S&R": "CLIPLoader"}, "widgets_values": ["split_files/text_encoders/qwen_3_06b_base.safetensors", "cosmos", "default"], "widgets_values_named": {"clip_name": "split_files/text_encoders/qwen_3_06b_base.safetensors", "type": "cosmos", "device": "default"}}, {"id": 3, "type": "VAELoader", "pos": [80, 560], "size": [300, 100], "flags": {}, "order": 2, "mode": 0, "inputs": [], "outputs": [{"name": "VAE", "type": "VAE", "links": [9]}], "properties": {"Node name for S&R": "VAELoader"}, "widgets_values": ["split_files/vae/qwen_image_vae.safetensors"], "widgets_values_named": {"vae_name": "split_files/vae/qwen_image_vae.safetensors"}}, {"id": 4, "type": "ModelSamplingAuraFlow", "pos": [410, 120], "size": [300, 100], "flags": {}, "order": 3, "mode": 0, "inputs": [{"name": "model", "type": "MODEL", "link": 1}], "outputs": [{"name": "MODEL", "type": "MODEL", "links": [4]}], "properties": {"Node name for S&R": "ModelSamplingAuraFlow"}, "widgets_values": [3.0, "flow"], "widgets_values_named": {"shift": 3.0, "sampling": "flow"}}, {"id": 5, "type": "CLIPTextEncode", "pos": [410, 340], "size": [300, 100], "flags": {}, "order": 4, "mode": 0, "inputs": [{"name": "clip", "type": "CLIP", "link": 2}], "outputs": [{"name": "CONDITIONING", "type": "CONDITIONING", "links": [5]}], "properties": {"Node name for S&R": "CLIPTextEncode"}, "widgets_values": ["1girl, solo, looking at viewer, upper body, detailed face, soft lighting"], "widgets_values_named": {"text": "1girl, solo, looking at viewer, upper body, detailed face, soft lighting"}}, {"id": 6, "type": "CLIPTextEncode", "pos": [410, 560], "size": [300, 100], "flags": {}, "order": 5, "mode": 0, "inputs": [{"name": "clip", "type": "CLIP", "link": 3}], "outputs": [{"name": "CONDITIONING", "type": "CONDITIONING", "links": [6]}], "properties": {"Node name for S&R": "CLIPTextEncode"}, "widgets_values": ["lowres, bad anatomy, bad hands, extra digits, fewer digits, cropped, worst quality, low quality, jpeg artifacts, signature, watermark, username, blurry"], "widgets_values_named": {"text": "lowres, bad anatomy, bad hands, extra digits, fewer digits, cropped, worst quality, low quality, jpeg artifacts, signature, watermark, username, blurry"}}, {"id": 7, "type": "EmptyCosmosLatentVideo", "pos": [80, 780], "size": [300, 100], "flags": {}, "order": 6, "mode": 0, "inputs": [], "outputs": [{"name": "LATENT", "type": "LATENT", "links": [7]}], "properties": {"Node name for S&R": "EmptyCosmosLatentVideo"}, "widgets_values": [1024, 1024, 1, 1], "widgets_values_named": {"width": 1024, "height": 1024, "length": 1, "batch_size": 1}}, {"id": 8, "type": "KSampler", "pos": [740, 120], "size": [300, 100], "flags": {}, "order": 7, "mode": 0, "inputs": [{"name": "model", "type": "MODEL", "link": 4}, {"name": "positive", "type": "CONDITIONING", "link": 5}, {"name": "negative", "type": "CONDITIONING", "link": 6}, {"name": "latent_image", "type": "LATENT", "link": 7}], "outputs": [{"name": "LATENT", "type": "LATENT", "links": [8]}], "properties": {"Node name for S&R": "KSampler"}, "widgets_values": [0, "randomize", 8, 1.0, "euler_ancestral", "normal", 1.0], "widgets_values_named": {"seed": 0, "control_after_generate": "randomize", "steps": 8, "cfg": 1.0, "sampler_name": "euler_ancestral", "scheduler": "normal", "denoise": 1.0}}, {"id": 9, "type": "VAEDecode", "pos": [1070, 120], "size": [300, 100], "flags": {}, "order": 8, "mode": 0, "inputs": [{"name": "samples", "type": "LATENT", "link": 8}, {"name": "vae", "type": "VAE", "link": 9}], "outputs": [{"name": "IMAGE", "type": "IMAGE", "links": [10]}], "properties": {"Node name for S&R": "VAEDecode"}, "widgets_values": [], "widgets_values_named": {}}, {"id": 10, "type": "SaveImage", "pos": [1400, 120], "size": [300, 100], "flags": {}, "order": 9, "mode": 0, "inputs": [{"name": "images", "type": "IMAGE", "link": 10}], "outputs": [{"name": "images", "type": "IMAGE", "links": []}], "properties": {"Node name for S&R": "SaveImage"}, "widgets_values": ["app/txt2img-anima-turbo-q5km"], "widgets_values_named": {"filename_prefix": "app/txt2img-anima-turbo-q5km"}}], "links": [[1, 1, 0, 4, 0, "MODEL"], [2, 2, 0, 5, 0, "CLIP"], [3, 2, 0, 6, 0, "CLIP"], [4, 4, 0, 8, 0, "MODEL"], [5, 5, 0, 8, 1, "CONDITIONING"], [6, 6, 0, 8, 2, "CONDITIONING"], [7, 7, 0, 8, 3, "LATENT"], [8, 8, 0, 9, 0, "LATENT"], [9, 3, 0, 9, 1, "VAE"], [10, 9, 0, 10, 0, "IMAGE"]], "groups": [], "config": {}, "extra": {"linearMode": true, "linearData": {"inputs": [[5, "text"], [7, "width"], [7, "height"], [8, "seed"], [8, "steps"]], "outputs": [10]}, "ds": {"scale": 0.75, "offset": [0, 0]}}, "version": 0.4}', '改图-Anima-Turbo.json': '{"id": null, "revision": 0, "last_node_id": 12, "last_link_id": 13, "nodes": [{"id": 1, "type": "UnetLoaderGGUF", "pos": [80, 120], "size": [300, 100], "flags": {}, "order": 0, "mode": 0, "inputs": [], "outputs": [{"name": "MODEL", "type": "MODEL", "links": [1]}], "properties": {"Node name for S&R": "UnetLoaderGGUF"}, "widgets_values": ["anima-turbo-v1.1-Q5_K_M.gguf"], "widgets_values_named": {"unet_name": "anima-turbo-v1.1-Q5_K_M.gguf"}}, {"id": 2, "type": "CLIPLoader", "pos": [80, 340], "size": [300, 100], "flags": {}, "order": 1, "mode": 0, "inputs": [], "outputs": [{"name": "CLIP", "type": "CLIP", "links": [2, 3]}], "properties": {"Node name for S&R": "CLIPLoader"}, "widgets_values": ["split_files/text_encoders/qwen_3_06b_base.safetensors", "cosmos", "default"], "widgets_values_named": {"clip_name": "split_files/text_encoders/qwen_3_06b_base.safetensors", "type": "cosmos", "device": "default"}}, {"id": 3, "type": "VAELoader", "pos": [80, 560], "size": [300, 100], "flags": {}, "order": 2, "mode": 0, "inputs": [], "outputs": [{"name": "VAE", "type": "VAE", "links": [6, 12]}], "properties": {"Node name for S&R": "VAELoader"}, "widgets_values": ["split_files/vae/qwen_image_vae.safetensors"], "widgets_values_named": {"vae_name": "split_files/vae/qwen_image_vae.safetensors"}}, {"id": 4, "type": "ModelSamplingAuraFlow", "pos": [410, 120], "size": [300, 100], "flags": {}, "order": 3, "mode": 0, "inputs": [{"name": "model", "type": "MODEL", "link": 1}], "outputs": [{"name": "MODEL", "type": "MODEL", "links": [7]}], "properties": {"Node name for S&R": "ModelSamplingAuraFlow"}, "widgets_values": [3.0, "flow"], "widgets_values_named": {"shift": 3.0, "sampling": "flow"}}, {"id": 5, "type": "CLIPTextEncode", "pos": [410, 340], "size": [300, 100], "flags": {}, "order": 4, "mode": 0, "inputs": [{"name": "clip", "type": "CLIP", "link": 2}], "outputs": [{"name": "CONDITIONING", "type": "CONDITIONING", "links": [8]}], "properties": {"Node name for S&R": "CLIPTextEncode"}, "widgets_values": ["1girl, solo, looking at viewer, upper body, detailed face, soft lighting"], "widgets_values_named": {"text": "1girl, solo, looking at viewer, upper body, detailed face, soft lighting"}}, {"id": 6, "type": "CLIPTextEncode", "pos": [410, 560], "size": [300, 100], "flags": {}, "order": 5, "mode": 0, "inputs": [{"name": "clip", "type": "CLIP", "link": 3}], "outputs": [{"name": "CONDITIONING", "type": "CONDITIONING", "links": [9]}], "properties": {"Node name for S&R": "CLIPTextEncode"}, "widgets_values": ["lowres, bad anatomy, bad hands, extra digits, fewer digits, cropped, worst quality, low quality, jpeg artifacts, signature, watermark, username, blurry"], "widgets_values_named": {"text": "lowres, bad anatomy, bad hands, extra digits, fewer digits, cropped, worst quality, low quality, jpeg artifacts, signature, watermark, username, blurry"}}, {"id": 7, "type": "LoadImage", "pos": [80, 780], "size": [300, 100], "flags": {}, "order": 6, "mode": 0, "inputs": [], "outputs": [{"name": "IMAGE", "type": "IMAGE", "links": [4]}, {"name": "MASK", "type": "MASK", "links": []}], "properties": {"Node name for S&R": "LoadImage"}, "widgets_values": ["__REFERENCE__"], "widgets_values_named": {"image": "__REFERENCE__"}}, {"id": 8, "type": "ImageScale", "pos": [410, 780], "size": [300, 100], "flags": {}, "order": 7, "mode": 0, "inputs": [{"name": "image", "type": "IMAGE", "link": 4}], "outputs": [{"name": "IMAGE", "type": "IMAGE", "links": [5]}], "properties": {"Node name for S&R": "ImageScale"}, "widgets_values": ["lanczos", 1024, 1024, "disabled"], "widgets_values_named": {"upscale_method": "lanczos", "width": 1024, "height": 1024, "crop": "disabled"}}, {"id": 9, "type": "VAEEncode", "pos": [740, 120], "size": [300, 100], "flags": {}, "order": 8, "mode": 0, "inputs": [{"name": "pixels", "type": "IMAGE", "link": 5}, {"name": "vae", "type": "VAE", "link": 6}], "outputs": [{"name": "LATENT", "type": "LATENT", "links": [10]}], "properties": {"Node name for S&R": "VAEEncode"}, "widgets_values": [], "widgets_values_named": {}}, {"id": 10, "type": "KSampler", "pos": [1070, 120], "size": [300, 100], "flags": {}, "order": 9, "mode": 0, "inputs": [{"name": "model", "type": "MODEL", "link": 7}, {"name": "positive", "type": "CONDITIONING", "link": 8}, {"name": "negative", "type": "CONDITIONING", "link": 9}, {"name": "latent_image", "type": "LATENT", "link": 10}], "outputs": [{"name": "LATENT", "type": "LATENT", "links": [11]}], "properties": {"Node name for S&R": "KSampler"}, "widgets_values": [0, "randomize", 8, 1.0, "euler_ancestral", "normal", 0.6], "widgets_values_named": {"seed": 0, "control_after_generate": "randomize", "steps": 8, "cfg": 1.0, "sampler_name": "euler_ancestral", "scheduler": "normal", "denoise": 0.6}}, {"id": 11, "type": "VAEDecode", "pos": [1400, 120], "size": [300, 100], "flags": {}, "order": 10, "mode": 0, "inputs": [{"name": "samples", "type": "LATENT", "link": 11}, {"name": "vae", "type": "VAE", "link": 12}], "outputs": [{"name": "IMAGE", "type": "IMAGE", "links": [13]}], "properties": {"Node name for S&R": "VAEDecode"}, "widgets_values": [], "widgets_values_named": {}}, {"id": 12, "type": "SaveImage", "pos": [1730, 120], "size": [300, 100], "flags": {}, "order": 11, "mode": 0, "inputs": [{"name": "images", "type": "IMAGE", "link": 13}], "outputs": [{"name": "images", "type": "IMAGE", "links": []}], "properties": {"Node name for S&R": "SaveImage"}, "widgets_values": ["app/img2img-anima-turbo-q5km"], "widgets_values_named": {"filename_prefix": "app/img2img-anima-turbo-q5km"}}], "links": [[1, 1, 0, 4, 0, "MODEL"], [2, 2, 0, 5, 0, "CLIP"], [3, 2, 0, 6, 0, "CLIP"], [4, 7, 0, 8, 0, "IMAGE"], [5, 8, 0, 9, 0, "IMAGE"], [6, 3, 0, 9, 1, "VAE"], [7, 4, 0, 10, 0, "MODEL"], [8, 5, 0, 10, 1, "CONDITIONING"], [9, 6, 0, 10, 2, "CONDITIONING"], [10, 9, 0, 10, 3, "LATENT"], [11, 10, 0, 11, 0, "LATENT"], [12, 3, 0, 11, 1, "VAE"], [13, 11, 0, 12, 0, "IMAGE"]], "groups": [], "config": {}, "extra": {"linearMode": true, "linearData": {"inputs": [[5, "text"], [7, "image"], [10, "denoise"], [8, "width"], [8, "height"], [10, "seed"], [10, "steps"]], "outputs": [12]}, "ds": {"scale": 0.75, "offset": [0, 0]}}, "version": 0.4}', '文生图-Chroma1-HD.json': '{"id": null, "revision": 0, "last_node_id": 10, "last_link_id": 10, "nodes": [{"id": 1, "type": "UnetLoaderGGUF", "pos": [80, 120], "size": [300, 100], "flags": {}, "order": 0, "mode": 0, "inputs": [], "outputs": [{"name": "MODEL", "type": "MODEL", "links": [1]}], "properties": {"Node name for S&R": "UnetLoaderGGUF"}, "widgets_values": ["Chroma1-HD-Q8_0.gguf"], "widgets_values_named": {"unet_name": "Chroma1-HD-Q8_0.gguf"}}, {"id": 2, "type": "CLIPLoaderGGUF", "pos": [80, 340], "size": [300, 100], "flags": {}, "order": 1, "mode": 0, "inputs": [], "outputs": [{"name": "CLIP", "type": "CLIP", "links": [2, 3]}], "properties": {"Node name for S&R": "CLIPLoaderGGUF"}, "widgets_values": ["t5-v1_1-xxl-encoder-Q5_K_M.gguf", "chroma"], "widgets_values_named": {"clip_name": "t5-v1_1-xxl-encoder-Q5_K_M.gguf", "type": "chroma"}}, {"id": 3, "type": "VAELoader", "pos": [80, 560], "size": [300, 100], "flags": {}, "order": 2, "mode": 0, "inputs": [], "outputs": [{"name": "VAE", "type": "VAE", "links": [9]}], "properties": {"Node name for S&R": "VAELoader"}, "widgets_values": ["ae.safetensors"], "widgets_values_named": {"vae_name": "ae.safetensors"}}, {"id": 4, "type": "ModelSamplingAuraFlow", "pos": [410, 120], "size": [300, 100], "flags": {}, "order": 3, "mode": 0, "inputs": [{"name": "model", "type": "MODEL", "link": 1}], "outputs": [{"name": "MODEL", "type": "MODEL", "links": [4]}], "properties": {"Node name for S&R": "ModelSamplingAuraFlow"}, "widgets_values": [1.0, "flow"], "widgets_values_named": {"shift": 1.0, "sampling": "flow"}}, {"id": 5, "type": "CLIPTextEncode", "pos": [410, 340], "size": [300, 100], "flags": {}, "order": 4, "mode": 0, "inputs": [{"name": "clip", "type": "CLIP", "link": 2}], "outputs": [{"name": "CONDITIONING", "type": "CONDITIONING", "links": [5]}], "properties": {"Node name for S&R": "CLIPTextEncode"}, "widgets_values": ["1girl, solo, looking at viewer, upper body, detailed face, soft lighting"], "widgets_values_named": {"text": "1girl, solo, looking at viewer, upper body, detailed face, soft lighting"}}, {"id": 6, "type": "CLIPTextEncode", "pos": [410, 560], "size": [300, 100], "flags": {}, "order": 5, "mode": 0, "inputs": [{"name": "clip", "type": "CLIP", "link": 3}], "outputs": [{"name": "CONDITIONING", "type": "CONDITIONING", "links": [6]}], "properties": {"Node name for S&R": "CLIPTextEncode"}, "widgets_values": ["lowres, bad anatomy, bad hands, extra digits, fewer digits, cropped, worst quality, low quality, jpeg artifacts, signature, watermark, username, blurry"], "widgets_values_named": {"text": "lowres, bad anatomy, bad hands, extra digits, fewer digits, cropped, worst quality, low quality, jpeg artifacts, signature, watermark, username, blurry"}}, {"id": 7, "type": "EmptySD3LatentImage", "pos": [80, 780], "size": [300, 100], "flags": {}, "order": 6, "mode": 0, "inputs": [], "outputs": [{"name": "LATENT", "type": "LATENT", "links": [7]}], "properties": {"Node name for S&R": "EmptySD3LatentImage"}, "widgets_values": [1024, 1024, 1], "widgets_values_named": {"width": 1024, "height": 1024, "batch_size": 1}}, {"id": 8, "type": "KSampler", "pos": [740, 120], "size": [300, 100], "flags": {}, "order": 7, "mode": 0, "inputs": [{"name": "model", "type": "MODEL", "link": 4}, {"name": "positive", "type": "CONDITIONING", "link": 5}, {"name": "negative", "type": "CONDITIONING", "link": 6}, {"name": "latent_image", "type": "LATENT", "link": 7}], "outputs": [{"name": "LATENT", "type": "LATENT", "links": [8]}], "properties": {"Node name for S&R": "KSampler"}, "widgets_values": [0, "randomize", 30, 3.5, "euler_ancestral", "normal", 1.0], "widgets_values_named": {"seed": 0, "control_after_generate": "randomize", "steps": 30, "cfg": 3.5, "sampler_name": "euler_ancestral", "scheduler": "normal", "denoise": 1.0}}, {"id": 9, "type": "VAEDecode", "pos": [1070, 120], "size": [300, 100], "flags": {}, "order": 8, "mode": 0, "inputs": [{"name": "samples", "type": "LATENT", "link": 8}, {"name": "vae", "type": "VAE", "link": 9}], "outputs": [{"name": "IMAGE", "type": "IMAGE", "links": [10]}], "properties": {"Node name for S&R": "VAEDecode"}, "widgets_values": [], "widgets_values_named": {}}, {"id": 10, "type": "SaveImage", "pos": [1400, 120], "size": [300, 100], "flags": {}, "order": 9, "mode": 0, "inputs": [{"name": "images", "type": "IMAGE", "link": 10}], "outputs": [{"name": "images", "type": "IMAGE", "links": []}], "properties": {"Node name for S&R": "SaveImage"}, "widgets_values": ["app/txt2img-chroma1-hd-q8"], "widgets_values_named": {"filename_prefix": "app/txt2img-chroma1-hd-q8"}}], "links": [[1, 1, 0, 4, 0, "MODEL"], [2, 2, 0, 5, 0, "CLIP"], [3, 2, 0, 6, 0, "CLIP"], [4, 4, 0, 8, 0, "MODEL"], [5, 5, 0, 8, 1, "CONDITIONING"], [6, 6, 0, 8, 2, "CONDITIONING"], [7, 7, 0, 8, 3, "LATENT"], [8, 8, 0, 9, 0, "LATENT"], [9, 3, 0, 9, 1, "VAE"], [10, 9, 0, 10, 0, "IMAGE"]], "groups": [], "config": {}, "extra": {"linearMode": true, "linearData": {"inputs": [[5, "text"], [6, "text"], [7, "width"], [7, "height"], [8, "seed"], [8, "steps"], [8, "cfg"]], "outputs": [10]}, "ds": {"scale": 0.75, "offset": [0, 0]}}, "version": 0.4}', '改图-Chroma1-HD.json': '{"id": null, "revision": 0, "last_node_id": 12, "last_link_id": 13, "nodes": [{"id": 1, "type": "UnetLoaderGGUF", "pos": [80, 120], "size": [300, 100], "flags": {}, "order": 0, "mode": 0, "inputs": [], "outputs": [{"name": "MODEL", "type": "MODEL", "links": [1]}], "properties": {"Node name for S&R": "UnetLoaderGGUF"}, "widgets_values": ["Chroma1-HD-Q8_0.gguf"], "widgets_values_named": {"unet_name": "Chroma1-HD-Q8_0.gguf"}}, {"id": 2, "type": "CLIPLoaderGGUF", "pos": [80, 340], "size": [300, 100], "flags": {}, "order": 1, "mode": 0, "inputs": [], "outputs": [{"name": "CLIP", "type": "CLIP", "links": [2, 3]}], "properties": {"Node name for S&R": "CLIPLoaderGGUF"}, "widgets_values": ["t5-v1_1-xxl-encoder-Q5_K_M.gguf", "chroma"], "widgets_values_named": {"clip_name": "t5-v1_1-xxl-encoder-Q5_K_M.gguf", "type": "chroma"}}, {"id": 3, "type": "VAELoader", "pos": [80, 560], "size": [300, 100], "flags": {}, "order": 2, "mode": 0, "inputs": [], "outputs": [{"name": "VAE", "type": "VAE", "links": [6, 12]}], "properties": {"Node name for S&R": "VAELoader"}, "widgets_values": ["ae.safetensors"], "widgets_values_named": {"vae_name": "ae.safetensors"}}, {"id": 4, "type": "ModelSamplingAuraFlow", "pos": [410, 120], "size": [300, 100], "flags": {}, "order": 3, "mode": 0, "inputs": [{"name": "model", "type": "MODEL", "link": 1}], "outputs": [{"name": "MODEL", "type": "MODEL", "links": [7]}], "properties": {"Node name for S&R": "ModelSamplingAuraFlow"}, "widgets_values": [1.0, "flow"], "widgets_values_named": {"shift": 1.0, "sampling": "flow"}}, {"id": 5, "type": "CLIPTextEncode", "pos": [410, 340], "size": [300, 100], "flags": {}, "order": 4, "mode": 0, "inputs": [{"name": "clip", "type": "CLIP", "link": 2}], "outputs": [{"name": "CONDITIONING", "type": "CONDITIONING", "links": [8]}], "properties": {"Node name for S&R": "CLIPTextEncode"}, "widgets_values": ["1girl, solo, looking at viewer, upper body, detailed face, soft lighting"], "widgets_values_named": {"text": "1girl, solo, looking at viewer, upper body, detailed face, soft lighting"}}, {"id": 6, "type": "CLIPTextEncode", "pos": [410, 560], "size": [300, 100], "flags": {}, "order": 5, "mode": 0, "inputs": [{"name": "clip", "type": "CLIP", "link": 3}], "outputs": [{"name": "CONDITIONING", "type": "CONDITIONING", "links": [9]}], "properties": {"Node name for S&R": "CLIPTextEncode"}, "widgets_values": ["lowres, bad anatomy, bad hands, extra digits, fewer digits, cropped, worst quality, low quality, jpeg artifacts, signature, watermark, username, blurry"], "widgets_values_named": {"text": "lowres, bad anatomy, bad hands, extra digits, fewer digits, cropped, worst quality, low quality, jpeg artifacts, signature, watermark, username, blurry"}}, {"id": 7, "type": "LoadImage", "pos": [80, 780], "size": [300, 100], "flags": {}, "order": 6, "mode": 0, "inputs": [], "outputs": [{"name": "IMAGE", "type": "IMAGE", "links": [4]}, {"name": "MASK", "type": "MASK", "links": []}], "properties": {"Node name for S&R": "LoadImage"}, "widgets_values": ["__REFERENCE__"], "widgets_values_named": {"image": "__REFERENCE__"}}, {"id": 8, "type": "ImageScale", "pos": [410, 780], "size": [300, 100], "flags": {}, "order": 7, "mode": 0, "inputs": [{"name": "image", "type": "IMAGE", "link": 4}], "outputs": [{"name": "IMAGE", "type": "IMAGE", "links": [5]}], "properties": {"Node name for S&R": "ImageScale"}, "widgets_values": ["lanczos", 1024, 1024, "disabled"], "widgets_values_named": {"upscale_method": "lanczos", "width": 1024, "height": 1024, "crop": "disabled"}}, {"id": 9, "type": "VAEEncode", "pos": [740, 120], "size": [300, 100], "flags": {}, "order": 8, "mode": 0, "inputs": [{"name": "pixels", "type": "IMAGE", "link": 5}, {"name": "vae", "type": "VAE", "link": 6}], "outputs": [{"name": "LATENT", "type": "LATENT", "links": [10]}], "properties": {"Node name for S&R": "VAEEncode"}, "widgets_values": [], "widgets_values_named": {}}, {"id": 10, "type": "KSampler", "pos": [1070, 120], "size": [300, 100], "flags": {}, "order": 9, "mode": 0, "inputs": [{"name": "model", "type": "MODEL", "link": 7}, {"name": "positive", "type": "CONDITIONING", "link": 8}, {"name": "negative", "type": "CONDITIONING", "link": 9}, {"name": "latent_image", "type": "LATENT", "link": 10}], "outputs": [{"name": "LATENT", "type": "LATENT", "links": [11]}], "properties": {"Node name for S&R": "KSampler"}, "widgets_values": [0, "randomize", 30, 3.5, "euler_ancestral", "normal", 0.6], "widgets_values_named": {"seed": 0, "control_after_generate": "randomize", "steps": 30, "cfg": 3.5, "sampler_name": "euler_ancestral", "scheduler": "normal", "denoise": 0.6}}, {"id": 11, "type": "VAEDecode", "pos": [1400, 120], "size": [300, 100], "flags": {}, "order": 10, "mode": 0, "inputs": [{"name": "samples", "type": "LATENT", "link": 11}, {"name": "vae", "type": "VAE", "link": 12}], "outputs": [{"name": "IMAGE", "type": "IMAGE", "links": [13]}], "properties": {"Node name for S&R": "VAEDecode"}, "widgets_values": [], "widgets_values_named": {}}, {"id": 12, "type": "SaveImage", "pos": [1730, 120], "size": [300, 100], "flags": {}, "order": 11, "mode": 0, "inputs": [{"name": "images", "type": "IMAGE", "link": 13}], "outputs": [{"name": "images", "type": "IMAGE", "links": []}], "properties": {"Node name for S&R": "SaveImage"}, "widgets_values": ["app/img2img-chroma1-hd-q8"], "widgets_values_named": {"filename_prefix": "app/img2img-chroma1-hd-q8"}}], "links": [[1, 1, 0, 4, 0, "MODEL"], [2, 2, 0, 5, 0, "CLIP"], [3, 2, 0, 6, 0, "CLIP"], [4, 7, 0, 8, 0, "IMAGE"], [5, 8, 0, 9, 0, "IMAGE"], [6, 3, 0, 9, 1, "VAE"], [7, 4, 0, 10, 0, "MODEL"], [8, 5, 0, 10, 1, "CONDITIONING"], [9, 6, 0, 10, 2, "CONDITIONING"], [10, 9, 0, 10, 3, "LATENT"], [11, 10, 0, 11, 0, "LATENT"], [12, 3, 0, 11, 1, "VAE"], [13, 11, 0, 12, 0, "IMAGE"]], "groups": [], "config": {}, "extra": {"linearMode": true, "linearData": {"inputs": [[5, "text"], [6, "text"], [7, "image"], [10, "denoise"], [8, "width"], [8, "height"], [10, "seed"], [10, "steps"], [10, "cfg"]], "outputs": [12]}, "ds": {"scale": 0.75, "offset": [0, 0]}}, "version": 0.4}'}

# ---- 按勾选框算出要下什么、装什么（纯计算，无副作用 —— 所以测试能直接 exec 这一段）
# `globals()` 取的就是本格的命名空间；勾选框变量名由 MODELS_SPEC 的 var 给出。
picked = [m for m in MODELS_SPEC if globals()[m['var']]]
assert picked, '一个模型都没勾。至少勾一个 —— 建议留 Illustrious，约 7 分钟就能用上画布。'
SELECTED_IDS = [m['id'] for m in picked]
keep = {(sub, name) for m in picked for sub, name in m['warm']}
# LoRA 是共享的，但**绑架构** —— 只有选中模型里有匹配架构时才带上。
arches = {m['arch'] for m in picked}
keep |= {(sub, name) for spec in LORA_SPEC for sub, name in [spec['key']]
         if spec['arch'] in arches}
wanted_apps = {a for m in picked for a in m['apps']}
WARM_ITEMS = [i for i in ALL_WARM_ITEMS if (i['subdir'], i['filename']) in keep]
APP_JSON = {k: v for k, v in ALL_APP_JSON.items() if k in wanted_apps}

print(f'已勾选 {len(picked)}/{len(MODELS_SPEC)} 个模型 → '
      f'下载 {len(WARM_ITEMS)} 条、应用 {len(APP_JSON)} 个')
for m in picked:
    print('  - ' + m['display'])
if TUNNEL_MODE == 'off':
    print('公网隧道：关（用 3/3 的内嵌画布）')
elif TUNNEL_MODE == 'quick':
    print('公网隧道：quick —— 不用 token，但地址每次重启都会换')
else:
    print('公网隧道：named —— 需要 CF_TUNNEL_TOKEN，缺了会明确报错')
print('ComfyUI-Manager：' + ('装（可装任意节点/模型）' if ENABLE_MANAGER else '不装'))
print('持久化：' + ('Drive（output / user / models/loras）' if PERSIST_TO_DRIVE else 'Colab 临时盘'))
if EXTRA_NODES:
    print(f"额外节点：{EXTRA_NODES}")

# ---- 辅助函数：**定义在这一格**，后面的格子直接用
# 为什么不单开一格：单开就意味着「跳过它 → 后面全 NameError」，而那一格本身没有
# 任何独立价值。跳着跑就会撞上 `py_bg 未定义`。
REMOTE_DIR = pathlib.Path('/content/imagegen')
COMFY = pathlib.Path('/content/ComfyUI')


def py(script, *args, check=True):
    """在 VM 里跑一个远端脚本，环境变量按约定注入。"""
    cmd = [sys.executable, str(REMOTE_DIR / script), *args]
    proc = subprocess.run(cmd, env={**os.environ, **ENV},
                          capture_output=True, text=True)
    if proc.stdout:
        print(proc.stdout[-4000:])
    if proc.returncode and check:
        # 远端脚本把诊断打在 stdout（如 comfy_start 的「找不到 /content/ComfyUI」），
        # stderr 一般为空。只报 stderr 会让人看到一句没有信息量的「退出码 1」。
        detail = (proc.stderr or '')[-2000:] or (proc.stdout or '')[-2000:]
        raise RuntimeError(f'{script} 退出码 {proc.returncode}\n{detail}')
    return proc


def py_bg(script, log, *args):
    """后台跑并返回进程；长了不占单元格，但必须等完成的步骤要再 wait_process。"""
    handle = open(log, 'ab')
    proc = subprocess.Popen([sys.executable, str(REMOTE_DIR / script), *args],
                            env={**os.environ, **ENV}, stdout=handle,
                            stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL,
                            start_new_session=True)
    print(f'后台 pid={proc.pid} -> {log}')
    return proc


def tail(path, n=15):
    p = pathlib.Path(path)
    if not p.is_file():
        return f'（还没有 {path}）'
    return '\n'.join(p.read_text(errors='replace').splitlines()[-n:])


def wait_process(proc, log, marker, timeout=1800):
    """等一个必须完成的后台阶段；每 10 秒打印日志，避免后续格子抢跑。"""
    deadline = time.time() + timeout
    while proc.poll() is None and time.time() < deadline:
        print(f'仍在运行（pid={proc.pid}），日志尾部：')
        print(tail(log, 8))
        time.sleep(10)
    if proc.poll() is None:
        proc.kill()
        raise RuntimeError(f'{log} 超过 {timeout}s，已停止后台进程')
    output = tail(log, 30)
    print(output)
    if proc.returncode != 0 or marker not in output:
        raise RuntimeError(f'{log} 未完成（退出码 {proc.returncode}，需要标记 {marker!r}）')
    print(f'完成：{marker}')


## 1/3　装 ComfyUI + 后台下模型

**约 7 分钟，这一格会等到装完才结束**（每 10 秒打印一次进度）。
装 ComfyUI 本体、ComfyUI-GGUF（GGUF 权重没它一个都加载不了）、IP-Adapter 节点、
**ComfyUI-Manager**，以及 Illustrious 基础权重。

> ⚠️ **Manager 能装任意代码，而且它会把从公网地址进来的请求当成本地请求。**
> 也就是说：**拿到你网址的人，可以在你的 Colab 上装任意东西。**
> 不想承担就把配置格里的 `ENABLE_MANAGER` 取消勾选。

然后**开始下你勾的模型** —— 放后台，不挡第 2 步。

> ⚠️ **必须等这一格跑完再往下走**，否则第 2 步会报「找不到 /content/ComfyUI」。


In [ ]:
# 跳格守卫：直接跑本格会缺 py/py_bg/wait_process（它们定义在第 1 格里）。
assert 'py_bg' in globals(), (
    '本格依赖第 1 格「⚙️ 配置」—— 它定义了 py/py_bg/wait_process 和全部数据。'
    '请先跑那一格，或者直接点「全部运行」。')

REMOTE_DIR.mkdir(parents=True, exist_ok=True)
for name, text in REMOTE_SRC.items():
    (REMOTE_DIR / name).write_text(text, encoding='utf-8')
(REMOTE_DIR / 'warm.json').write_text(json.dumps(WARM_ITEMS, ensure_ascii=False),
                                      encoding='utf-8')

# 应用不在这里写 —— 它们的落点取决于 user 目录（可能挂在 Drive 上），
# 而那要等第 2 步挂载之后才知道。
# 两个环境开关，都由配置格决定：
#   COMFY_EXTRA_NODES —— 白名单 key（不接受任意仓库）
#   IMAGEGEN_MANAGER  —— '0' 就不装 ComfyUI-Manager
# 「基础权重」（Illustrious 检查点 / IP-Adapter / CLIP vision，约 13 GB）只给 SDXL 用，
# 一个 SDXL 都没勾就不下 —— 否则「没勾的不会下载」就是假话。
any_sdxl = any(m['arch'] == 'sdxl' for m in picked)
ENV = {**ENV, 'COMFY_EXTRA_NODES': EXTRA_NODES,
       'IMAGEGEN_MANAGER': '1' if ENABLE_MANAGER else '0',
       'IMAGEGEN_BASE_WEIGHTS': '1' if any_sdxl else '0'}
if not any_sdxl:
    print('没勾 SDXL 系模型 —— 跳过 Illustrious / IP-Adapter / CLIP vision。')

# ---- 统一持久化根：所有需要留存的东西从这里派生。
# 这一步必须在 bootstrap 之前完成，否则 Manager / warm 下载会各写一份。
for _key in ('IMAGEGEN_OUTPUT_DIR', 'IMAGEGEN_USER_DIR', 'IMAGEGEN_LORA_DIR'):
    ENV.pop(_key, None)
if PERSIST_TO_DRIVE:
    from google.colab import drive
    mount_point = pathlib.Path('/content/drive')
    if not mount_point.is_mount():
        drive.mount(str(mount_point))
    if not mount_point.is_mount() or not (mount_point / 'MyDrive').is_dir():
        raise RuntimeError('Google Drive 没有成功挂载到 /content/drive；为避免假持久化，已停止。')
    probe = mount_point / 'MyDrive' / '.imagegen-write-test'
    try:
        probe.write_text('ok', encoding='utf-8')
        probe.unlink()
    except OSError as exc:
        raise RuntimeError(f'Google Drive 不可写，未启用持久化：{exc}') from exc
    PERSIST_ROOT = mount_point / 'MyDrive/imagegen'
    OUTPUT_DIR = PERSIST_ROOT / 'output'
    USER_DIR = PERSIST_ROOT / 'user'
    LORA_DIR = PERSIST_ROOT / 'models/loras'
    for path in (OUTPUT_DIR, USER_DIR / 'default/workflows', LORA_DIR):
        path.mkdir(parents=True, exist_ok=True)
    ENV = {**ENV, 'IMAGEGEN_OUTPUT_DIR': str(OUTPUT_DIR),
           'IMAGEGEN_USER_DIR': str(USER_DIR),
           'IMAGEGEN_LORA_DIR': str(LORA_DIR)}
    print(f'持久化根 → {PERSIST_ROOT}')
else:
    PERSIST_ROOT = COMFY
    OUTPUT_DIR = COMFY / 'output'
    USER_DIR = COMFY / 'user'
    LORA_DIR = COMFY / 'models/loras'
    USER_DIR.mkdir(parents=True, exist_ok=True)
    LORA_DIR.mkdir(parents=True, exist_ok=True)
    print('持久化：Colab 临时盘')

proc = py_bg('comfy_bootstrap.py', '/content/bootstrap.log')
wait_process(proc, '/content/bootstrap.log', 'SETUP_DONE', timeout=1800)

# 上一步只装了 ComfyUI 本体与节点。**你勾的模型现在才开始下** ——
# 放后台是为了不挡住第 2 步（权重几十 GB，画布没必要等它）。进度看第 3 步。
py_bg('warm_models.py', '/content/warm.log')

# ---- 确认 GPU 真的能用
# 这一格最阴的失败方式：**torch 被换成了不含 CUDA 的构建**，于是 ComfyUI 起不来、
# 只丢一句 `Torch not compiled with CUDA enabled`。与其让你对着一屏 log 猜，
# 不如在这里直接说清楚 —— 而且是在等了几分钟安装**之后立刻**说。
import torch
if not torch.cuda.is_available():
    cuda = torch.version.cuda or '（无 —— 装成了 CPU 版）'
    raise SystemExit(
        f'这个运行时的 torch 用不了 GPU：版本 {torch.__version__}，CUDA {cuda}\n'
        'ComfyUI 起不来就是这个原因，重跑第 2 步也没用。\n'
        '最省事：菜单「运行时 → 断开并删除运行时」，然后从头重跑 —— '
        '新运行时的 torch 是好的。')
print(f'GPU：{torch.cuda.get_device_name(0)}　torch {torch.__version__}')
print('下载已开始，可以继续往下跑；进度看第 3 步。')


## 2/3　起 ComfyUI（和可选隧道）

约 1 分钟。起 ComfyUI。默认只起服务，界面在下一格（3/3 画布）。

配置格的 `TUNNEL_MODE` 不是 `off` 时，这里会额外开一个公网地址并打印出来。


In [ ]:
# 跳格守卫：直接跑本格会缺上面那格定义的函数。
assert 'py_bg' in globals(), (
    '本格依赖第 1 格「⚙️ 配置」，请先跑那一格，或直接点「全部运行」。')
# ---- 公网隧道（可选）：按配置格的 TUNNEL_MODE 走
# 让远端脚本知道起哪种隧道 —— quick 不需要 token，named 需要。
ENV = {**ENV, 'IMAGEGEN_TUNNEL_MODE': TUNNEL_MODE}

if TUNNEL_MODE == 'off':
    ENV = {**ENV, 'IMAGEGEN_TUNNEL_MODE': 'off'}
    py('cloudflared_up.py')
    print('公网隧道已关闭 —— 用下面内嵌的画布。')
elif TUNNEL_MODE == 'quick':
    # 地址由 Cloudflare 随机分配、每次重启都换，所以脚本会把它打印出来。
    py('cloudflared_up.py')
else:
    # named 才要 token。放在分支里面 —— quick 用户没存这个 secret，
    # 在外面无条件读会让他们的日志里挂一句莫名其妙的报错。
    try:
        from google.colab import userdata
        TOKEN = userdata.get('CF_TUNNEL_TOKEN')
    except Exception as exc:
        raise SystemExit(f'选了 named 隧道，但读不到 CF_TUNNEL_TOKEN：{exc}\n'
                         '去左侧 🔑 存一条并勾 Notebook access，'
                         '或者把配置里的 TUNNEL_MODE 改成 off 或 quick。')
    env_file = pathlib.Path(ENV['IMAGEGEN_TUNNEL_ENV'])
    env_file.parent.mkdir(parents=True, exist_ok=True)
    env_file.write_text(f'TUNNEL_TOKEN={TOKEN}\n', encoding='utf-8')
    env_file.chmod(0o600)
    print(f'token 已写入 {env_file}（600）')
    py('cloudflared_up.py')

# ---- 生成元数据不用另外存：ComfyUI 会把工作流和提示词写进 PNG 文本块。
# 只把 output/user/loras 三个原生目录切到统一根，模型权重仍留在本地 ComfyUI 目录。
comfy_args = ['--output-directory', str(OUTPUT_DIR),
              '--user-directory', str(USER_DIR)]
model_paths = COMFY / 'extra_model_paths.yaml'
if PERSIST_TO_DRIVE:
    model_paths.write_text(
        'imagegen_drive:\n'
        '  base_path: ' + str(PERSIST_ROOT) + '\n'
        '  is_default: true\n'
        '  loras: models/loras\n', encoding='utf-8')
    comfy_args += ['--extra-model-paths-config', str(model_paths)]
    print(f'LoRA → {LORA_DIR}（ComfyUI 原生 extra_model_paths.yaml）')

# ---- 必须加，否则经代理打开会 403
#
# ComfyUI 默认装一个 origin 中间件：凡是带 `Sec-Fetch-Site: cross-site` 的请求
# 一律 403。而在 Colab 里**两条入口都是 cross-site**（页面在 google.com，
# 界面在 colab.dev / trycloudflare.com），所以连页面本身都打不开。
# 给了这个参数，那个中间件就不再装上。
#
# 代价：它默认 `*`，即任意站点的 JS 都能访问这个实例。
comfy_args.append('--enable-cors-header')

# Manager 走 pip 线，得给它开关；桥接插件硬性检查这个
# （读它的 __init__.py：`if not args.enable_manager: return`）。
if ENABLE_MANAGER:
    comfy_args.append('--enable-manager')

# comfy_start.py 会把 COMFY_ARGS 拼到启动参数上。
ENV = {**ENV, 'COMFY_ARGS': ' '.join(comfy_args)}

# ---- 给模型下载桥接插件配 Civitai token（可选）
# token **只从 Colab Secrets 读，绝不写进笔记本** —— 这文件是要分享出去的。
# 落点读它的 bridge_api_keys.py 确认过：
#   folder_paths.get_system_user_directory("manager")/bridge-api-keys.json
# 即 <USER_DIR>/__manager/bridge-api-keys.json。
if ENABLE_MANAGER:
    try:
        from google.colab import userdata
        civitai = userdata.get('CIVITAI_TOKEN')
    except Exception:
        civitai = None
    manager_dir = USER_DIR / '__manager'
    manager_dir.mkdir(parents=True, exist_ok=True)
    keys = manager_dir / 'bridge-api-keys.json'
    if civitai:
        keys.write_text(json.dumps({'civitai': civitai.strip()}), encoding='utf-8')
        keys.chmod(0o600)
        print('Civitai token 已写入桥接插件（模型下载页签直接能用）')
    else:
        print('没读到 CIVITAI_TOKEN（可选）—— 想下 Civitai 的模型，')
        print('   要么在左侧 🔑 存一条，要么在界面里手动填一次。')

# ---- 写 App Mode 应用
# **同名文件优先用已经存在的那个**：
# Drive 上你手改过、或在 ComfyUI 里另存过的版本，不会被笔记本覆盖。
# 想恢复成笔记本的版本：把那个文件删掉，再重跑本格。
workflows_dir = USER_DIR / 'default/workflows'
installed, kept = [], []
for name, text in APP_JSON.items():
    target = workflows_dir / name
    if target.exists():
        kept.append(name)
        continue
    target.write_text(text, encoding='utf-8')
    installed.append(name)
# 清单：让无头编排那边的「清旧应用」认得这批文件是它建的，不会误删用户自己的工作流。
(workflows_dir / '.imagegen-apps.json').write_text(
    json.dumps({'files': sorted(APP_JSON)}, ensure_ascii=False), encoding='utf-8')
print(f'应用：新装 {len(installed)} 个，保留你已有的 {len(kept)} 个')
for name in kept:
    print(f'   保留 {name}')

# ---- 起 ComfyUI
assert (COMFY / 'main.py').is_file(), (
    f'{COMFY}/main.py 不存在 —— 第 1 步「装 ComfyUI」还没跑完。'
    '回去把第 1 步跑完（它会等到 SETUP_DONE 才结束），再跑本步。')
py('comfy_start.py')
print('ComfyUI 已启动。画布在最后单独一格 —— 想用时重跑那一格。')


## 3/3　画布

把 ComfyUI 嵌在本页里。**什么都不用配**，跑完上面两格再跑这一格就有界面。

嵌不出来的话，把配置格的 `TUNNEL_MODE` 改成 `quick` —— 那会有个公网地址，更稳。


In [ ]:
# 把 ComfyUI 嵌到本页。单独一格、不跟随开关：想用就重跑，不用就晾着。
# 走隧道时它和隧道地址是**并列**的两个入口，不互斥 —— 手机用地址，电脑上直接在这里用。
from google.colab import output
output.serve_kernel_port_as_iframe(8188, path='/', height='900')
print('入口：上面这个内嵌画布。若还开着隧道，地址在第 2 步的输出里。')


## 状态

随时重跑，只读不写 —— **它不会重启任何东西**。

应用数量、模型下载进度、以及「出图前该检查什么」都在这里。


In [ ]:
# 跳格守卫：直接跑本格会缺上面那格定义的函数。
assert 'py_bg' in globals(), (
    '本格依赖第 1 格「⚙️ 配置」，请先跑那一格，或直接点「全部运行」。')
def sh(cmd):
    return subprocess.run(['bash', '-lc', cmd], capture_output=True, text=True).stdout.strip()

comfy_running = bool(sh("pgrep -af 'main.py' | head -1"))
tunnel_running = bool(sh('pgrep -f "cloudflared.*tunne[l]" | head -1'))
print('ComfyUI    :', '在跑' if comfy_running else '**没在跑**')
# **不写死域名** —— 隧道域名由 Cloudflare 侧 ingress 决定，代码里不该有个人域名。
if tunnel_running:
    # quick 的地址是随机的，从日志里捞出来显示 —— 用户随时可能忘掉它。
    import re as _re
    _url = _re.search(r'https://[a-z0-9][a-z0-9-]*\.trycloudflare\.com',
                      tail('/content/cloudflared.log', 400) or '')
    print('公网隧道   : 在跑' + (f' → {_url.group(0)}（quick，重启会换）' if _url
                                 else '（域名见你自己 Cloudflare 的 ingress）'))
else:
    print('公网隧道   : 未启用（用内嵌画布即可）')
print('bootstrap  :', '完成' if 'SETUP_DONE' in tail('/content/bootstrap.log', 60) else '进行中/未跑')
print('模型下载   :', '完成' if 'WARM_DONE' in tail('/content/warm.log', 200) else '进行中/未跑')
# **本格要能单独重跑** —— 用户最常重跑的就是这一格，别因为缺 USER_DIR 就炸。
workflows = (USER_DIR if 'USER_DIR' in globals() else COMFY / 'user') / 'default/workflows'
files = sorted(p.name for p in workflows.glob('*.json')
               if not p.name.startswith('.'))
print(f'App 应用   : {len(files)} 个')
print()
# ---- 下一步该做什么。本格只读状态、**不会重启任何东西**，所以得说清楚该重跑哪一格，
# 否则用户会一直重跑这一格然后发现毫无变化。
if not (COMFY / 'main.py').is_file():
    print('→ 环境不在（会话被回收过）。**从头重跑第 1 步**，约 10 分钟。')
elif not comfy_running:
    print('→ 环境在，但 ComfyUI 没在跑。**重跑第 2 步**。')
elif TUNNEL_MODE != 'off' and not tunnel_running:
    print('→ ComfyUI 在跑，但隧道断了（外网会看到 1033）。**重跑第 2 步** ——'
          '它会重起隧道并打印新地址。')
else:
    print('→ 一切正常。')
print()
# ---- 预检：应用引用的模型文件是否真的都在磁盘上
# 为什么需要：ComfyUI 在 **提交时**校验整张图的所有输入，文件缺失就直接把请求打回，
# 报「缺少所需的模型文件」。应用里的 LoRA 开关**救不了这种情况** —— 开关只决定用不用，
# 文件必须在。所以出图失败先看这一段。
missing = []
for model in picked:
    for sub, name in model['warm']:
        path = (LORA_DIR / name if sub == 'loras'
                else COMFY / 'models' / sub / name)
        if not path.exists():
            missing.append(f'{model["display"]} → {sub}/{name}')
if missing:
    print(f'⚠️ 预检：{len(missing)} 个文件还不在磁盘上（模型可能还在下）')
    for line in missing:
        print('   ', line)
    print('   等下载完再重跑本格；一直不下就是下载失败了，看上面的日志。')
else:
    print(f'预检：{len(picked)} 个模型的文件都在磁盘上 ✓')
print()
print('--- 模型下载日志尾部 ---')
print(tail('/content/warm.log', 8))
